<a href="https://colab.research.google.com/github/amzad-786githumb/SPP_GAN_Research/blob/main/12_SPP_GAN_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ==================================================================================================
# NOTEBOOK 12 — SPP-GAN TRAINING
# ==================================================================================================
#
# SPP-GAN:
# A Statistical-Guided Privacy-Preserving GAN Framework
# for High-Fidelity Synthetic Tabular Data Generation
#
# PURPOSE
# -------
# Train the final SPP-GAN model using:
#
#   • Notebook 02 train-only transformed data
#   • Notebook 03 train-only statistical characterization
#   • Notebook 08 frozen SPP-GAN architecture
#   • Notebook 09 differentiable statistical guidance
#   • Notebook 10 DP-SGD mechanism
#   • Notebook 11 configured RDP privacy accounting
#
# PRIVACY
# -------
#   • Protected component : discriminator / critic
#   • DP mechanism         : DP-SGD
#   • Sampling             : Poisson
#   • Clipping             : flat L2
#   • Noise                : Gaussian
#   • Accountant            : RDP
#
# IMPORTANT PRIVACY BOUNDARY
# --------------------------
#   generator_private          = False
#   statistical_guidance_private= False
#   preprocessing_private      = False
#   end_to_end_privacy_claim   = False
#
# Notebook 12 records the actual training accountant epsilon.
# Formal end-to-end privacy is NOT claimed because the statistical
# reference and preprocessing remain outside the DP mechanism.
#
# DATA POLICY
# ----------
#   TRAIN       : used
#   VALIDATION  : monitoring only
#   TEST        : NEVER used for training or model selection
#
# ==================================================================================================

print("=" * 100)
print("NOTEBOOK 12 — SPP-GAN TRAINING")
print("=" * 100)
print("Framework              : SPP-GAN")
print("Training mechanism     : DP-SGD")
print("Protected component    : SPP-GAN discriminator")
print("Statistical guidance   : Notebook 09")
print("Privacy accounting     : Notebook 11")
print("Training data          : Notebook 02 TRAIN only")
print("Test data              : NOT USED")
print("=" * 100)

NOTEBOOK 12 — SPP-GAN TRAINING
Framework              : SPP-GAN
Training mechanism     : DP-SGD
Protected component    : SPP-GAN discriminator
Statistical guidance   : Notebook 09
Privacy accounting     : Notebook 11
Training data          : Notebook 02 TRAIN only
Test data              : NOT USED


In [2]:
# ==================================================================================================
# SECTION 2 — LOAD CONFIGURATION
# ==================================================================================================

print("=" * 100)
print("2. LOAD CONFIGURATION")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# 1. Imports
# --------------------------------------------------------------------------------------------------

from pathlib import Path
from datetime import datetime, timezone
import json
import hashlib
import numpy as np
import pandas as pd


# --------------------------------------------------------------------------------------------------
# 2. Verify and Mount Google Drive
# --------------------------------------------------------------------------------------------------

DRIVE_ROOT = Path("/content/drive")
MYDRIVE_ROOT = DRIVE_ROOT / "MyDrive"
PROJECT_ROOT = MYDRIVE_ROOT / "SPP_GAN_Research"

print()
print("-" * 100)
print("GOOGLE DRIVE VERIFICATION")
print("-" * 100)

# Detect whether the canonical MyDrive mount is already usable.
drive_mounted = (
    MYDRIVE_ROOT.exists()
    and MYDRIVE_ROOT.is_dir()
)

if not drive_mounted:

    print("Google Drive MyDrive is not currently mounted.")
    print("Attempting to mount Google Drive...")

    try:

        from google.colab import drive

        drive.mount(
            str(DRIVE_ROOT),
            force_remount=False,
        )

    except Exception as exc:

        raise RuntimeError(
            "Google Drive mounting failed. "
            "Please authenticate the Google Drive mount and rerun Section 2."
        ) from exc


# --------------------------------------------------------------------------------------------------
# 3. Validate Actual MyDrive Mount
# --------------------------------------------------------------------------------------------------

if not MYDRIVE_ROOT.exists():

    raise FileNotFoundError(
        f"Google Drive MyDrive not available after mount: "
        f"{MYDRIVE_ROOT}"
    )

if not MYDRIVE_ROOT.is_dir():

    raise RuntimeError(
        f"MyDrive path exists but is not a directory: "
        f"{MYDRIVE_ROOT}"
    )


# Verify that the mount is writable without creating project artifacts.
drive_probe_path = (
    MYDRIVE_ROOT
    / ".spp_gan_notebook12_mount_probe"
)

try:

    drive_probe_path.write_text(
        "SPP-GAN Notebook 12 Drive verification",
        encoding="utf-8",
    )

    if not drive_probe_path.exists():

        raise RuntimeError(
            "Google Drive write verification failed."
        )

    drive_probe_path.unlink()

except Exception as exc:

    raise RuntimeError(
        "Google Drive MyDrive is visible but failed the "
        "write-access verification."
    ) from exc


print(
    f"✓ MyDrive        : {MYDRIVE_ROOT}"
)

print(
    "✓ Google Drive mount verified"
)

print(
    "✓ Google Drive write access verified"
)


# --------------------------------------------------------------------------------------------------
# 4. Verify Canonical Project Root
# --------------------------------------------------------------------------------------------------

if not PROJECT_ROOT.exists():

    raise FileNotFoundError(
        f"SРP-GAN project root not found: "
        f"{PROJECT_ROOT}"
    )

if not PROJECT_ROOT.is_dir():

    raise RuntimeError(
        f"SРP-GAN project root is not a directory: "
        f"{PROJECT_ROOT}"
    )


print(
    f"✓ Project root   : {PROJECT_ROOT}"
)


# --------------------------------------------------------------------------------------------------
# 5. Canonical Notebook Paths
# --------------------------------------------------------------------------------------------------

NB08_ROOT = (
    PROJECT_ROOT
    / "results"
    / "notebooks"
    / "notebook_08"
)

NB09_ROOT = (
    PROJECT_ROOT
    / "results"
    / "notebooks"
    / "notebook_09"
)

NB10_ROOT = (
    PROJECT_ROOT
    / "results"
    / "notebooks"
    / "notebook_10"
)

NB11_ROOT = (
    PROJECT_ROOT
    / "results"
    / "notebooks"
    / "notebook_11"
)

NB12_ROOT = (
    PROJECT_ROOT
    / "results"
    / "notebooks"
    / "notebook_12"
)


# --------------------------------------------------------------------------------------------------
# 6. Canonical Persisted Configuration Artifacts
# --------------------------------------------------------------------------------------------------

NB09_CONFIG_PATH = (
    NB09_ROOT
    / "configuration"
    / "sppgan_statistical_guidance_configuration.json"
)

NB10_PRIVACY_CONFIG_PATH = (
    NB10_ROOT
    / "configuration"
    / "sppgan_privacy_configuration.json"
)

NB10_PRIVACY_METADATA_PATH = (
    NB10_ROOT
    / "metadata"
    / "sppgan_privacy_metadata.csv"
)

NB11_ACCOUNTING_CONFIG_PATH = (
    NB11_ROOT
    / "configuration"
    / "sppgan_privacy_accounting_configuration.json"
)

NB11_CONFIGURED_ACCOUNTING_PATH = (
    NB11_ROOT
    / "accounting"
    / "sppgan_configured_schedule_rdp_accounting.csv"
)


# --------------------------------------------------------------------------------------------------
# 7. Verify Required Notebook Roots
# --------------------------------------------------------------------------------------------------

REQUIRED_NOTEBOOK_ROOTS = {
    "Notebook 08": NB08_ROOT,
    "Notebook 09": NB09_ROOT,
    "Notebook 10": NB10_ROOT,
    "Notebook 11": NB11_ROOT,
    "Notebook 12": NB12_ROOT,
}

print()
print("-" * 100)
print("CANONICAL NOTEBOOK ROOTS")
print("-" * 100)

for notebook_name, notebook_path in REQUIRED_NOTEBOOK_ROOTS.items():

    if not notebook_path.exists():

        raise FileNotFoundError(
            f"{notebook_name} root not found:\n"
            f"{notebook_path}"
        )

    if not notebook_path.is_dir():

        raise RuntimeError(
            f"{notebook_name} root is not a directory:\n"
            f"{notebook_path}"
        )

    print(
        f"✓ {notebook_name:<18}: {notebook_path}"
    )


# --------------------------------------------------------------------------------------------------
# 8. Required Artifact Verification
# --------------------------------------------------------------------------------------------------

REQUIRED_CONFIGURATION_ARTIFACTS = {

    "Notebook 09 statistical guidance configuration":
        NB09_CONFIG_PATH,

    "Notebook 10 privacy configuration":
        NB10_PRIVACY_CONFIG_PATH,

    "Notebook 10 privacy metadata":
        NB10_PRIVACY_METADATA_PATH,

    "Notebook 11 privacy accounting configuration":
        NB11_ACCOUNTING_CONFIG_PATH,

    "Notebook 11 configured RDP accounting":
        NB11_CONFIGURED_ACCOUNTING_PATH,
}


print()
print("-" * 100)
print("REQUIRED PERSISTED CONFIGURATION ARTIFACTS")
print("-" * 100)

for artifact_name, artifact_path in (
    REQUIRED_CONFIGURATION_ARTIFACTS.items()
):

    if not artifact_path.exists():

        raise FileNotFoundError(
            "Required persisted artifact not found:\n"
            f"{artifact_name}: {artifact_path}"
        )

    if not artifact_path.is_file():

        raise RuntimeError(
            f"Required artifact is not a file:\n"
            f"{artifact_name}: {artifact_path}"
        )

    print(
        f"✓ {artifact_name:<45}: "
        f"{artifact_path}"
    )


# --------------------------------------------------------------------------------------------------
# 9. Utility Functions
# --------------------------------------------------------------------------------------------------

def load_json_file(
    path,
):

    with open(
        path,
        "r",
        encoding="utf-8",
    ) as handle:

        payload = json.load(
            handle
        )

    if not isinstance(
        payload,
        dict,
    ):

        raise TypeError(
            f"Expected JSON object at {path}, "
            f"found {type(payload).__name__}."
        )

    return payload


def sha256_file(
    path,
    chunk_size=1024 * 1024,
):

    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as handle:

        while True:

            chunk = handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def normalize_key(
    key,
):

    return (
        str(key)
        .strip()
        .lower()
        .replace(
            "-",
            "_",
        )
        .replace(
            " ",
            "_",
        )
    )


def find_key_occurrences(
    obj,
    candidate_keys,
    path="",
):

    occurrences = []

    if isinstance(
        obj,
        dict,
    ):

        for key, value in obj.items():

            current_path = (
                f"{path}.{key}"
                if path
                else str(key)
            )

            if normalize_key(
                key
            ) in candidate_keys:

                occurrences.append(
                    {
                        "path":
                            current_path,

                        "value":
                            value,
                    }
                )

            occurrences.extend(
                find_key_occurrences(
                    value,
                    candidate_keys,
                    current_path,
                )
            )

    elif isinstance(
        obj,
        list,
    ):

        for index, value in enumerate(
            obj
        ):

            current_path = (
                f"{path}[{index}]"
            )

            occurrences.extend(
                find_key_occurrences(
                    value,
                    candidate_keys,
                    current_path,
                )
            )

    return occurrences


def canonicalize_configuration_value(
    parameter_name,
    value,
):

    if not isinstance(
        value,
        str,
    ):

        return value

    normalized = (
        value
        .strip()
        .lower()
    )

    # ----------------------------------------------------------------------------------------------
    # Accountant
    # ----------------------------------------------------------------------------------------------

    if parameter_name == "accountant":

        aliases = {

            "rdp":
                "rdp",

            "renyi_differential_privacy":
                "rdp",

            "rényi_differential_privacy":
                "rdp",
        }

        return aliases.get(
            normalized,
            normalized,
        )

    # ----------------------------------------------------------------------------------------------
    # Sampling mechanism
    # ----------------------------------------------------------------------------------------------

    if parameter_name == "sampling mechanism":

        aliases = {

            "poisson":
                "poisson",

            "poisson_sampling":
                "poisson",

            "poisson_sampling_mechanism":
                "poisson",
        }

        return aliases.get(
            normalized,
            normalized,
        )

    # ----------------------------------------------------------------------------------------------
    # Clipping mechanism
    # ----------------------------------------------------------------------------------------------

    if parameter_name == "clipping mechanism":

        aliases = {

            "flat":
                "flat",

            "flat_l2":
                "flat",

            "flat l2":
                "flat",

            "l2":
                "flat",

            "l2_flat":
                "flat",

            "flat_l2_clipping":
                "flat",
        }

        return aliases.get(
            normalized,
            normalized,
        )

    # ----------------------------------------------------------------------------------------------
    # Loss reduction
    # ----------------------------------------------------------------------------------------------

    if parameter_name == "loss reduction":

        aliases = {

            "mean":
                "mean",

            "average":
                "mean",
        }

        return aliases.get(
            normalized,
            normalized,
        )

    return normalized


def resolve_global_parameter(
    config,
    candidate_keys,
    parameter_name,
    value_type="numeric",
):

    normalized_candidates = {
        normalize_key(
            key
        )
        for key in candidate_keys
    }

    occurrences = find_key_occurrences(
        config,
        normalized_candidates,
    )

    if not occurrences:

        raise KeyError(
            f"Notebook 10 configuration does not expose "
            f"required parameter '{parameter_name}'."
        )

    values = []

    for occurrence in occurrences:

        value = occurrence[
            "value"
        ]

        # ------------------------------------------------------------------------------------------
        # Numeric
        # ------------------------------------------------------------------------------------------

        if value_type == "numeric":

            if isinstance(
                value,
                bool,
            ):

                continue

            if isinstance(
                value,
                (
                    int,
                    float,
                    np.integer,
                    np.floating,
                ),
            ):

                numeric_value = float(
                    value
                )

                if np.isfinite(
                    numeric_value
                ):

                    values.append(
                        (
                            occurrence[
                                "path"
                            ],
                            numeric_value,
                        )
                    )

        # ------------------------------------------------------------------------------------------
        # Boolean
        # ------------------------------------------------------------------------------------------

        elif value_type == "boolean":

            if isinstance(
                value,
                bool,
            ):

                values.append(
                    (
                        occurrence[
                            "path"
                        ],
                        value,
                    )
                )

        # ------------------------------------------------------------------------------------------
        # String / enumeration
        # ------------------------------------------------------------------------------------------

        elif value_type == "string":

            if isinstance(
                value,
                str,
            ):

                canonical_value = (
                    canonicalize_configuration_value(
                        parameter_name,
                        value,
                    )
                )

                if canonical_value:

                    values.append(
                        (
                            occurrence[
                                "path"
                            ],
                            canonical_value,
                        )
                    )

        else:

            raise ValueError(
                f"Unsupported value_type={value_type!r} "
                f"for parameter '{parameter_name}'."
            )

    if not values:

        raise KeyError(
            f"No valid persisted value found for "
            f"'{parameter_name}'."
        )

    unique_values = []

    for _, value in values:

        if value not in unique_values:

            unique_values.append(
                value
            )

    if len(
        unique_values
    ) != 1:

        raise RuntimeError(
            f"Conflicting Notebook 10 configuration values "
            f"for '{parameter_name}':\n"
            +
            "\n".join(
                f"  {path} = {value}"
                for path, value in values
            )
        )

    resolved_value = unique_values[
        0
    ]

    print(
        f"✓ {parameter_name:<30}: "
        f"{resolved_value}"
    )

    if len(
        values
    ) > 1:

        print(
            f"  Validated occurrences             : "
            f"{len(values)}"
        )

    return resolved_value


# --------------------------------------------------------------------------------------------------
# 10. Load Persisted Configurations
# --------------------------------------------------------------------------------------------------

NB09_STATISTICAL_GUIDANCE_CONFIG = load_json_file(
    NB09_CONFIG_PATH
)

NB10_PRIVACY_CONFIG = load_json_file(
    NB10_PRIVACY_CONFIG_PATH
)

NB11_ACCOUNTING_CONFIG = load_json_file(
    NB11_ACCOUNTING_CONFIG_PATH
)

NB10_PRIVACY_METADATA_DF = pd.read_csv(
    NB10_PRIVACY_METADATA_PATH
)

NB11_CONFIGURED_ACCOUNTING_DF = pd.read_csv(
    NB11_CONFIGURED_ACCOUNTING_PATH
)


# --------------------------------------------------------------------------------------------------
# 11. Validate Notebook 09 Statistical Guidance Configuration
# --------------------------------------------------------------------------------------------------

print()
print("Notebook 09 statistical guidance:")

NB09_LAMBDA_STAT = None

NB09_LAMBDA_OCCURRENCES = find_key_occurrences(
    NB09_STATISTICAL_GUIDANCE_CONFIG,
    {
        "lambda_stat",
        "λ_stat",
    },
)

for occurrence in NB09_LAMBDA_OCCURRENCES:

    value = occurrence[
        "value"
    ]

    if isinstance(
        value,
        (
            int,
            float,
            np.integer,
            np.floating,
        ),
    ):

        value = float(
            value
        )

        if np.isfinite(
            value
        ):

            if NB09_LAMBDA_STAT is None:

                NB09_LAMBDA_STAT = value

            elif not np.isclose(
                NB09_LAMBDA_STAT,
                value,
                rtol=0.0,
                atol=1e-12,
            ):

                raise RuntimeError(
                    "Conflicting Notebook 09 λ_stat values."
                )

if NB09_LAMBDA_STAT is None:

    raise KeyError(
        "λ_stat was not found in persisted Notebook 09 "
        "statistical guidance configuration."
    )

if NB09_LAMBDA_STAT < 0:

    raise ValueError(
        f"Invalid λ_stat={NB09_LAMBDA_STAT}."
    )

print(
    f"  λ_stat : {NB09_LAMBDA_STAT:.6f}"
)

print(
    "  Source : persisted Notebook 09 configuration"
)


# --------------------------------------------------------------------------------------------------
# 12. Validate Notebook 10 Privacy Metadata Schema
# --------------------------------------------------------------------------------------------------

REQUIRED_NB10_METADATA_COLUMNS = [

    "dataset",
    "n_train",
    "target_epsilon",
    "delta",
    "batch_size",
    "sample_rate",
    "epochs",
    "nominal_steps_per_epoch",
    "nominal_total_steps",
    "max_grad_norm",
    "noise_multiplier",
    "accountant",
    "sampling",
    "clipping",
    "loss_reduction",
    "protected_component",
    "per_example_gradients",
    "generator_private",
    "statistical_guidance_private",
    "preprocessing_private",
    "achieved_epsilon",
    "achieved_epsilon_status",
    "training_status",
    "synthetic_generation_status",
    "end_to_end_privacy_claim",
    "status",
]

missing_nb10_columns = [
    column
    for column in REQUIRED_NB10_METADATA_COLUMNS
    if column not in NB10_PRIVACY_METADATA_DF.columns
]

if missing_nb10_columns:

    raise RuntimeError(
        "Notebook 10 privacy metadata is missing required columns:\n"
        +
        "\n".join(
            f"  - {column}"
            for column in missing_nb10_columns
        )
    )


if len(
    NB10_PRIVACY_METADATA_DF
) != 3:

    raise RuntimeError(
        "Notebook 10 privacy metadata must contain exactly "
        f"3 canonical dataset rows. "
        f"Observed: {len(NB10_PRIVACY_METADATA_DF)}"
    )


DATASET_IDS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]


if set(
    NB10_PRIVACY_METADATA_DF[
        "dataset"
    ].astype(
        str
    )
) != set(
    DATASET_IDS
):

    raise RuntimeError(
        "Notebook 10 metadata dataset registry does not match "
        "the canonical SPP-GAN dataset registry."
    )


print()
print("Notebook 10 privacy metadata:")

print(
    f"  Rows    : {len(NB10_PRIVACY_METADATA_DF)}"
)

print(
    f"  Columns : {list(NB10_PRIVACY_METADATA_DF.columns)}"
)


# --------------------------------------------------------------------------------------------------
# 13. Expected Training Dataset Sizes
# --------------------------------------------------------------------------------------------------

EXPECTED_TRAIN_ROWS = {

    "adult_income":
        34189,

    "bank_marketing":
        31647,

    "diabetes_130us":
        71236,
}


for dataset_id in DATASET_IDS:

    if dataset_id not in EXPECTED_TRAIN_ROWS:

        raise RuntimeError(
            f"Missing expected training-row definition "
            f"for {dataset_id}."
        )


# --------------------------------------------------------------------------------------------------
# 14. Resolve Notebook 10 Global Privacy Parameters
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)
print("NOTEBOOK 10 PRIVACY PARAMETERS")
print("-" * 100)


DP_BATCH_SIZE = int(
    resolve_global_parameter(
        NB10_PRIVACY_CONFIG,
        [
            "dp_batch_size",
            "batch_size",
        ],
        "DP batch size",
        value_type="numeric",
    )
)


TARGET_EPSILON = float(
    resolve_global_parameter(
        NB10_PRIVACY_CONFIG,
        [
            "target_epsilon",
        ],
        "target epsilon",
        value_type="numeric",
    )
)


MAX_GRAD_NORM = float(
    resolve_global_parameter(
        NB10_PRIVACY_CONFIG,
        [
            "max_grad_norm",
            "dp_max_grad_norm",
        ],
        "maximum gradient norm",
        value_type="numeric",
    )
)


DP_EPOCHS = int(
    resolve_global_parameter(
        NB10_PRIVACY_CONFIG,
        [
            "dp_epochs",
            "epochs",
            "training_epochs",
        ],
        "training epochs",
        value_type="numeric",
    )
)


ACCOUNTANT = resolve_global_parameter(
    NB10_PRIVACY_CONFIG,
    [
        "accountant",
    ],
    "accountant",
    value_type="string",
)


SAMPLING_MECHANISM = resolve_global_parameter(
    NB10_PRIVACY_CONFIG,
    [
        "sampling_mechanism",
        "sampling",
    ],
    "sampling mechanism",
    value_type="string",
)


CLIPPING_MECHANISM = resolve_global_parameter(
    NB10_PRIVACY_CONFIG,
    [
        "clipping_mechanism",
        "clipping",
    ],
    "clipping mechanism",
    value_type="string",
)


LOSS_REDUCTION = resolve_global_parameter(
    NB10_PRIVACY_CONFIG,
    [
        "loss_reduction",
    ],
    "loss reduction",
    value_type="string",
)


# --------------------------------------------------------------------------------------------------
# 15. Resolve Privacy Boundary
# --------------------------------------------------------------------------------------------------

GENERATOR_PRIVATE = resolve_global_parameter(
    NB10_PRIVACY_CONFIG,
    [
        "generator_private",
    ],
    "generator_private",
    value_type="boolean",
)


STATISTICAL_GUIDANCE_PRIVATE = resolve_global_parameter(
    NB10_PRIVACY_CONFIG,
    [
        "statistical_guidance_private",
    ],
    "statistical_guidance_private",
    value_type="boolean",
)


PREPROCESSING_PRIVATE = resolve_global_parameter(
    NB10_PRIVACY_CONFIG,
    [
        "preprocessing_private",
    ],
    "preprocessing_private",
    value_type="boolean",
)


END_TO_END_PRIVACY_CLAIM = resolve_global_parameter(
    NB10_PRIVACY_CONFIG,
    [
        "end_to_end_privacy_claim",
    ],
    "end_to_end_privacy_claim",
    value_type="boolean",
)


# --------------------------------------------------------------------------------------------------
# 16. Validate Global Privacy Configuration
# --------------------------------------------------------------------------------------------------

if DP_BATCH_SIZE <= 0:

    raise ValueError(
        "DP batch size must be positive."
    )

if DP_EPOCHS <= 0:

    raise ValueError(
        "DP epochs must be positive."
    )

if TARGET_EPSILON <= 0:

    raise ValueError(
        "Target epsilon must be positive."
    )

if MAX_GRAD_NORM <= 0:

    raise ValueError(
        "Maximum gradient norm must be positive."
    )

if ACCOUNTANT != "rdp":

    raise RuntimeError(
        f"Unexpected accountant: {ACCOUNTANT}. "
        "Expected canonical value: rdp."
    )

if SAMPLING_MECHANISM != "poisson":

    raise RuntimeError(
        f"Unexpected sampling mechanism: "
        f"{SAMPLING_MECHANISM}. "
        "Expected canonical value: poisson."
    )

if CLIPPING_MECHANISM != "flat":

    raise RuntimeError(
        f"Unexpected clipping mechanism: "
        f"{CLIPPING_MECHANISM}. "
        "Expected canonical value: flat."
    )

if LOSS_REDUCTION != "mean":

    raise RuntimeError(
        f"Unexpected loss reduction: "
        f"{LOSS_REDUCTION}. "
        "Expected canonical value: mean."
    )


# --------------------------------------------------------------------------------------------------
# 17. Validate Privacy Boundary
# --------------------------------------------------------------------------------------------------

if GENERATOR_PRIVATE is not False:

    raise RuntimeError(
        "Notebook 10 declares generator_private=False. "
        "Notebook 12 cannot silently change this boundary."
    )

if STATISTICAL_GUIDANCE_PRIVATE is not False:

    raise RuntimeError(
        "Notebook 10 declares statistical_guidance_private=False. "
        "Notebook 12 cannot silently change this boundary."
    )

if PREPROCESSING_PRIVATE is not False:

    raise RuntimeError(
        "Notebook 10 declares preprocessing_private=False. "
        "Notebook 12 cannot silently change this boundary."
    )

if END_TO_END_PRIVACY_CLAIM is not False:

    raise RuntimeError(
        "Notebook 10 declares end_to_end_privacy_claim=False. "
        "Notebook 12 cannot claim end-to-end DP."
    )


# --------------------------------------------------------------------------------------------------
# 18. Validate Dataset-Level Privacy Metadata
# --------------------------------------------------------------------------------------------------

PRIVACY_PARAMETERS = {}

for dataset_id in DATASET_IDS:

    matching_rows = NB10_PRIVACY_METADATA_DF[
        NB10_PRIVACY_METADATA_DF[
            "dataset"
        ].astype(
            str
        )
        ==
        dataset_id
    ]

    if len(
        matching_rows
    ) != 1:

        raise RuntimeError(
            f"{dataset_id}: expected exactly one Notebook 10 "
            f"privacy metadata row; observed {len(matching_rows)}."
        )

    row = matching_rows.iloc[
        0
    ]

    n_train = int(
        row[
            "n_train"
        ]
    )

    target_epsilon = float(
        row[
            "target_epsilon"
        ]
    )

    delta = float(
        row[
            "delta"
        ]
    )

    batch_size = int(
        row[
            "batch_size"
        ]
    )

    sample_rate = float(
        row[
            "sample_rate"
        ]
    )

    epochs = int(
        row[
            "epochs"
        ]
    )

    max_grad_norm = float(
        row[
            "max_grad_norm"
        ]
    )

    noise_multiplier = float(
        row[
            "noise_multiplier"
        ]
    )

    metadata_accountant = (
        canonicalize_configuration_value(
            "accountant",
            str(
                row[
                    "accountant"
                ]
            ),
        )
    )

    metadata_sampling = (
        canonicalize_configuration_value(
            "sampling mechanism",
            str(
                row[
                    "sampling"
                ]
            ),
        )
    )

    metadata_clipping = (
        canonicalize_configuration_value(
            "clipping mechanism",
            str(
                row[
                    "clipping"
                ]
            ),
        )
    )

    metadata_loss_reduction = (
        canonicalize_configuration_value(
            "loss reduction",
            str(
                row[
                    "loss_reduction"
                ]
            ),
        )
    )

    # ----------------------------------------------------------------------------------------------
    # Dataset identity
    # ----------------------------------------------------------------------------------------------

    if n_train != EXPECTED_TRAIN_ROWS[
        dataset_id
    ]:

        raise RuntimeError(
            f"{dataset_id}: training-row mismatch.\n"
            f"Expected : {EXPECTED_TRAIN_ROWS[dataset_id]}\n"
            f"Observed : {n_train}"
        )

    # ----------------------------------------------------------------------------------------------
    # Batch size
    # ----------------------------------------------------------------------------------------------

    if batch_size != DP_BATCH_SIZE:

        raise RuntimeError(
            f"{dataset_id}: batch-size mismatch.\n"
            f"Configuration : {DP_BATCH_SIZE}\n"
            f"Metadata      : {batch_size}"
        )

    # ----------------------------------------------------------------------------------------------
    # Poisson sampling rate
    # ----------------------------------------------------------------------------------------------

    expected_sample_rate = (
        float(
            DP_BATCH_SIZE
        )
        /
        float(
            n_train
        )
    )

    if not np.isclose(
        sample_rate,
        expected_sample_rate,
        rtol=1e-6,
        atol=1e-12,
    ):

        raise RuntimeError(
            f"{dataset_id}: sample-rate mismatch.\n"
            f"Persisted q : {sample_rate}\n"
            f"Expected q  : {expected_sample_rate}"
        )

    # ----------------------------------------------------------------------------------------------
    # Epochs
    # ----------------------------------------------------------------------------------------------

    if epochs != DP_EPOCHS:

        raise RuntimeError(
            f"{dataset_id}: epoch mismatch.\n"
            f"Configuration : {DP_EPOCHS}\n"
            f"Metadata      : {epochs}"
        )

    # ----------------------------------------------------------------------------------------------
    # Target epsilon
    # ----------------------------------------------------------------------------------------------

    if not np.isclose(
        target_epsilon,
        TARGET_EPSILON,
        rtol=0.0,
        atol=1e-12,
    ):

        raise RuntimeError(
            f"{dataset_id}: target epsilon mismatch.\n"
            f"Configuration : {TARGET_EPSILON}\n"
            f"Metadata      : {target_epsilon}"
        )

    # ----------------------------------------------------------------------------------------------
    # Maximum gradient norm
    # ----------------------------------------------------------------------------------------------

    if not np.isclose(
        max_grad_norm,
        MAX_GRAD_NORM,
        rtol=0.0,
        atol=1e-12,
    ):

        raise RuntimeError(
            f"{dataset_id}: maximum gradient norm mismatch.\n"
            f"Configuration : {MAX_GRAD_NORM}\n"
            f"Metadata      : {max_grad_norm}"
        )

    # ----------------------------------------------------------------------------------------------
    # Accountant
    # ----------------------------------------------------------------------------------------------

    if metadata_accountant != ACCOUNTANT:

        raise RuntimeError(
            f"{dataset_id}: accountant mismatch.\n"
            f"Configuration : {ACCOUNTANT}\n"
            f"Metadata      : {metadata_accountant}"
        )

    # ----------------------------------------------------------------------------------------------
    # Sampling
    # ----------------------------------------------------------------------------------------------

    if metadata_sampling != SAMPLING_MECHANISM:

        raise RuntimeError(
            f"{dataset_id}: sampling mechanism mismatch.\n"
            f"Configuration : {SAMPLING_MECHANISM}\n"
            f"Metadata      : {metadata_sampling}"
        )

    # ----------------------------------------------------------------------------------------------
    # Clipping
    # ----------------------------------------------------------------------------------------------

    if metadata_clipping != CLIPPING_MECHANISM:

        raise RuntimeError(
            f"{dataset_id}: clipping mechanism mismatch.\n"
            f"Configuration : {CLIPPING_MECHANISM}\n"
            f"Metadata      : {metadata_clipping}"
        )

    # ----------------------------------------------------------------------------------------------
    # Loss reduction
    # ----------------------------------------------------------------------------------------------

    if metadata_loss_reduction != LOSS_REDUCTION:

        raise RuntimeError(
            f"{dataset_id}: loss-reduction mismatch.\n"
            f"Configuration : {LOSS_REDUCTION}\n"
            f"Metadata      : {metadata_loss_reduction}"
        )

    # ----------------------------------------------------------------------------------------------
    # Delta
    # ----------------------------------------------------------------------------------------------

    if not (
        np.isfinite(
            delta
        )
        and
        0 < delta < 1
    ):

        raise ValueError(
            f"{dataset_id}: invalid delta={delta}."
        )

    # ----------------------------------------------------------------------------------------------
    # Noise multiplier
    # ----------------------------------------------------------------------------------------------

    if not (
        np.isfinite(
            noise_multiplier
        )
        and
        noise_multiplier > 0
    ):

        raise ValueError(
            f"{dataset_id}: invalid noise multiplier="
            f"{noise_multiplier}."
        )

    PRIVACY_PARAMETERS[
        dataset_id
    ] = {

        "n_train":
            n_train,

        "target_epsilon":
            target_epsilon,

        "delta":
            delta,

        "batch_size":
            batch_size,

        "poisson_sample_rate":
            sample_rate,

        "epochs":
            epochs,

        "max_grad_norm":
            max_grad_norm,

        "noise_multiplier":
            noise_multiplier,

        "accountant":
            metadata_accountant,

        "sampling":
            metadata_sampling,

        "clipping":
            metadata_clipping,

        "loss_reduction":
            metadata_loss_reduction,
    }


# --------------------------------------------------------------------------------------------------
# 19. Validate Notebook 11 Configured-Schedule Accounting
# --------------------------------------------------------------------------------------------------

REQUIRED_NB11_ACCOUNTING_COLUMNS = [

    "dataset",
    "configured_schedule_epsilon",
]

missing_nb11_columns = [
    column
    for column in REQUIRED_NB11_ACCOUNTING_COLUMNS
    if column not in NB11_CONFIGURED_ACCOUNTING_DF.columns
]

if missing_nb11_columns:

    raise RuntimeError(
        "Notebook 11 configured accounting is missing required columns:\n"
        +
        "\n".join(
            f"  - {column}"
            for column in missing_nb11_columns
        )
    )


if set(
    NB11_CONFIGURED_ACCOUNTING_DF[
        "dataset"
    ].astype(
        str
    )
) != set(
    DATASET_IDS
):

    raise RuntimeError(
        "Notebook 11 configured-schedule accounting dataset registry "
        "does not match the canonical dataset registry."
    )


for dataset_id in DATASET_IDS:

    accounting_rows = (
        NB11_CONFIGURED_ACCOUNTING_DF[
            NB11_CONFIGURED_ACCOUNTING_DF[
                "dataset"
            ].astype(
                str
            )
            ==
            dataset_id
        ]
    )

    if len(
        accounting_rows
    ) != 1:

        raise RuntimeError(
            f"{dataset_id}: expected exactly one Notebook 11 "
            f"configured accounting row; observed "
            f"{len(accounting_rows)}."
        )

    configured_epsilon = float(
        accounting_rows.iloc[
            0
        ][
            "configured_schedule_epsilon"
        ]
    )

    if not (
        np.isfinite(
            configured_epsilon
        )
        and
        configured_epsilon > 0
    ):

        raise RuntimeError(
            f"{dataset_id}: invalid configured-schedule "
            f"epsilon={configured_epsilon}."
        )

    PRIVACY_PARAMETERS[
        dataset_id
    ][
        "configured_schedule_epsilon"
    ] = configured_epsilon


# --------------------------------------------------------------------------------------------------
# 20. Training Configuration
# --------------------------------------------------------------------------------------------------
# Checkpoint policy:
#   • checkpoint only after a completed epoch
#   • automatic resume from latest valid checkpoint
#   • optimizer/privacy-accountant/RNG/training-history state preserved
# --------------------------------------------------------------------------------------------------

CHECKPOINT_INTERVAL = 10
FORCE_RESTART = False
MAX_PERIODIC_CHECKPOINTS = 5


if CHECKPOINT_INTERVAL != 10:

    raise RuntimeError(
        "SPP-GAN checkpoint interval must be exactly 10 epochs."
    )

if MAX_PERIODIC_CHECKPOINTS <= 0:

    raise RuntimeError(
        "MAX_PERIODIC_CHECKPOINTS must be positive."
    )


# --------------------------------------------------------------------------------------------------
# 21. Cross-Artifact Training Configuration Summary
# --------------------------------------------------------------------------------------------------

TRAINING_CONFIGURATION = {

    "framework":
        "SPP-GAN",

    "lambda_stat":
        float(
            NB09_LAMBDA_STAT
        ),

    "dp_batch_size":
        int(
            DP_BATCH_SIZE
        ),

    "target_epsilon":
        float(
            TARGET_EPSILON
        ),

    "max_grad_norm":
        float(
            MAX_GRAD_NORM
        ),

    "epochs":
        int(
            DP_EPOCHS
        ),

    "accountant":
        ACCOUNTANT,

    "sampling_mechanism":
        SAMPLING_MECHANISM,

    "clipping_mechanism":
        CLIPPING_MECHANISM,

    "loss_reduction":
        LOSS_REDUCTION,

    "generator_private":
        bool(
            GENERATOR_PRIVATE
        ),

    "statistical_guidance_private":
        bool(
            STATISTICAL_GUIDANCE_PRIVATE
        ),

    "preprocessing_private":
        bool(
            PREPROCESSING_PRIVATE
        ),

    "end_to_end_privacy_claim":
        bool(
            END_TO_END_PRIVACY_CLAIM
        ),

    "checkpoint_interval":
        int(
            CHECKPOINT_INTERVAL
        ),

    "force_restart":
        bool(
            FORCE_RESTART
        ),

    "max_periodic_checkpoints":
        int(
            MAX_PERIODIC_CHECKPOINTS
        ),
}


# --------------------------------------------------------------------------------------------------
# 22. Persisted Source Artifact Fingerprints
# --------------------------------------------------------------------------------------------------

CONFIGURATION_SOURCE_ARTIFACTS = {

    "nb09_statistical_guidance_configuration":
        NB09_CONFIG_PATH,

    "nb10_privacy_configuration":
        NB10_PRIVACY_CONFIG_PATH,

    "nb10_privacy_metadata":
        NB10_PRIVACY_METADATA_PATH,

    "nb11_accounting_configuration":
        NB11_ACCOUNTING_CONFIG_PATH,

    "nb11_configured_rdp_accounting":
        NB11_CONFIGURED_ACCOUNTING_PATH,
}

CONFIGURATION_SOURCE_HASHES = {}

for artifact_name, artifact_path in (
    CONFIGURATION_SOURCE_ARTIFACTS.items()
):

    CONFIGURATION_SOURCE_HASHES[
        artifact_name
    ] = sha256_file(
        artifact_path
    )


# --------------------------------------------------------------------------------------------------
# 23. Configuration Snapshot
# --------------------------------------------------------------------------------------------------

NB12_CONFIG_ROOT = (
    NB12_ROOT
    / "configuration"
)

NB12_CONFIG_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

NB12_CONFIGURATION_SNAPSHOT_PATH = (
    NB12_CONFIG_ROOT
    / "sppgan_training_configuration.json"
)

NB12_CONFIGURATION_SNAPSHOT = {

    "framework":
        "SPP-GAN",

    "source_notebooks": {

        "notebook_09":
            str(
                NB09_CONFIG_PATH
            ),

        "notebook_10":
            str(
                NB10_PRIVACY_CONFIG_PATH
            ),

        "notebook_10_metadata":
            str(
                NB10_PRIVACY_METADATA_PATH
            ),

        "notebook_11":
            str(
                NB11_ACCOUNTING_CONFIG_PATH
            ),

        "notebook_11_configured_accounting":
            str(
                NB11_CONFIGURED_ACCOUNTING_PATH
            ),
    },

    "training_configuration":
        TRAINING_CONFIGURATION,

    "dataset_privacy_parameters":
        PRIVACY_PARAMETERS,

    "source_artifact_sha256":
        CONFIGURATION_SOURCE_HASHES,

    "privacy_boundary": {

        "protected_component":
            "critic",

        "generator_private":
            bool(
                GENERATOR_PRIVATE
            ),

        "statistical_guidance_private":
            bool(
                STATISTICAL_GUIDANCE_PRIVATE
            ),

        "preprocessing_private":
            bool(
                PREPROCESSING_PRIVATE
            ),

        "end_to_end_privacy_claim":
            bool(
                END_TO_END_PRIVACY_CLAIM
            ),

        "achieved_training_epsilon":
            "deferred_to_notebook_12_training_accountant",
    },
}

with open(
    NB12_CONFIGURATION_SNAPSHOT_PATH,
    "w",
    encoding="utf-8",
) as handle:

    json.dump(
        NB12_CONFIGURATION_SNAPSHOT,
        handle,
        indent=2,
        ensure_ascii=False,
    )


# --------------------------------------------------------------------------------------------------
# 24. Reload Configuration Snapshot
# --------------------------------------------------------------------------------------------------

with open(
    NB12_CONFIGURATION_SNAPSHOT_PATH,
    "r",
    encoding="utf-8",
) as handle:

    reloaded_configuration_snapshot = json.load(
        handle
    )

if reloaded_configuration_snapshot[
    "framework"
] != "SPP-GAN":

    raise RuntimeError(
        "Reloaded Notebook 12 configuration snapshot "
        "has an unexpected framework."
    )

if (
    reloaded_configuration_snapshot[
        "training_configuration"
    ]
    != TRAINING_CONFIGURATION
):

    raise RuntimeError(
        "Reloaded Notebook 12 training configuration "
        "does not match the in-memory configuration."
    )

if (
    reloaded_configuration_snapshot[
        "dataset_privacy_parameters"
    ]
    != PRIVACY_PARAMETERS
):

    raise RuntimeError(
        "Reloaded Notebook 12 privacy parameters "
        "do not match the in-memory configuration."
    )

if (
    reloaded_configuration_snapshot[
        "source_artifact_sha256"
    ]
    != CONFIGURATION_SOURCE_HASHES
):

    raise RuntimeError(
        "Reloaded Notebook 12 source-artifact fingerprints "
        "do not match the in-memory fingerprints."
    )


# --------------------------------------------------------------------------------------------------
# 25. Final Section 2 Validation
# --------------------------------------------------------------------------------------------------

SECTION_2_CHECKS = {

    "drive_mounted":
        MYDRIVE_ROOT.exists(),

    "drive_is_directory":
        MYDRIVE_ROOT.is_dir(),

    "project_root_exists":
        PROJECT_ROOT.exists(),

    "project_root_is_directory":
        PROJECT_ROOT.is_dir(),

    "nb08_root_exists":
        NB08_ROOT.exists(),

    "nb09_root_exists":
        NB09_ROOT.exists(),

    "nb10_root_exists":
        NB10_ROOT.exists(),

    "nb11_root_exists":
        NB11_ROOT.exists(),

    "nb12_root_exists":
        NB12_ROOT.exists(),

    "nb09_configuration_exists":
        NB09_CONFIG_PATH.exists(),

    "nb10_configuration_exists":
        NB10_PRIVACY_CONFIG_PATH.exists(),

    "nb10_metadata_exists":
        NB10_PRIVACY_METADATA_PATH.exists(),

    "nb11_configuration_exists":
        NB11_ACCOUNTING_CONFIG_PATH.exists(),

    "nb11_accounting_exists":
        NB11_CONFIGURED_ACCOUNTING_PATH.exists(),

    "nb09_lambda_stat_valid":
        np.isfinite(
            NB09_LAMBDA_STAT
        )
        and
        NB09_LAMBDA_STAT >= 0,

    "nb10_metadata_three_datasets":
        len(
            NB10_PRIVACY_METADATA_DF
        ) == 3,

    "dp_batch_size_valid":
        DP_BATCH_SIZE > 0,

    "target_epsilon_valid":
        TARGET_EPSILON > 0,

    "max_grad_norm_valid":
        MAX_GRAD_NORM > 0,

    "epochs_valid":
        DP_EPOCHS > 0,

    "accountant_rdp":
        ACCOUNTANT == "rdp",

    "sampling_poisson":
        SAMPLING_MECHANISM == "poisson",

    "clipping_flat":
        CLIPPING_MECHANISM == "flat",

    "loss_reduction_mean":
        LOSS_REDUCTION == "mean",

    "generator_private_false":
        GENERATOR_PRIVATE is False,

    "statistical_guidance_private_false":
        STATISTICAL_GUIDANCE_PRIVATE is False,

    "preprocessing_private_false":
        PREPROCESSING_PRIVATE is False,

    "end_to_end_privacy_claim_false":
        END_TO_END_PRIVACY_CLAIM is False,

    "configured_accounting_three_datasets":
        len(
            NB11_CONFIGURED_ACCOUNTING_DF
        ) == 3,

    "checkpoint_interval_ten":
        CHECKPOINT_INTERVAL == 10,

    "configuration_snapshot_exists":
        NB12_CONFIGURATION_SNAPSHOT_PATH.exists(),

    "configuration_snapshot_reload_valid":
        (
            reloaded_configuration_snapshot[
                "framework"
            ]
            == "SPP-GAN"
        ),
}


FAILED_SECTION_2_CHECKS = [
    name
    for name, passed in SECTION_2_CHECKS.items()
    if not passed
]


print()
print("-" * 100)
print("SECTION 2 VALIDATION")
print("-" * 100)

for check_name, passed in SECTION_2_CHECKS.items():

    print(
        f"{'✓' if passed else '✗'} "
        f"{check_name}"
    )


if FAILED_SECTION_2_CHECKS:

    raise RuntimeError(
        "Section 2 validation failed:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in FAILED_SECTION_2_CHECKS
        )
    )


# --------------------------------------------------------------------------------------------------
# 26. Final Output
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("SECTION 2 — LOAD CONFIGURATION COMPLETE")
print("=" * 100)

print(
    f"✓ λ_stat                    : "
    f"{NB09_LAMBDA_STAT:.6f}"
)

print(
    f"✓ DP batch size             : "
    f"{DP_BATCH_SIZE}"
)

print(
    f"✓ Target epsilon            : "
    f"{TARGET_EPSILON:.6f}"
)

print(
    f"✓ Maximum gradient norm     : "
    f"{MAX_GRAD_NORM:.6f}"
)

print(
    f"✓ Training epochs           : "
    f"{DP_EPOCHS}"
)

print(
    f"✓ Accountant                : "
    f"{ACCOUNTANT}"
)

print(
    f"✓ Sampling mechanism        : "
    f"{SAMPLING_MECHANISM}"
)

print(
    f"✓ Clipping mechanism        : "
    f"{CLIPPING_MECHANISM}"
)

print(
    f"✓ Loss reduction            : "
    f"{LOSS_REDUCTION}"
)

print(
    f"✓ Generator private         : "
    f"{GENERATOR_PRIVATE}"
)

print(
    f"✓ Statistical guidance priv.: "
    f"{STATISTICAL_GUIDANCE_PRIVATE}"
)

print(
    f"✓ Preprocessing private     : "
    f"{PREPROCESSING_PRIVATE}"
)

print(
    f"✓ End-to-end DP claim       : "
    f"{END_TO_END_PRIVACY_CLAIM}"
)

print(
    f"✓ Checkpoint interval       : "
    f"{CHECKPOINT_INTERVAL} epochs"
)

print(
    f"✓ Force restart             : "
    f"{FORCE_RESTART}"
)

print(
    f"✓ Configured accounting     : "
    f"loaded for {len(PRIVACY_PARAMETERS)} datasets"
)

print(
    f"✓ Source fingerprints       : "
    f"{len(CONFIGURATION_SOURCE_HASHES)} artifacts"
)

print(
    f"✓ Configuration snapshot    : "
    f"{NB12_CONFIGURATION_SNAPSHOT_PATH}"
)

print(
    f"✓ Section 2 checks          : "
    f"{len(SECTION_2_CHECKS)}/{len(SECTION_2_CHECKS)} PASS"
)

print()
print("STATUS: PASS — SECTION 2 READY FOR FREEZE")
print("=" * 100)

2. LOAD CONFIGURATION

----------------------------------------------------------------------------------------------------
GOOGLE DRIVE VERIFICATION
----------------------------------------------------------------------------------------------------
✓ MyDrive        : /content/drive/MyDrive
✓ Google Drive mount verified
✓ Google Drive write access verified
✓ Project root   : /content/drive/MyDrive/SPP_GAN_Research

----------------------------------------------------------------------------------------------------
CANONICAL NOTEBOOK ROOTS
----------------------------------------------------------------------------------------------------
✓ Notebook 08       : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08
✓ Notebook 09       : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_09
✓ Notebook 10       : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_10
✓ Notebook 11       : /content/drive/MyDrive/SPP_GAN_Research/results/noteboo

In [3]:
# ==================================================================================================
# SECTION 3 — LOAD PROCESSED TRAINING DATA
# ==================================================================================================

print("=" * 100)
print("3. LOAD PROCESSED TRAINING DATA")
print("=" * 100)

from pathlib import Path
import pandas as pd
import numpy as np


# --------------------------------------------------------------------------------------------------
# 1. Canonical Notebook 02 Artifact Locations
# --------------------------------------------------------------------------------------------------

CANONICAL_NB02_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_02"
)

CANONICAL_NATIVE_ROOT = (
    CANONICAL_NB02_ROOT
    / "native"
)

CANONICAL_PREPROCESSORS_ROOT = (
    CANONICAL_NB02_ROOT
    / "preprocessors"
)

CANONICAL_SCHEMA_ROOT = (
    CANONICAL_NB02_ROOT
    / "schemas"
)

CANONICAL_METADATA_ROOT = (
    CANONICAL_SCHEMA_ROOT
    / "metadata"
)

CANONICAL_NATIVE_MANIFEST = (
    CANONICAL_NATIVE_ROOT
    / "native_dataset_manifest.csv"
)

CANONICAL_PREPROCESSOR_MANIFEST = (
    CANONICAL_SCHEMA_ROOT
    / "preprocessor_manifest.csv"
)


# --------------------------------------------------------------------------------------------------
# 2. Canonical Dataset Registry
# --------------------------------------------------------------------------------------------------

EXPECTED_DATASET_IDS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

TARGET_COLUMNS = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted",
}

PROVENANCE_COLUMN = "__original_row_id__"


if list(DATASET_IDS) != EXPECTED_DATASET_IDS:
    raise RuntimeError(
        "Notebook 12 DATASET_IDS does not match the frozen three-dataset registry.\n"
        f"Expected : {EXPECTED_DATASET_IDS}\n"
        f"Observed : {list(DATASET_IDS)}"
    )


# --------------------------------------------------------------------------------------------------
# 3. Verify Notebook 02 Frozen Artifact Layer
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)
print("NOTEBOOK 02 CANONICAL ARTIFACT VERIFICATION")
print("-" * 100)

REQUIRED_NB02_PATHS = {
    "Notebook 02 root": CANONICAL_NB02_ROOT,
    "Native root": CANONICAL_NATIVE_ROOT,
    "Preprocessor root": CANONICAL_PREPROCESSORS_ROOT,
    "Schema root": CANONICAL_SCHEMA_ROOT,
    "Metadata root": CANONICAL_METADATA_ROOT,
    "Native manifest": CANONICAL_NATIVE_MANIFEST,
    "Preprocessor manifest": CANONICAL_PREPROCESSOR_MANIFEST,
}

for label, path in REQUIRED_NB02_PATHS.items():
    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{label:<25}: {path}"
    )

missing_nb02_paths = [
    str(path)
    for path in REQUIRED_NB02_PATHS.values()
    if not path.exists()
]

if missing_nb02_paths:
    raise FileNotFoundError(
        "Required Notebook 02 persisted artifacts are incomplete.\n\n"
        + "\n".join(
            f"  - {path}"
            for path in missing_nb02_paths
        )
        + "\n\n"
        "Notebook 12 will NOT reconstruct Notebook 02 artifacts."
    )


# --------------------------------------------------------------------------------------------------
# 4. Load Authoritative Native Dataset Manifest
# --------------------------------------------------------------------------------------------------

NATIVE_MANIFEST_DF = pd.read_csv(
    CANONICAL_NATIVE_MANIFEST,
    low_memory=False,
)

if NATIVE_MANIFEST_DF.empty:
    raise RuntimeError(
        "Notebook 02 native_dataset_manifest.csv is empty."
    )

REQUIRED_NATIVE_MANIFEST_COLUMNS = {
    "dataset_id",
    "split",
    "relative_path",
    "absolute_path",
    "rows",
    "columns",
}

missing_manifest_columns = (
    REQUIRED_NATIVE_MANIFEST_COLUMNS
    - set(NATIVE_MANIFEST_DF.columns)
)

if missing_manifest_columns:
    raise RuntimeError(
        "Notebook 02 native manifest is missing required columns:\n"
        + "\n".join(
            f"  - {column}"
            for column in sorted(missing_manifest_columns)
        )
    )

print()
print(
    f"✓ Native manifest loaded : "
    f"{len(NATIVE_MANIFEST_DF)} rows"
)


# --------------------------------------------------------------------------------------------------
# 5. Validate Manifest Dataset/Train Coverage
# --------------------------------------------------------------------------------------------------

manifest_dataset_ids = sorted(
    NATIVE_MANIFEST_DF["dataset_id"]
    .astype(str)
    .str.strip()
    .unique()
)

missing_dataset_ids = sorted(
    set(EXPECTED_DATASET_IDS)
    - set(manifest_dataset_ids)
)

if missing_dataset_ids:
    raise RuntimeError(
        "Notebook 02 native manifest is missing required datasets:\n"
        + "\n".join(
            f"  - {dataset_id}"
            for dataset_id in missing_dataset_ids
        )
    )

TRAIN_MANIFEST_COUNTS = {}

for dataset_id in DATASET_IDS:

    matches = NATIVE_MANIFEST_DF[
        (
            NATIVE_MANIFEST_DF["dataset_id"]
            .astype(str)
            .str.strip()
            == dataset_id
        )
        &
        (
            NATIVE_MANIFEST_DF["split"]
            .astype(str)
            .str.strip()
            .str.lower()
            == "train"
        )
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Notebook 02 native manifest must contain exactly "
            f"one TRAIN record for {dataset_id}. "
            f"Observed: {len(matches)}."
        )

    TRAIN_MANIFEST_COUNTS[dataset_id] = matches.iloc[0]


# --------------------------------------------------------------------------------------------------
# 6. Derive Expected TRAIN Rows and Schema from Manifest
# --------------------------------------------------------------------------------------------------
#
# IMPORTANT:
# TRAIN_ROWS is NOT hard-coded.
#
# The Notebook 02 native manifest is the authoritative persisted source
# for the expected dimensions of the native TRAIN artifacts.
#
# The actual CSV is independently loaded and checked against these values.
# --------------------------------------------------------------------------------------------------

TRAIN_ROWS = {}
TRAIN_COLUMNS = {}

for dataset_id in DATASET_IDS:

    manifest_row = TRAIN_MANIFEST_COUNTS[dataset_id]

    try:
        manifest_rows = int(manifest_row["rows"])
        manifest_columns = int(manifest_row["columns"])
    except (TypeError, ValueError) as exc:
        raise RuntimeError(
            f"{dataset_id}: invalid rows/columns values in "
            "Notebook 02 native manifest."
        ) from exc

    if manifest_rows <= 0:
        raise RuntimeError(
            f"{dataset_id}: manifest TRAIN row count must be positive. "
            f"Observed: {manifest_rows}"
        )

    if manifest_columns <= 0:
        raise RuntimeError(
            f"{dataset_id}: manifest TRAIN column count must be positive. "
            f"Observed: {manifest_columns}"
        )

    TRAIN_ROWS[dataset_id] = manifest_rows
    TRAIN_COLUMNS[dataset_id] = manifest_columns


print()
print("-" * 100)
print("MANIFEST-DERIVED TRAIN SCHEMA")
print("-" * 100)

for dataset_id in DATASET_IDS:
    print(
        f"  {dataset_id:<20} | "
        f"Manifest rows={TRAIN_ROWS[dataset_id]:>8,} | "
        f"Manifest columns={TRAIN_COLUMNS[dataset_id]:>3}"
    )


# --------------------------------------------------------------------------------------------------
# 7. Resolve Canonical TRAIN Files from Native Manifest
# --------------------------------------------------------------------------------------------------

TRAIN_DATA_PATHS = {}
TRAIN_MANIFEST_ROWS = {}

for dataset_id in DATASET_IDS:

    matches = NATIVE_MANIFEST_DF[
        (
            NATIVE_MANIFEST_DF["dataset_id"]
            .astype(str)
            .str.strip()
            == dataset_id
        )
        &
        (
            NATIVE_MANIFEST_DF["split"]
            .astype(str)
            .str.strip()
            .str.lower()
            == "train"
        )
    ]

    if len(matches) != 1:
        raise RuntimeError(
            f"Notebook 02 native manifest must contain exactly "
            f"one TRAIN record for {dataset_id}. "
            f"Observed: {len(matches)}."
        )

    manifest_row = matches.iloc[0]

    TRAIN_MANIFEST_ROWS[dataset_id] = manifest_row

    absolute_path_value = str(
        manifest_row["absolute_path"]
    ).strip()

    relative_path_value = str(
        manifest_row["relative_path"]
    ).strip()

    if not relative_path_value:
        raise RuntimeError(
            f"{dataset_id}: Notebook 02 manifest contains an empty "
            "relative TRAIN path."
        )

    if absolute_path_value:
        candidate_path = Path(
            absolute_path_value
        )
    else:
        candidate_path = (
            CANONICAL_NB02_ROOT
            / relative_path_value
        )

    if not candidate_path.exists():

        fallback_path = (
            CANONICAL_NB02_ROOT
            / relative_path_value
        )

        if fallback_path.exists():
            candidate_path = fallback_path
        else:
            raise FileNotFoundError(
                f"Notebook 02 TRAIN artifact does not exist for "
                f"{dataset_id}.\n\n"
                f"Manifest absolute path:\n"
                f"{absolute_path_value}\n\n"
                f"Manifest relative path:\n"
                f"{relative_path_value}"
            )

    candidate_path = candidate_path.resolve()

    try:
        candidate_path.relative_to(
            CANONICAL_NATIVE_ROOT.resolve()
        )
    except ValueError:
        raise RuntimeError(
            f"{dataset_id}: resolved TRAIN artifact is outside "
            f"the canonical Notebook 02 native layer.\n"
            f"Resolved path: {candidate_path}\n"
            f"Expected root: {CANONICAL_NATIVE_ROOT}"
        )

    if candidate_path.suffix.lower() != ".csv":
        raise RuntimeError(
            f"{dataset_id}: authoritative Notebook 02 native TRAIN "
            "artifact must be CSV.\n"
            f"Resolved artifact: {candidate_path}"
        )

    TRAIN_DATA_PATHS[dataset_id] = candidate_path


# --------------------------------------------------------------------------------------------------
# 8. Load Native TRAIN Data Sequentially
# --------------------------------------------------------------------------------------------------
#
# Important:
# - Only TRAIN data are loaded.
# - Validation and TEST data are not loaded.
# - Native tabular representation is retained.
# - No preprocessing is fitted here.
# - No encoded/scaled matrix is used here.
# - Target and provenance columns remain available in the native dataset.
#
# RAM policy:
# One dataset is processed at a time during validation.
# The validated dataframes are retained because they are required by
# downstream SPP-GAN training.
# --------------------------------------------------------------------------------------------------

TRAIN_DATAFRAMES = {}
TRAIN_METADATA = {}

for dataset_id in DATASET_IDS:

    path = TRAIN_DATA_PATHS[dataset_id]

    print()
    print("-" * 100)
    print(f"DATASET : {dataset_id}")
    print("-" * 100)

    print(
        f"  ✓ TRAIN artifact : {path}"
    )

    # ----------------------------------------------------------------------------------------------
    # Load native TRAIN CSV
    # ----------------------------------------------------------------------------------------------

    df = pd.read_csv(
        path,
        low_memory=False,
    )

    # ----------------------------------------------------------------------------------------------
    # Basic structural validation
    # ----------------------------------------------------------------------------------------------

    if df.empty:
        raise RuntimeError(
            f"{dataset_id}: native TRAIN dataset is empty."
        )

    if not df.columns.is_unique:
        raise RuntimeError(
            f"{dataset_id}: native TRAIN columns are not unique."
        )

    # ----------------------------------------------------------------------------------------------
    # Manifest-derived ROW validation
    # ----------------------------------------------------------------------------------------------

    expected_rows = TRAIN_ROWS[dataset_id]
    observed_rows = len(df)

    if observed_rows != expected_rows:
        raise RuntimeError(
            f"{dataset_id}: TRAIN row mismatch.\n"
            f"Manifest expected : {expected_rows:,}\n"
            f"Observed          : {observed_rows:,}"
        )

    # ----------------------------------------------------------------------------------------------
    # Manifest-derived COLUMN validation
    # ----------------------------------------------------------------------------------------------

    expected_columns = TRAIN_COLUMNS[dataset_id]
    observed_columns = len(df.columns)

    if observed_columns != expected_columns:
        raise RuntimeError(
            f"{dataset_id}: TRAIN column-count mismatch.\n"
            f"Manifest expected : {expected_columns}\n"
            f"Observed          : {observed_columns}"
        )

    # ----------------------------------------------------------------------------------------------
    # Resolve target column
    # ----------------------------------------------------------------------------------------------

    target_column = TARGET_COLUMNS[dataset_id]

    if target_column not in df.columns:
        raise RuntimeError(
            f"{dataset_id}: expected target column "
            f"'{target_column}' is missing."
        )

    # ----------------------------------------------------------------------------------------------
    # Resolve provenance column
    # ----------------------------------------------------------------------------------------------

    if PROVENANCE_COLUMN not in df.columns:
        raise RuntimeError(
            f"{dataset_id}: required Notebook 02 provenance "
            f"column '{PROVENANCE_COLUMN}' is missing."
        )

    # ----------------------------------------------------------------------------------------------
    # Verify target is non-empty
    # ----------------------------------------------------------------------------------------------

    if df[target_column].isna().all():
        raise RuntimeError(
            f"{dataset_id}: target column '{target_column}' "
            "is entirely missing."
        )

    # ----------------------------------------------------------------------------------------------
    # Verify provenance
    # ----------------------------------------------------------------------------------------------

    if df[PROVENANCE_COLUMN].isna().any():
        raise RuntimeError(
            f"{dataset_id}: provenance column contains missing values."
        )

    if not df[PROVENANCE_COLUMN].is_unique:
        raise RuntimeError(
            f"{dataset_id}: provenance column is not unique."
        )

    # ----------------------------------------------------------------------------------------------
    # Verify manifest dimensions independently
    # ----------------------------------------------------------------------------------------------

    manifest_rows = int(
        TRAIN_MANIFEST_ROWS[dataset_id]["rows"]
    )

    manifest_columns = int(
        TRAIN_MANIFEST_ROWS[dataset_id]["columns"]
    )

    if manifest_rows != observed_rows:
        raise RuntimeError(
            f"{dataset_id}: manifest row count mismatch.\n"
            f"Manifest : {manifest_rows}\n"
            f"Loaded   : {observed_rows}"
        )

    if manifest_columns != observed_columns:
        raise RuntimeError(
            f"{dataset_id}: manifest column count mismatch.\n"
            f"Manifest : {manifest_columns}\n"
            f"Loaded   : {observed_columns}"
        )

    # ----------------------------------------------------------------------------------------------
    # Persist validated native TRAIN dataframe
    # ----------------------------------------------------------------------------------------------

    TRAIN_DATAFRAMES[dataset_id] = df

    TRAIN_METADATA[dataset_id] = {
        "dataset_id": dataset_id,
        "split": "train",
        "path": str(path),
        "rows": int(df.shape[0]),
        "columns": int(df.shape[1]),
        "manifest_rows": manifest_rows,
        "manifest_columns": manifest_columns,
        "target_column": target_column,
        "provenance_column": PROVENANCE_COLUMN,
        "target_present": True,
        "provenance_present": True,
    }

    print(
        f"  ✓ Rows              : "
        f"{df.shape[0]:,}"
    )

    print(
        f"  ✓ Columns           : "
        f"{df.shape[1]}"
    )

    print(
        f"  ✓ Target column     : "
        f"{target_column}"
    )

    print(
        f"  ✓ Provenance column : "
        f"{PROVENANCE_COLUMN}"
    )

    print(
        "  ✓ Manifest row/schema agreement verified"
    )

    print(
        "  ✓ Native TRAIN data validated"
    )


# --------------------------------------------------------------------------------------------------
# 9. Explicit Manifest-Derived Row/Schema Checkpoint
# --------------------------------------------------------------------------------------------------
#
# CP-12.03
#
# This checkpoint establishes that every loaded TRAIN dataframe has the
# exact row and column dimensions persisted by Notebook 02.
#
# No hard-coded row counts are used as the authoritative source.
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("CP-12.03 — MANIFEST-DERIVED TRAIN DATA INTEGRITY CHECKPOINT")
print("=" * 100)

CP_12_03_ROWS_SCHEMA = {}

for dataset_id in DATASET_IDS:

    df = TRAIN_DATAFRAMES[dataset_id]

    manifest_rows = TRAIN_ROWS[dataset_id]
    manifest_columns = TRAIN_COLUMNS[dataset_id]

    observed_rows = int(df.shape[0])
    observed_columns = int(df.shape[1])

    row_match = (
        observed_rows
        == manifest_rows
    )

    schema_match = (
        observed_columns
        == manifest_columns
    )

    target_match = (
        TARGET_COLUMNS[dataset_id]
        in df.columns
    )

    provenance_match = (
        PROVENANCE_COLUMN
        in df.columns
    )

    CP_12_03_ROWS_SCHEMA[dataset_id] = {
        "manifest_rows": manifest_rows,
        "observed_rows": observed_rows,
        "row_match": row_match,
        "manifest_columns": manifest_columns,
        "observed_columns": observed_columns,
        "schema_match": schema_match,
        "target_present": target_match,
        "provenance_present": provenance_match,
    }

    print()
    print(f"DATASET : {dataset_id}")

    print(
        f"  {'✓' if row_match else '✗'} "
        f"Rows       | "
        f"Manifest={manifest_rows:,} | "
        f"Observed={observed_rows:,}"
    )

    print(
        f"  {'✓' if schema_match else '✗'} "
        f"Columns    | "
        f"Manifest={manifest_columns} | "
        f"Observed={observed_columns}"
    )

    print(
        f"  {'✓' if target_match else '✗'} "
        f"Target     | "
        f"{TARGET_COLUMNS[dataset_id]}"
    )

    print(
        f"  {'✓' if provenance_match else '✗'} "
        f"Provenance | "
        f"{PROVENANCE_COLUMN}"
    )

    if not (
        row_match
        and schema_match
        and target_match
        and provenance_match
    ):
        raise RuntimeError(
            f"CP-12.03 failed for dataset: {dataset_id}"
        )

CP_12_03_PASS = True

print()
print("✓ CP-12.03 PASS")
print("✓ Manifest-derived TRAIN row counts verified")
print("✓ Manifest-derived TRAIN schema dimensions verified")
print("✓ Target columns verified")
print("✓ Provenance columns verified")
print("✓ Notebook 02 TRAIN integrity established")


# --------------------------------------------------------------------------------------------------
# 10. Cross-Dataset Training Row Validation
# --------------------------------------------------------------------------------------------------

for dataset_id in DATASET_IDS:

    observed_rows = len(
        TRAIN_DATAFRAMES[dataset_id]
    )

    expected_rows = TRAIN_ROWS[dataset_id]

    if observed_rows != expected_rows:
        raise RuntimeError(
            f"{dataset_id}: final TRAIN row validation failed."
        )


# --------------------------------------------------------------------------------------------------
# 11. Explicit Data-Usage Boundary
# --------------------------------------------------------------------------------------------------

VALIDATION_DATA_LOADED = False
TEST_DATA_LOADED = False
SYNTHETIC_DATA_LOADED = False

if VALIDATION_DATA_LOADED:
    raise RuntimeError(
        "Validation data must not be loaded in Section 3."
    )

if TEST_DATA_LOADED:
    raise RuntimeError(
        "Test data must not be loaded in Section 3."
    )

if SYNTHETIC_DATA_LOADED:
    raise RuntimeError(
        "Synthetic data must not be loaded in Section 3."
    )


# --------------------------------------------------------------------------------------------------
# 12. Final TRAIN Data Summary
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("TRAIN DATA SUMMARY")
print("=" * 100)

for dataset_id in DATASET_IDS:

    df = TRAIN_DATAFRAMES[dataset_id]

    print(
        f"  {dataset_id:<20} | "
        f"Rows={df.shape[0]:>8,} | "
        f"Columns={df.shape[1]:>5} | "
        f"Target={TRAIN_METADATA[dataset_id]['target_column']:<12} | "
        f"Path={TRAIN_DATA_PATHS[dataset_id]}"
    )


# --------------------------------------------------------------------------------------------------
# 13. Section 3 Validation
# --------------------------------------------------------------------------------------------------

SECTION_3_CHECKS = {

    "nb02_root_exists":
        CANONICAL_NB02_ROOT.exists(),

    "native_root_exists":
        CANONICAL_NATIVE_ROOT.exists(),

    "schema_root_exists":
        CANONICAL_SCHEMA_ROOT.exists(),

    "metadata_root_exists":
        CANONICAL_METADATA_ROOT.exists(),

    "native_manifest_exists":
        CANONICAL_NATIVE_MANIFEST.exists(),

    "preprocessor_manifest_exists":
        CANONICAL_PREPROCESSOR_MANIFEST.exists(),

    "native_manifest_nonempty":
        not NATIVE_MANIFEST_DF.empty,

    "three_training_datasets":
        len(TRAIN_DATAFRAMES) == 3,

    "manifest_train_records":
        all(
            dataset_id in TRAIN_MANIFEST_ROWS
            for dataset_id in DATASET_IDS
        ),

    "manifest_row_schema_checkpoint":
        CP_12_03_PASS,

    "adult_income_train_rows":
        len(
            TRAIN_DATAFRAMES["adult_income"]
        )
        ==
        TRAIN_ROWS["adult_income"],

    "bank_marketing_train_rows":
        len(
            TRAIN_DATAFRAMES["bank_marketing"]
        )
        ==
        TRAIN_ROWS["bank_marketing"],

    "diabetes_train_rows":
        len(
            TRAIN_DATAFRAMES["diabetes_130us"]
        )
        ==
        TRAIN_ROWS["diabetes_130us"],

    "adult_income_train_columns":
        len(
            TRAIN_DATAFRAMES["adult_income"].columns
        )
        ==
        TRAIN_COLUMNS["adult_income"],

    "bank_marketing_train_columns":
        len(
            TRAIN_DATAFRAMES["bank_marketing"].columns
        )
        ==
        TRAIN_COLUMNS["bank_marketing"],

    "diabetes_train_columns":
        len(
            TRAIN_DATAFRAMES["diabetes_130us"].columns
        )
        ==
        TRAIN_COLUMNS["diabetes_130us"],

    "adult_income_target":
        TRAIN_METADATA["adult_income"]["target_column"]
        ==
        "income",

    "bank_marketing_target":
        TRAIN_METADATA["bank_marketing"]["target_column"]
        ==
        "y",

    "diabetes_target":
        TRAIN_METADATA["diabetes_130us"]["target_column"]
        ==
        "readmitted",

    "adult_income_provenance":
        PROVENANCE_COLUMN
        in
        TRAIN_DATAFRAMES["adult_income"].columns,

    "bank_marketing_provenance":
        PROVENANCE_COLUMN
        in
        TRAIN_DATAFRAMES["bank_marketing"].columns,

    "diabetes_provenance":
        PROVENANCE_COLUMN
        in
        TRAIN_DATAFRAMES["diabetes_130us"].columns,

    "adult_income_provenance_unique":
        TRAIN_DATAFRAMES["adult_income"][PROVENANCE_COLUMN].is_unique,

    "bank_marketing_provenance_unique":
        TRAIN_DATAFRAMES["bank_marketing"][PROVENANCE_COLUMN].is_unique,

    "diabetes_provenance_unique":
        TRAIN_DATAFRAMES["diabetes_130us"][PROVENANCE_COLUMN].is_unique,

    "validation_not_loaded":
        VALIDATION_DATA_LOADED is False,

    "test_not_loaded":
        TEST_DATA_LOADED is False,

    "synthetic_not_loaded":
        SYNTHETIC_DATA_LOADED is False,
}


FAILED_SECTION_3_CHECKS = [
    name
    for name, passed in SECTION_3_CHECKS.items()
    if not passed
]


print()
print("-" * 100)
print("SECTION 3 VALIDATION")
print("-" * 100)

for check_name, passed in SECTION_3_CHECKS.items():

    print(
        f"{'✓' if passed else '✗'} "
        f"{check_name}"
    )


if FAILED_SECTION_3_CHECKS:

    raise RuntimeError(
        "Section 3 validation failed:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in FAILED_SECTION_3_CHECKS
        )
    )


# --------------------------------------------------------------------------------------------------
# 14. Completion Output
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("SECTION 3 — LOAD PROCESSED TRAINING DATA COMPLETE")
print("=" * 100)

print(
    "✓ Authoritative Notebook 02 native TRAIN data loaded."
)

print(
    "✓ adult_income      : "
    f"{TRAIN_DATAFRAMES['adult_income'].shape[0]:,} rows × "
    f"{TRAIN_DATAFRAMES['adult_income'].shape[1]} columns"
)

print(
    "✓ bank_marketing    : "
    f"{TRAIN_DATAFRAMES['bank_marketing'].shape[0]:,} rows × "
    f"{TRAIN_DATAFRAMES['bank_marketing'].shape[1]} columns"
)

print(
    "✓ diabetes_130us    : "
    f"{TRAIN_DATAFRAMES['diabetes_130us'].shape[0]:,} rows × "
    f"{TRAIN_DATAFRAMES['diabetes_130us'].shape[1]} columns"
)

print(
    "✓ Manifest-derived row/schema checkpoint passed."
)

print(
    "✓ Validation data were not loaded."
)

print(
    "✓ Test data were not loaded."
)

print(
    "✓ Synthetic data were not loaded."
)

print(
    f"✓ Section 3 checks    : "
    f"{len(SECTION_3_CHECKS) - len(FAILED_SECTION_3_CHECKS)}/"
    f"{len(SECTION_3_CHECKS)} PASS"
)

print()
print(
    "STATUS: PASS — SECTION 3 TRAIN DATA LOAD READY FOR NEXT SECTION"
)

print("=" * 100)

3. LOAD PROCESSED TRAINING DATA

----------------------------------------------------------------------------------------------------
NOTEBOOK 02 CANONICAL ARTIFACT VERIFICATION
----------------------------------------------------------------------------------------------------
✓ Notebook 02 root         : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02
✓ Native root              : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/native
✓ Preprocessor root        : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/preprocessors
✓ Schema root              : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/schemas
✓ Metadata root            : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/schemas/metadata
✓ Native manifest          : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/native/native_dataset_manifest.csv
✓ Preprocessor manifest    : /content/drive/MyDrive/SPP_GAN_Rese

In [4]:
# ==================================================================================================
# SECTION 4 — LOAD STATISTICAL GUIDANCE
# ==================================================================================================

print("=" * 100)
print("4. LOAD STATISTICAL GUIDANCE")
print("=" * 100)

import json
import hashlib
from pathlib import Path


# --------------------------------------------------------------------------------------------------
# 1. Resolve Canonical Notebook 03 Artifact Layer
# --------------------------------------------------------------------------------------------------

CANONICAL_NB03_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_03"
)

CANONICAL_NB03_GUIDANCE_ROOT = (
    CANONICAL_NB03_ROOT
    / "guidance"
)


# --------------------------------------------------------------------------------------------------
# 2. Verify Canonical Notebook 03 Artifact Layer
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)
print("NOTEBOOK 03 CANONICAL ARTIFACT VERIFICATION")
print("-" * 100)

print(
    f"✓ Notebook 03 root : {CANONICAL_NB03_ROOT}"
)

print(
    f"✓ Guidance root    : {CANONICAL_NB03_GUIDANCE_ROOT}"
)

if not CANONICAL_NB03_ROOT.is_dir():

    raise FileNotFoundError(
        "Canonical Notebook 03 processed-artifact root does not exist:\n"
        f"{CANONICAL_NB03_ROOT}\n\n"
        "Notebook 12 will NOT reconstruct Notebook 03 statistical guidance."
    )

if not CANONICAL_NB03_GUIDANCE_ROOT.is_dir():

    raise FileNotFoundError(
        "Canonical Notebook 03 guidance directory does not exist:\n"
        f"{CANONICAL_NB03_GUIDANCE_ROOT}\n\n"
        "Notebook 12 will NOT reconstruct Notebook 03 statistical guidance."
    )


# --------------------------------------------------------------------------------------------------
# 3. Frozen Dataset Registry
# --------------------------------------------------------------------------------------------------

EXPECTED_DATASET_IDS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

if list(DATASET_IDS) != EXPECTED_DATASET_IDS:

    raise RuntimeError(
        "Notebook 12 DATASET_IDS does not match the frozen dataset registry.\n"
        f"Expected : {EXPECTED_DATASET_IDS}\n"
        f"Observed : {list(DATASET_IDS)}"
    )


# --------------------------------------------------------------------------------------------------
# 4. Guidance Artifact Registry
# --------------------------------------------------------------------------------------------------

GUIDANCE_FILENAMES = {
    dataset_id:
        f"{dataset_id}_spp_gan_statistical_guidance.json"
    for dataset_id in DATASET_IDS
}

GUIDANCE_PATHS = {
    dataset_id:
        CANONICAL_NB03_GUIDANCE_ROOT
        / GUIDANCE_FILENAMES[dataset_id]
    for dataset_id in DATASET_IDS
}


# --------------------------------------------------------------------------------------------------
# 5. Required Guidance Schema
# --------------------------------------------------------------------------------------------------

REQUIRED_GUIDANCE_KEYS = {
    "guidance_version",
    "guidance_type",
    "source_reference_version",
    "source_reference_type",
    "feature_schema",
    "numeric_feature_guidance",
    "categorical_feature_guidance",
    "strongest_numeric_pearson_dependencies",
    "strongest_numeric_spearman_dependencies",
    "strongest_categorical_dependencies",
    "target_policy",
    "identifier_policy",
    "provenance_policy",
    "evidence_policy",
}


# --------------------------------------------------------------------------------------------------
# 6. SHA-256 Utility
# --------------------------------------------------------------------------------------------------

def calculate_sha256(
    path,
    chunk_size=1024 * 1024
):

    digest = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as file_handle:

        while True:

            chunk = file_handle.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


# --------------------------------------------------------------------------------------------------
# 7. Load Persisted Notebook 03 Guidance
# --------------------------------------------------------------------------------------------------

STATISTICAL_GUIDANCE = {}
GUIDANCE_SHA256 = {}
GUIDANCE_ARTIFACT_METADATA = {}

print()
print("-" * 100)
print("LOADING PERSISTED NOTEBOOK 03 STATISTICAL GUIDANCE")
print("-" * 100)

for dataset_id in DATASET_IDS:

    guidance_path = GUIDANCE_PATHS[
        dataset_id
    ]

    if not guidance_path.is_file():

        raise FileNotFoundError(
            f"Statistical guidance missing for {dataset_id}:\n"
            f"{guidance_path}\n\n"
            "Notebook 12 will NOT reconstruct the guidance."
        )

    with open(
        guidance_path,
        "r",
        encoding="utf-8"
    ) as file_handle:

        guidance = json.load(
            file_handle
        )

    if not isinstance(
        guidance,
        dict
    ):

        raise TypeError(
            f"{dataset_id}: statistical guidance must be a JSON object."
        )

    missing_keys = (
        REQUIRED_GUIDANCE_KEYS
        -
        set(guidance.keys())
    )

    if missing_keys:

        raise RuntimeError(
            f"{dataset_id}: statistical guidance is missing "
            "required keys:\n"
            +
            "\n".join(
                f"  - {key}"
                for key in sorted(
                    missing_keys
                )
            )
        )

    artifact_hash = calculate_sha256(
        guidance_path
    )

    STATISTICAL_GUIDANCE[
        dataset_id
    ] = guidance

    GUIDANCE_SHA256[
        dataset_id
    ] = artifact_hash

    GUIDANCE_ARTIFACT_METADATA[
        dataset_id
    ] = {
        "dataset_id":
            dataset_id,

        "artifact_type":
            "spp_gan_statistical_guidance",

        "path":
            str(guidance_path),

        "filename":
            guidance_path.name,

        "sha256":
            artifact_hash,

        "guidance_version":
            str(
                guidance[
                    "guidance_version"
                ]
            ),

        "guidance_type":
            str(
                guidance[
                    "guidance_type"
                ]
            ),

        "source_reference_version":
            str(
                guidance[
                    "source_reference_version"
                ]
            ),

        "source_reference_type":
            str(
                guidance[
                    "source_reference_type"
                ]
            ),
    }

    print(
        f"✓ {dataset_id:<20} "
        f"guidance loaded"
    )

    print(
        f"  SHA-256 : {artifact_hash}"
    )


# --------------------------------------------------------------------------------------------------
# 8. Validate Guidance Structure
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)
print("STATISTICAL GUIDANCE STRUCTURE VALIDATION")
print("-" * 100)

for dataset_id in DATASET_IDS:

    guidance = STATISTICAL_GUIDANCE[
        dataset_id
    ]

    if not str(
        guidance[
            "guidance_version"
        ]
    ).strip():

        raise RuntimeError(
            f"{dataset_id}: guidance_version is empty."
        )

    if not str(
        guidance[
            "guidance_type"
        ]
    ).strip():

        raise RuntimeError(
            f"{dataset_id}: guidance_type is empty."
        )

    if not str(
        guidance[
            "source_reference_version"
        ]
    ).strip():

        raise RuntimeError(
            f"{dataset_id}: source_reference_version is empty."
        )

    if not str(
        guidance[
            "source_reference_type"
        ]
    ).strip():

        raise RuntimeError(
            f"{dataset_id}: source_reference_type is empty."
        )

    if not isinstance(
        guidance[
            "feature_schema"
        ],
        dict
    ):

        raise TypeError(
            f"{dataset_id}: feature_schema must be a JSON object."
        )

    for field_name in [
        "numeric_feature_guidance",
        "categorical_feature_guidance",
        "strongest_numeric_pearson_dependencies",
        "strongest_numeric_spearman_dependencies",
        "strongest_categorical_dependencies",
    ]:

        if not isinstance(
            guidance[field_name],
            (dict, list)
        ):

            raise TypeError(
                f"{dataset_id}: {field_name} must be "
                "a JSON object or list."
            )

    print(
        f"✓ {dataset_id:<20} "
        "guidance structure validated"
    )


# --------------------------------------------------------------------------------------------------
# 9. Validate Dataset Identity
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)
print("GUIDANCE DATASET IDENTITY VALIDATION")
print("-" * 100)

for dataset_id in DATASET_IDS:

    guidance = STATISTICAL_GUIDANCE[
        dataset_id
    ]

    feature_schema = guidance[
        "feature_schema"
    ]

    schema_dataset_id = feature_schema.get(
        "dataset_id"
    )

    if schema_dataset_id is not None:

        if str(
            schema_dataset_id
        ).strip() != dataset_id:

            raise RuntimeError(
                f"{dataset_id}: guidance dataset identity mismatch.\n"
                f"Expected : {dataset_id}\n"
                f"Observed : {schema_dataset_id}"
            )

    print(
        f"✓ {dataset_id:<20} "
        "dataset identity validated"
    )


# --------------------------------------------------------------------------------------------------
# 10. Explicit Statistical Data-Usage Boundary
# --------------------------------------------------------------------------------------------------

GUIDANCE_RECONSTRUCTED = False
GUIDANCE_FROM_VALIDATION = False
GUIDANCE_FROM_TEST = False
GUIDANCE_FROM_SYNTHETIC = False

if GUIDANCE_RECONSTRUCTED:

    raise RuntimeError(
        "Notebook 12 must not reconstruct Notebook 03 statistical guidance."
    )

if GUIDANCE_FROM_VALIDATION:

    raise RuntimeError(
        "Statistical guidance must not be derived from validation data."
    )

if GUIDANCE_FROM_TEST:

    raise RuntimeError(
        "Statistical guidance must not be derived from test data."
    )

if GUIDANCE_FROM_SYNTHETIC:

    raise RuntimeError(
        "Statistical guidance must not be derived from synthetic data."
    )


GUIDANCE_SOURCE_POLICY = {
    dataset_id: {
        "source_notebook":
            "Notebook 03",

        "source_split":
            "TRAIN",

        "reconstructed_in_notebook_12":
            False,

        "validation_used":
            False,

        "test_used":
            False,

        "synthetic_used":
            False,
    }
    for dataset_id in DATASET_IDS
}


# --------------------------------------------------------------------------------------------------
# 11. Verify Guidance Artifact Fingerprints
# --------------------------------------------------------------------------------------------------

GUIDANCE_SHA256_RECHECK = {}

for dataset_id in DATASET_IDS:

    guidance_path = GUIDANCE_PATHS[
        dataset_id
    ]

    current_hash = calculate_sha256(
        guidance_path
    )

    GUIDANCE_SHA256_RECHECK[
        dataset_id
    ] = current_hash

    if current_hash != GUIDANCE_SHA256[
        dataset_id
    ]:

        raise RuntimeError(
            f"{dataset_id}: guidance artifact changed "
            "during Section 4 execution."
        )


# --------------------------------------------------------------------------------------------------
# 12. CP-12.04 — STATISTICAL GUIDANCE INTEGRITY CHECKPOINT
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("CP-12.04 — STATISTICAL GUIDANCE INTEGRITY CHECKPOINT")
print("=" * 100)

CP_12_04_RESULTS = {}

for dataset_id in DATASET_IDS:

    guidance = STATISTICAL_GUIDANCE[
        dataset_id
    ]

    artifact_exists = (
        GUIDANCE_PATHS[
            dataset_id
        ].is_file()
    )

    required_keys_present = (
        REQUIRED_GUIDANCE_KEYS
        <=
        set(
            guidance.keys()
        )
    )

    feature_schema_valid = isinstance(
        guidance[
            "feature_schema"
        ],
        dict
    )

    source_metadata_valid = (
        bool(
            str(
                guidance[
                    "source_reference_version"
                ]
            ).strip()
        )
        and
        bool(
            str(
                guidance[
                    "source_reference_type"
                ]
            ).strip()
        )
    )

    dataset_identity_valid = True

    schema_dataset_id = guidance[
        "feature_schema"
    ].get(
        "dataset_id"
    )

    if schema_dataset_id is not None:

        dataset_identity_valid = (
            str(
                schema_dataset_id
            ).strip()
            ==
            dataset_id
        )

    fingerprint_stable = (
        GUIDANCE_SHA256[
            dataset_id
        ]
        ==
        GUIDANCE_SHA256_RECHECK[
            dataset_id
        ]
    )

    source_boundary_valid = (
        GUIDANCE_SOURCE_POLICY[
            dataset_id
        ]["source_notebook"]
        ==
        "Notebook 03"

        and

        GUIDANCE_SOURCE_POLICY[
            dataset_id
        ]["source_split"]
        ==
        "TRAIN"

        and

        GUIDANCE_SOURCE_POLICY[
            dataset_id
        ]["reconstructed_in_notebook_12"]
        is False

        and

        GUIDANCE_SOURCE_POLICY[
            dataset_id
        ]["validation_used"]
        is False

        and

        GUIDANCE_SOURCE_POLICY[
            dataset_id
        ]["test_used"]
        is False

        and

        GUIDANCE_SOURCE_POLICY[
            dataset_id
        ]["synthetic_used"]
        is False
    )

    dataset_pass = all(
        [
            artifact_exists,
            required_keys_present,
            feature_schema_valid,
            source_metadata_valid,
            dataset_identity_valid,
            fingerprint_stable,
            source_boundary_valid,
        ]
    )

    CP_12_04_RESULTS[
        dataset_id
    ] = {
        "artifact_exists":
            artifact_exists,

        "required_keys_present":
            required_keys_present,

        "feature_schema_valid":
            feature_schema_valid,

        "source_metadata_valid":
            source_metadata_valid,

        "dataset_identity_valid":
            dataset_identity_valid,

        "fingerprint_stable":
            fingerprint_stable,

        "source_boundary_valid":
            source_boundary_valid,

        "pass":
            dataset_pass,
    }

    print()
    print(
        f"DATASET : {dataset_id}"
    )

    print(
        f"  {'✓' if artifact_exists else '✗'} "
        "Guidance artifact exists"
    )

    print(
        f"  {'✓' if required_keys_present else '✗'} "
        "Required guidance schema present"
    )

    print(
        f"  {'✓' if feature_schema_valid else '✗'} "
        "Feature schema valid"
    )

    print(
        f"  {'✓' if source_metadata_valid else '✗'} "
        "Source metadata valid"
    )

    print(
        f"  {'✓' if dataset_identity_valid else '✗'} "
        "Dataset identity valid"
    )

    print(
        f"  {'✓' if fingerprint_stable else '✗'} "
        "SHA-256 fingerprint stable"
    )

    print(
        f"  {'✓' if source_boundary_valid else '✗'} "
        "TRAIN-only source boundary valid"
    )

    if not dataset_pass:

        raise RuntimeError(
            f"CP-12.04 failed for dataset: {dataset_id}"
        )


CP_12_04_PASS = True

print()
print("✓ CP-12.04 PASS")
print("✓ Notebook 03 guidance artifacts verified")
print("✓ Guidance schema verified")
print("✓ Dataset identities verified")
print("✓ Source metadata verified")
print("✓ SHA-256 fingerprints verified")
print("✓ TRAIN-only source boundary established")
print("✓ No statistical profile reconstructed")


# --------------------------------------------------------------------------------------------------
# 13. Guidance Summary
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("STATISTICAL GUIDANCE SUMMARY")
print("=" * 100)

for dataset_id in DATASET_IDS:

    guidance = STATISTICAL_GUIDANCE[
        dataset_id
    ]

    print(
        f"  {dataset_id:<20} | "
        f"Version={guidance['guidance_version']} | "
        f"Type={guidance['guidance_type']} | "
        f"Source={guidance['source_reference_type']}"
    )


# --------------------------------------------------------------------------------------------------
# 14. Section 4 Validation
# --------------------------------------------------------------------------------------------------

SECTION_4_CHECKS = {

    "nb03_root_exists":
        CANONICAL_NB03_ROOT.is_dir(),

    "guidance_root_exists":
        CANONICAL_NB03_GUIDANCE_ROOT.is_dir(),

    "three_guidance_artifacts":
        len(STATISTICAL_GUIDANCE)
        ==
        len(DATASET_IDS),

    "all_guidance_paths_exist":
        all(
            GUIDANCE_PATHS[
                dataset_id
            ].is_file()
            for dataset_id in DATASET_IDS
        ),

    "all_required_keys_present":
        all(
            REQUIRED_GUIDANCE_KEYS
            <=
            set(
                STATISTICAL_GUIDANCE[
                    dataset_id
                ].keys()
            )
            for dataset_id in DATASET_IDS
        ),

    "all_feature_schemas_valid":
        all(
            isinstance(
                STATISTICAL_GUIDANCE[
                    dataset_id
                ]["feature_schema"],
                dict
            )
            for dataset_id in DATASET_IDS
        ),

    "dataset_identities_valid":
        all(
            result[
                "dataset_identity_valid"
            ]
            for result in CP_12_04_RESULTS.values()
        ),

    "guidance_fingerprints_stable":
        all(
            result[
                "fingerprint_stable"
            ]
            for result in CP_12_04_RESULTS.values()
        ),

    "train_only_source_boundary":
        all(
            result[
                "source_boundary_valid"
            ]
            for result in CP_12_04_RESULTS.values()
        ),

    "guidance_not_reconstructed":
        GUIDANCE_RECONSTRUCTED is False,

    "validation_not_used":
        GUIDANCE_FROM_VALIDATION is False,

    "test_not_used":
        GUIDANCE_FROM_TEST is False,

    "synthetic_not_used":
        GUIDANCE_FROM_SYNTHETIC is False,

    "cp_12_04_pass":
        CP_12_04_PASS,
}


FAILED_SECTION_4_CHECKS = [
    name
    for name, passed in SECTION_4_CHECKS.items()
    if not passed
]


print()
print("-" * 100)
print("SECTION 4 VALIDATION")
print("-" * 100)

for check_name, passed in SECTION_4_CHECKS.items():

    print(
        f"{'✓' if passed else '✗'} "
        f"{check_name}"
    )


if FAILED_SECTION_4_CHECKS:

    raise RuntimeError(
        "Section 4 validation failed:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in FAILED_SECTION_4_CHECKS
        )
    )


# --------------------------------------------------------------------------------------------------
# 15. Completion Output
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("SECTION 4 — LOAD STATISTICAL GUIDANCE COMPLETE")
print("=" * 100)

print(
    "✓ Notebook 03 persisted statistical guidance loaded."
)

print(
    "✓ Statistical guidance remains TRAIN-only."
)

print(
    "✓ No statistical profile was reconstructed in Notebook 12."
)

print(
    "✓ Validation data were not used."
)

print(
    "✓ Test data were not used."
)

print(
    "✓ Synthetic data were not used."
)

print(
    "✓ Guidance artifact fingerprints were verified."
)

print(
    "✓ CP-12.04 statistical guidance integrity checkpoint passed."
)

print(
    f"✓ Section 4 checks    : "
    f"{len(SECTION_4_CHECKS) - len(FAILED_SECTION_4_CHECKS)}/"
    f"{len(SECTION_4_CHECKS)} PASS"
)

print()
print(
    "STATUS: PASS — SECTION 4 STATISTICAL GUIDANCE READY FOR NEXT SECTION"
)

print("=" * 100)

4. LOAD STATISTICAL GUIDANCE

----------------------------------------------------------------------------------------------------
NOTEBOOK 03 CANONICAL ARTIFACT VERIFICATION
----------------------------------------------------------------------------------------------------
✓ Notebook 03 root : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_03
✓ Guidance root    : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_03/guidance

----------------------------------------------------------------------------------------------------
LOADING PERSISTED NOTEBOOK 03 STATISTICAL GUIDANCE
----------------------------------------------------------------------------------------------------
✓ adult_income         guidance loaded
  SHA-256 : fe8eae5bf6521eb20c437abcabbac03b771e68e8459a404c96b8719d58962f77
✓ bank_marketing       guidance loaded
  SHA-256 : 32ffb00dec670b69cd6a642df4c663b9dcadcc614472863d57f9f23197e95d8e
✓ diabetes_130us       guidance loaded
  SHA-256 : 89

In [5]:
# ==================================================================================================
# SECTION 5 — LOAD SPP-GAN ARCHITECTURE
# ==================================================================================================

print("=" * 100)
print("5. LOAD SPP-GAN ARCHITECTURE")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. Verify Notebook 08 Root
# --------------------------------------------------------------------------------------------------

if "NB08_ROOT" not in globals():

    raise RuntimeError(
        "NB08_ROOT is not defined.\n"
        "Notebook 08 canonical root must be initialized before Section 5."
    )


NB08_ROOT = Path(
    NB08_ROOT
)


if not NB08_ROOT.exists():

    raise FileNotFoundError(
        f"Notebook 08 root does not exist:\n{NB08_ROOT}"
    )


if not NB08_ROOT.is_dir():

    raise NotADirectoryError(
        f"Notebook 08 root is not a directory:\n{NB08_ROOT}"
    )


print(
    f"✓ Notebook 08 root : {NB08_ROOT}"
)


# --------------------------------------------------------------------------------------------------
# 2. Canonical Notebook 08 Architecture Artifacts
# --------------------------------------------------------------------------------------------------

ARCHITECTURE_SUMMARY_PATH = (
    NB08_ROOT
    / "architecture"
    / "sppgan_architecture_summary.csv"
)

PARAMETER_COUNT_PATH = (
    NB08_ROOT
    / "metadata"
    / "sppgan_parameter_count.csv"
)

MODEL_CONFIG_PATH = (
    NB08_ROOT
    / "config"
    / "sppgan_model_configuration.json"
)

ARCHITECTURE_REGISTRY_PATH = (
    NB08_ROOT
    / "metadata"
    / "sppgan_architecture_artifacts.csv"
)


REQUIRED_ARCHITECTURE_FILES = {

    "architecture_summary":
        ARCHITECTURE_SUMMARY_PATH,

    "parameter_count":
        PARAMETER_COUNT_PATH,

    "model_configuration":
        MODEL_CONFIG_PATH,

    "architecture_registry":
        ARCHITECTURE_REGISTRY_PATH,
}


# --------------------------------------------------------------------------------------------------
# 3. Verify Required Architecture Artifacts
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)
print("NOTEBOOK 08 ARCHITECTURE ARTIFACT VERIFICATION")
print("-" * 100)


for artifact_name, artifact_path in (
    REQUIRED_ARCHITECTURE_FILES.items()
):

    if not artifact_path.exists():

        raise FileNotFoundError(
            f"Required Notebook 08 artifact is missing.\n"
            f"Artifact : {artifact_name}\n"
            f"Path     : {artifact_path}"
        )


    if not artifact_path.is_file():

        raise RuntimeError(
            f"Notebook 08 artifact is not a file.\n"
            f"Artifact : {artifact_name}\n"
            f"Path     : {artifact_path}"
        )


    if artifact_path.stat().st_size == 0:

        raise RuntimeError(
            f"Notebook 08 artifact is empty.\n"
            f"Artifact : {artifact_name}\n"
            f"Path     : {artifact_path}"
        )


    print(
        f"✓ {artifact_name:<24} : {artifact_path}"
    )


# --------------------------------------------------------------------------------------------------
# 4. Load Architecture Artifacts
# --------------------------------------------------------------------------------------------------

ARCHITECTURE_DF = pd.read_csv(
    ARCHITECTURE_SUMMARY_PATH
)

PARAMETER_DF = pd.read_csv(
    PARAMETER_COUNT_PATH
)

ARCHITECTURE_REGISTRY_DF = pd.read_csv(
    ARCHITECTURE_REGISTRY_PATH
)


with open(
    MODEL_CONFIG_PATH,
    "r",
    encoding="utf-8",
) as f:

    ARCHITECTURE_CONFIG = json.load(
        f
    )


if not isinstance(
    ARCHITECTURE_CONFIG,
    dict,
):

    raise TypeError(
        "Notebook 08 model configuration must be a JSON object."
    )


print()
print("-" * 100)
print("LOADED NOTEBOOK 08 ARCHITECTURE ARTIFACTS")
print("-" * 100)

print(
    f"✓ Architecture summary : {ARCHITECTURE_DF.shape}"
)

print(
    f"✓ Parameter metadata    : {PARAMETER_DF.shape}"
)

print(
    f"✓ Architecture registry: {ARCHITECTURE_REGISTRY_DF.shape}"
)

print(
    "✓ Model configuration   : loaded"
)


# --------------------------------------------------------------------------------------------------
# 5. Canonical Dataset Registry
# --------------------------------------------------------------------------------------------------

EXPECTED_DATASETS = set(
    DATASET_IDS
)


if EXPECTED_DATASETS != {
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
}:

    raise RuntimeError(
        "DATASET_IDS does not match the frozen SPP-GAN dataset registry."
    )


# --------------------------------------------------------------------------------------------------
# 6. Architecture Summary Schema
# --------------------------------------------------------------------------------------------------

REQUIRED_ARCHITECTURE_COLUMNS = {

    "dataset",
    "target",
    "generative_dimension",
    "transformed_dimension",
    "numerical_features",
    "categorical_features",
    "latent_dim",
    "generator_hidden_1",
    "generator_hidden_2",
    "critic_hidden_1",
    "critic_hidden_2",
    "total_trainable_parameters",
    "numerical_activation",
    "categorical_training_activation",
    "categorical_probability_mapping",
    "hard_decoding",
    "critic_output",
    "statistical_guidance",
    "differential_privacy",
    "privacy_accounting",
    "training",
}


missing_architecture_columns = (
    REQUIRED_ARCHITECTURE_COLUMNS
    -
    set(
        ARCHITECTURE_DF.columns
    )
)


if missing_architecture_columns:

    raise RuntimeError(
        "Notebook 08 architecture summary is missing required columns:\n"
        +
        "\n".join(
            f"  - {column}"
            for column in sorted(
                missing_architecture_columns
            )
        )
    )


print(
    "✓ Architecture-summary schema validated."
)


# --------------------------------------------------------------------------------------------------
# 7. Architecture Dataset Coverage
# --------------------------------------------------------------------------------------------------

if ARCHITECTURE_DF.empty:

    raise RuntimeError(
        "Notebook 08 architecture summary is empty."
    )


architecture_datasets = set(
    ARCHITECTURE_DF[
        "dataset"
    ].astype(
        str
    )
)


if architecture_datasets != EXPECTED_DATASETS:

    raise RuntimeError(
        "Architecture dataset coverage mismatch.\n"
        f"Expected: {sorted(EXPECTED_DATASETS)}\n"
        f"Found   : {sorted(architecture_datasets)}"
    )


if (
    ARCHITECTURE_DF[
        "dataset"
    ]
    .duplicated()
    .any()
):

    raise RuntimeError(
        "Duplicate dataset architecture records detected."
    )


print(
    "✓ Architecture dataset coverage validated."
)

print(
    "✓ Architecture dataset uniqueness validated."
)


# --------------------------------------------------------------------------------------------------
# 8. Numeric Architecture Validation
# --------------------------------------------------------------------------------------------------

NUMERIC_ARCHITECTURE_COLUMNS = [

    "generative_dimension",
    "transformed_dimension",
    "numerical_features",
    "categorical_features",
    "latent_dim",
    "generator_hidden_1",
    "generator_hidden_2",
    "critic_hidden_1",
    "critic_hidden_2",
    "total_trainable_parameters",
]


for column in NUMERIC_ARCHITECTURE_COLUMNS:

    values = pd.to_numeric(
        ARCHITECTURE_DF[
            column
        ],
        errors="coerce",
    )


    if values.isna().any():

        raise RuntimeError(
            f"Architecture column contains non-numeric values:\n"
            f"{column}"
        )


    if (
        values <= 0
    ).any():

        raise RuntimeError(
            f"Architecture column contains non-positive values:\n"
            f"{column}"
        )


print(
    "✓ Numeric architecture parameters validated."
)


# --------------------------------------------------------------------------------------------------
# 9. Generative-Dimension Consistency
# --------------------------------------------------------------------------------------------------

for _, row in ARCHITECTURE_DF.iterrows():

    dataset_id = str(
        row[
            "dataset"
        ]
    )


    expected_dimension = (
        int(
            row[
                "numerical_features"
            ]
        )
        +
        int(
            row[
                "categorical_features"
            ]
        )
        +
        1
    )


    actual_dimension = int(
        row[
            "generative_dimension"
        ]
    )


    if actual_dimension != expected_dimension:

        raise RuntimeError(
            f"{dataset_id}: generative-dimension mismatch.\n"
            f"Expected : {expected_dimension}\n"
            f"Observed : {actual_dimension}"
        )


print(
    "✓ Generative-dimension consistency validated."
)


# --------------------------------------------------------------------------------------------------
# 10. Frozen Transformed Dimensions
# --------------------------------------------------------------------------------------------------

EXPECTED_TRANSFORMED_DIMENSIONS = {

    "adult_income":
        105,

    "bank_marketing":
        51,

    "diabetes_130us":
        2329,
}


for dataset_id in DATASET_IDS:

    row = ARCHITECTURE_DF[
        ARCHITECTURE_DF[
            "dataset"
        ].astype(
            str
        )
        ==
        dataset_id
    ].iloc[
        0
    ]


    observed_dimension = int(
        row[
            "transformed_dimension"
        ]
    )


    expected_dimension = (
        EXPECTED_TRANSFORMED_DIMENSIONS[
            dataset_id
        ]
    )


    if observed_dimension != expected_dimension:

        raise RuntimeError(
            f"{dataset_id}: transformed-dimension mismatch.\n"
            f"Expected : {expected_dimension}\n"
            f"Observed : {observed_dimension}"
        )


print(
    "✓ Frozen transformed dimensions validated."
)


# --------------------------------------------------------------------------------------------------
# 11. Latent and Hidden-Layer Architecture Contract
# --------------------------------------------------------------------------------------------------

LATENT_DIMENSIONS = (
    ARCHITECTURE_DF[
        "latent_dim"
    ]
    .astype(
        int
    )
    .unique()
    .tolist()
)


GENERATOR_HIDDEN_1 = (
    ARCHITECTURE_DF[
        "generator_hidden_1"
    ]
    .astype(
        int
    )
    .unique()
    .tolist()
)


GENERATOR_HIDDEN_2 = (
    ARCHITECTURE_DF[
        "generator_hidden_2"
    ]
    .astype(
        int
    )
    .unique()
    .tolist()
)


CRITIC_HIDDEN_1 = (
    ARCHITECTURE_DF[
        "critic_hidden_1"
    ]
    .astype(
        int
    )
    .unique()
    .tolist()
)


CRITIC_HIDDEN_2 = (
    ARCHITECTURE_DF[
        "critic_hidden_2"
    ]
    .astype(
        int
    )
    .unique()
    .tolist()
)


if LATENT_DIMENSIONS != [128]:

    raise RuntimeError(
        f"Unexpected latent dimension: {LATENT_DIMENSIONS}"
    )


if GENERATOR_HIDDEN_1 != [256]:

    raise RuntimeError(
        f"Unexpected generator hidden layer 1: {GENERATOR_HIDDEN_1}"
    )


if GENERATOR_HIDDEN_2 != [256]:

    raise RuntimeError(
        f"Unexpected generator hidden layer 2: {GENERATOR_HIDDEN_2}"
    )


if CRITIC_HIDDEN_1 != [256]:

    raise RuntimeError(
        f"Unexpected critic hidden layer 1: {CRITIC_HIDDEN_1}"
    )


if CRITIC_HIDDEN_2 != [256]:

    raise RuntimeError(
        f"Unexpected critic hidden layer 2: {CRITIC_HIDDEN_2}"
    )


print(
    "✓ Latent dimension validated : 128"
)

print(
    "✓ Generator hidden layers validated : 256 / 256"
)

print(
    "✓ Critic hidden layers validated : 256 / 256"
)


# --------------------------------------------------------------------------------------------------
# 12. Categorical Output Contract
# --------------------------------------------------------------------------------------------------

categorical_activation_values = (
    ARCHITECTURE_DF[
        "categorical_training_activation"
    ]
    .astype(
        str
    )
    .str.strip()
    .str.lower()
    .unique()
    .tolist()
)


categorical_mapping_values = (
    ARCHITECTURE_DF[
        "categorical_probability_mapping"
    ]
    .astype(
        str
    )
    .str.strip()
    .str.lower()
    .unique()
    .tolist()
)


if categorical_activation_values != [
    "gumbel_softmax"
]:

    raise RuntimeError(
        "Unexpected categorical training activation.\n"
        f"Observed: {categorical_activation_values}"
    )


if categorical_mapping_values != [
    "softmax"
]:

    raise RuntimeError(
        "Unexpected categorical probability mapping.\n"
        f"Observed: {categorical_mapping_values}"
    )


print(
    "✓ Categorical training activation : gumbel_softmax"
)

print(
    "✓ Categorical probability mapping : softmax"
)


# --------------------------------------------------------------------------------------------------
# 13. Numerical Activation Contract
# --------------------------------------------------------------------------------------------------

NUMERICAL_ACTIVATION_VALUES = (
    ARCHITECTURE_DF[
        "numerical_activation"
    ]
    .astype(
        str
    )
    .str.strip()
    .str.lower()
    .unique()
    .tolist()
)


if NUMERICAL_ACTIVATION_VALUES != [
    "identity"
]:

    raise RuntimeError(
        "Notebook 08 numerical activation does not match "
        "the frozen architecture artifact.\n"
        f"Observed: {NUMERICAL_ACTIVATION_VALUES}"
    )


print(
    "✓ Numerical activation inherited from Notebook 08 : identity"
)

print(
    "  No architecture override is performed in Notebook 12."
)


# --------------------------------------------------------------------------------------------------
# 14. Critic Output Contract
# --------------------------------------------------------------------------------------------------

CRITIC_OUTPUT_VALUES = (
    ARCHITECTURE_DF[
        "critic_output"
    ]
    .astype(
        str
    )
    .str.strip()
    .str.lower()
    .unique()
    .tolist()
)


if CRITIC_OUTPUT_VALUES != [
    "scalar"
]:

    raise RuntimeError(
        "Unexpected critic output contract.\n"
        f"Observed: {CRITIC_OUTPUT_VALUES}"
    )


print(
    "✓ Critic output contract : scalar"
)


# --------------------------------------------------------------------------------------------------
# 15. Statistical Guidance Contract
# --------------------------------------------------------------------------------------------------
#
# Notebook 08 records the implementation source for statistical guidance
# in the architecture summary. The authoritative value is "notebook_09".
#
# Notebook 12 Section 4 has already loaded the persisted Notebook 03
# statistical guidance artifacts.
#
# This field is therefore treated as an implementation/provenance field,
# not as a Boolean enable/disable flag.
# --------------------------------------------------------------------------------------------------

STATISTICAL_GUIDANCE_VALUES = (
    ARCHITECTURE_DF[
        "statistical_guidance"
    ]
    .astype(
        str
    )
    .str.strip()
    .str.lower()
    .unique()
    .tolist()
)


EXPECTED_STATISTICAL_GUIDANCE_SOURCE = [
    "notebook_09"
]


if STATISTICAL_GUIDANCE_VALUES != (
    EXPECTED_STATISTICAL_GUIDANCE_SOURCE
):

    raise RuntimeError(
        "Unexpected statistical-guidance implementation source.\n"
        f"Expected: {EXPECTED_STATISTICAL_GUIDANCE_SOURCE}\n"
        f"Observed: {STATISTICAL_GUIDANCE_VALUES}"
    )


if "STATISTICAL_GUIDANCE" not in globals():

    raise RuntimeError(
        "STATISTICAL_GUIDANCE is not available.\n"
        "Notebook 03 statistical guidance must be loaded before Section 5."
    )


if set(
    STATISTICAL_GUIDANCE.keys()
) != EXPECTED_DATASETS:

    raise RuntimeError(
        "Statistical-guidance dataset coverage mismatch.\n"
        f"Expected: {sorted(EXPECTED_DATASETS)}\n"
        f"Found   : {sorted(STATISTICAL_GUIDANCE.keys())}"
    )


print(
    "✓ Statistical-guidance implementation source : Notebook 09"
)

print(
    "✓ Notebook 03 statistical-guidance linkage validated."
)


# --------------------------------------------------------------------------------------------------
# 16. Differential Privacy / Accounting Contract
# --------------------------------------------------------------------------------------------------

DP_VALUES = (
    ARCHITECTURE_DF[
        "differential_privacy"
    ]
    .astype(
        str
    )
    .str.strip()
    .str.lower()
    .unique()
    .tolist()
)


ACCOUNTING_VALUES = (
    ARCHITECTURE_DF[
        "privacy_accounting"
    ]
    .astype(
        str
    )
    .str.strip()
    .str.lower()
    .unique()
    .tolist()
)


if not DP_VALUES:

    raise RuntimeError(
        "Notebook 08 differential-privacy contract is empty."
    )


if not ACCOUNTING_VALUES:

    raise RuntimeError(
        "Notebook 08 privacy-accounting contract is empty."
    )


print(
    "✓ Differential-privacy architecture contract loaded."
)

print(
    "✓ Privacy-accounting architecture contract loaded."
)


# --------------------------------------------------------------------------------------------------
# 17. Parameter-Count Artifact Validation
# --------------------------------------------------------------------------------------------------

if PARAMETER_DF.empty:

    raise RuntimeError(
        "Notebook 08 parameter-count artifact is empty."
    )


if "dataset" not in PARAMETER_DF.columns:

    raise RuntimeError(
        "Notebook 08 parameter-count artifact does not contain "
        "the required 'dataset' column."
    )


if (
    PARAMETER_DF[
        "dataset"
    ]
    .duplicated()
    .any()
):

    raise RuntimeError(
        "Duplicate dataset entries detected in parameter-count artifact."
    )


parameter_datasets = set(
    PARAMETER_DF[
        "dataset"
    ].astype(
        str
    )
)


if parameter_datasets != EXPECTED_DATASETS:

    raise RuntimeError(
        "Parameter-count dataset coverage mismatch.\n"
        f"Expected: {sorted(EXPECTED_DATASETS)}\n"
        f"Found   : {sorted(parameter_datasets)}"
    )


print(
    "✓ Parameter-count dataset coverage validated."
)


# --------------------------------------------------------------------------------------------------
# 18. Parameter-Count Cross-Validation
# --------------------------------------------------------------------------------------------------

if "total_trainable_parameters" not in PARAMETER_DF.columns:

    raise RuntimeError(
        "Notebook 08 parameter-count artifact must contain "
        "'total_trainable_parameters'."
    )


for dataset_id in DATASET_IDS:

    architecture_row = ARCHITECTURE_DF[
        ARCHITECTURE_DF[
            "dataset"
        ].astype(
            str
        )
        ==
        dataset_id
    ].iloc[
        0
    ]


    parameter_row = PARAMETER_DF[
        PARAMETER_DF[
            "dataset"
        ].astype(
            str
        )
        ==
        dataset_id
    ].iloc[
        0
    ]


    architecture_total = int(
        architecture_row[
            "total_trainable_parameters"
        ]
    )


    parameter_total = int(
        parameter_row[
            "total_trainable_parameters"
        ]
    )


    if architecture_total != parameter_total:

        raise RuntimeError(
            f"{dataset_id}: total trainable parameter mismatch.\n"
            f"Architecture summary : {architecture_total}\n"
            f"Parameter metadata   : {parameter_total}"
        )


print(
    "✓ Architecture-summary / parameter-count consistency validated."
)


# --------------------------------------------------------------------------------------------------
# 19. Architecture Registry Validation
# --------------------------------------------------------------------------------------------------

if ARCHITECTURE_REGISTRY_DF.empty:

    raise RuntimeError(
        "Notebook 08 architecture registry is empty."
    )


if "dataset" in ARCHITECTURE_REGISTRY_DF.columns:

    registry_datasets = set(
        ARCHITECTURE_REGISTRY_DF[
            "dataset"
        ].astype(
            str
        )
    )


    if not registry_datasets.issuperset(
        EXPECTED_DATASETS
    ):

        raise RuntimeError(
            "Notebook 08 architecture registry does not cover "
            "all canonical datasets.\n"
            f"Expected: {sorted(EXPECTED_DATASETS)}\n"
            f"Found   : {sorted(registry_datasets)}"
        )


print(
    "✓ Architecture registry validated."
)


# --------------------------------------------------------------------------------------------------
# 20. Model Configuration Validation
# --------------------------------------------------------------------------------------------------

if not ARCHITECTURE_CONFIG:

    raise RuntimeError(
        "Notebook 08 model configuration is empty."
    )


def find_configuration_values(
    obj,
    candidate_keys,
    path="",
):

    matches = []

    normalized_candidates = {

        str(key)
        .strip()
        .lower()
        .replace("-", "_")
        .replace(" ", "_")

        for key in candidate_keys
    }


    if isinstance(
        obj,
        dict,
    ):

        for key, value in obj.items():

            normalized_key = (
                str(key)
                .strip()
                .lower()
                .replace("-", "_")
                .replace(" ", "_")
            )


            current_path = (
                f"{path}.{key}"
                if path
                else str(key)
            )


            if normalized_key in normalized_candidates:

                matches.append(
                    (
                        current_path,
                        value,
                    )
                )


            matches.extend(
                find_configuration_values(
                    value,
                    candidate_keys,
                    current_path,
                )
            )


    elif isinstance(
        obj,
        list,
    ):

        for index, value in enumerate(
            obj
        ):

            matches.extend(
                find_configuration_values(
                    value,
                    candidate_keys,
                    f"{path}[{index}]",
                )
            )


    return matches


def validate_single_architecture_config_value(
    candidate_keys,
    expected_value,
    label,
):

    matches = find_configuration_values(
        ARCHITECTURE_CONFIG,
        candidate_keys,
    )


    if not matches:

        raise RuntimeError(
            f"Notebook 08 model configuration does not expose "
            f"required architecture parameter: {label}"
        )


    normalized_values = []


    for path, value in matches:

        if isinstance(
            value,
            bool,
        ):

            normalized_value = value

        elif isinstance(
            value,
            (
                int,
                float,
                np.integer,
                np.floating,
            ),
        ):

            normalized_value = int(
                value
            )

        else:

            normalized_value = str(
                value
            ).strip().lower()


        normalized_values.append(
            (
                path,
                normalized_value,
            )
        )


    unique_values = {
        value
        for _, value
        in normalized_values
    }


    if len(
        unique_values
    ) != 1:

        raise RuntimeError(
            f"Conflicting Notebook 08 configuration values "
            f"for {label}:\n"
            +
            "\n".join(
                f"  {path} = {value}"
                for path, value
                in normalized_values
            )
        )


    observed_value = next(
        iter(
            unique_values
        )
    )


    expected_normalized = (

        str(
            expected_value
        ).strip().lower()

        if isinstance(
            expected_value,
            str,
        )

        else expected_value
    )


    if observed_value != expected_normalized:

        raise RuntimeError(
            f"Notebook 08 configuration mismatch for {label}.\n"
            f"Expected : {expected_normalized}\n"
            f"Observed : {observed_value}"
        )


    return observed_value


validate_single_architecture_config_value(
    ["latent_dim"],
    128,
    "latent dimension",
)


validate_single_architecture_config_value(
    ["hidden_dim_1"],
    256,
    "hidden dimension 1",
)


validate_single_architecture_config_value(
    ["hidden_dim_2"],
    256,
    "hidden dimension 2",
)


validate_single_architecture_config_value(
    ["numerical_activation"],
    "identity",
    "numerical activation",
)


validate_single_architecture_config_value(
    ["categorical_training_activation"],
    "gumbel_softmax",
    "categorical training activation",
)


validate_single_architecture_config_value(
    ["categorical_probability_mapping"],
    "softmax",
    "categorical probability mapping",
)


print(
    "✓ Notebook 08 model configuration cross-validated."
)


# --------------------------------------------------------------------------------------------------
# 21. Create Validated SPP-GAN Architecture Registry
# --------------------------------------------------------------------------------------------------

SPPGAN_ARCHITECTURE = {}


for _, row in ARCHITECTURE_DF.iterrows():

    dataset_id = str(
        row[
            "dataset"
        ]
    )


    SPPGAN_ARCHITECTURE[
        dataset_id
    ] = {

        column:
            row[
                column
            ]

        for column
        in ARCHITECTURE_DF.columns
    }


# --------------------------------------------------------------------------------------------------
# 22. Expose Dataset-Specific Architecture Parameters
# --------------------------------------------------------------------------------------------------

ARCHITECTURE_PARAMETERS = {}


for dataset_id in DATASET_IDS:

    architecture = (
        SPPGAN_ARCHITECTURE[
            dataset_id
        ]
    )


    ARCHITECTURE_PARAMETERS[
        dataset_id
    ] = {

        "generative_dimension":
            int(
                architecture[
                    "generative_dimension"
                ]
            ),

        "transformed_dimension":
            int(
                architecture[
                    "transformed_dimension"
                ]
            ),

        "numerical_features":
            int(
                architecture[
                    "numerical_features"
                ]
            ),

        "categorical_features":
            int(
                architecture[
                    "categorical_features"
                ]
            ),

        "latent_dim":
            int(
                architecture[
                    "latent_dim"
                ]
            ),

        "generator_hidden_dims":
            [
                int(
                    architecture[
                        "generator_hidden_1"
                    ]
                ),
                int(
                    architecture[
                        "generator_hidden_2"
                    ]
                ),
            ],

        "critic_hidden_dims":
            [
                int(
                    architecture[
                        "critic_hidden_1"
                    ]
                ),
                int(
                    architecture[
                        "critic_hidden_2"
                    ]
                ),
            ],

        "total_trainable_parameters":
            int(
                architecture[
                    "total_trainable_parameters"
                ]
            ),

        "numerical_activation":
            str(
                architecture[
                    "numerical_activation"
                ]
            ).strip().lower(),

        "categorical_training_activation":
            str(
                architecture[
                    "categorical_training_activation"
                ]
            ).strip().lower(),

        "categorical_probability_mapping":
            str(
                architecture[
                    "categorical_probability_mapping"
                ]
            ).strip().lower(),

        "hard_decoding":
            str(
                architecture[
                    "hard_decoding"
                ]
            ),

        "critic_output":
            str(
                architecture[
                    "critic_output"
                ]
            ),
    }


# --------------------------------------------------------------------------------------------------
# 23. Native TRAIN / Architecture Linkage
# --------------------------------------------------------------------------------------------------

if "TRAIN_DATAFRAMES" not in globals():

    raise RuntimeError(
        "TRAIN_DATAFRAMES is unavailable.\n"
        "Notebook 12 Section 3 must be completed before Section 5."
    )


for dataset_id in DATASET_IDS:

    if dataset_id not in TRAIN_DATAFRAMES:

        raise RuntimeError(
            f"{dataset_id}: native TRAIN data are unavailable."
        )


    native_column_count = (
        TRAIN_DATAFRAMES[
            dataset_id
        ].shape[
            1
        ]
    )


    generative_dimension = (
        ARCHITECTURE_PARAMETERS[
            dataset_id
        ][
            "generative_dimension"
        ]
    )


    expected_native_columns = (
        generative_dimension
        +
        1
    )


    if native_column_count != expected_native_columns:

        raise RuntimeError(
            f"{dataset_id}: native TRAIN / generative schema mismatch.\n"
            f"Native columns      : {native_column_count}\n"
            f"Generative dimension: {generative_dimension}\n"
            f"Expected native     : {expected_native_columns}"
        )


print(
    "✓ Native TRAIN / generative-schema linkage validated."
)


# --------------------------------------------------------------------------------------------------
# 24. Final Architecture Summary
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("SPP-GAN ARCHITECTURE LOAD AND VALIDATION COMPLETE")
print("=" * 100)

print()
print(
    f"Validated datasets : {len(SPPGAN_ARCHITECTURE)}"
)


for dataset_id in DATASET_IDS:

    architecture = (
        ARCHITECTURE_PARAMETERS[
            dataset_id
        ]
    )


    print()
    print(
        f"{dataset_id}"
    )

    print(
        f"  Generative dimension   : "
        f"{architecture['generative_dimension']}"
    )

    print(
        f"  Transformed dimension  : "
        f"{architecture['transformed_dimension']}"
    )

    print(
        f"  Numerical features     : "
        f"{architecture['numerical_features']}"
    )

    print(
        f"  Categorical features   : "
        f"{architecture['categorical_features']}"
    )

    print(
        f"  Latent dimension       : "
        f"{architecture['latent_dim']}"
    )

    print(
        f"  Generator hidden       : "
        f"{architecture['generator_hidden_dims']}"
    )

    print(
        f"  Critic hidden          : "
        f"{architecture['critic_hidden_dims']}"
    )

    print(
        f"  Trainable parameters   : "
        f"{architecture['total_trainable_parameters']:,}"
    )


# --------------------------------------------------------------------------------------------------
# 25. Final Section 5 Validation
# --------------------------------------------------------------------------------------------------

SECTION_5_CHECKS = {

    "nb08_root_exists":
        NB08_ROOT.exists(),

    "architecture_summary_exists":
        ARCHITECTURE_SUMMARY_PATH.exists(),

    "parameter_count_exists":
        PARAMETER_COUNT_PATH.exists(),

    "model_config_exists":
        MODEL_CONFIG_PATH.exists(),

    "architecture_registry_exists":
        ARCHITECTURE_REGISTRY_PATH.exists(),

    "architecture_summary_nonempty":
        not ARCHITECTURE_DF.empty,

    "parameter_count_nonempty":
        not PARAMETER_DF.empty,

    "architecture_registry_nonempty":
        not ARCHITECTURE_REGISTRY_DF.empty,

    "three_canonical_datasets":
        architecture_datasets == EXPECTED_DATASETS,

    "architecture_unique":
        not ARCHITECTURE_DF[
            "dataset"
        ].duplicated().any(),

    "generative_dimensions_valid":
        all(
            int(
                row[
                    "generative_dimension"
                ]
            )
            ==
            int(
                row[
                    "numerical_features"
                ]
            )
            +
            int(
                row[
                    "categorical_features"
                ]
            )
            +
            1
            for _, row
            in ARCHITECTURE_DF.iterrows()
        ),

    "transformed_dimensions_valid":
        all(
            ARCHITECTURE_PARAMETERS[
                dataset_id
            ][
                "transformed_dimension"
            ]
            ==
            EXPECTED_TRANSFORMED_DIMENSIONS[
                dataset_id
            ]
            for dataset_id
            in DATASET_IDS
        ),

    "latent_dimension_128":
        LATENT_DIMENSIONS == [128],

    "generator_hidden_256_256":
        GENERATOR_HIDDEN_1 == [256]
        and
        GENERATOR_HIDDEN_2 == [256],

    "critic_hidden_256_256":
        CRITIC_HIDDEN_1 == [256]
        and
        CRITIC_HIDDEN_2 == [256],

    "categorical_gumbel_softmax":
        categorical_activation_values
        ==
        ["gumbel_softmax"],

    "categorical_softmax_mapping":
        categorical_mapping_values
        ==
        ["softmax"],

    "numerical_identity":
        NUMERICAL_ACTIVATION_VALUES
        ==
        ["identity"],

    "critic_scalar":
        CRITIC_OUTPUT_VALUES
        ==
        ["scalar"],

    "statistical_guidance_source_notebook_09":
        STATISTICAL_GUIDANCE_VALUES
        ==
        ["notebook_09"],

    "statistical_guidance_dataset_coverage":
        set(
            STATISTICAL_GUIDANCE.keys()
        )
        ==
        EXPECTED_DATASETS,

    "parameter_dataset_coverage":
        parameter_datasets == EXPECTED_DATASETS,

    "parameter_count_cross_validation":
        all(
            int(
                ARCHITECTURE_DF[
                    ARCHITECTURE_DF[
                        "dataset"
                    ].astype(
                        str
                    )
                    ==
                    dataset_id
                ].iloc[
                    0
                ][
                    "total_trainable_parameters"
                ]
            )
            ==
            int(
                PARAMETER_DF[
                    PARAMETER_DF[
                        "dataset"
                    ].astype(
                        str
                    )
                    ==
                    dataset_id
                ].iloc[
                    0
                ][
                    "total_trainable_parameters"
                ]
            )
            for dataset_id
            in DATASET_IDS
        ),

    "model_configuration_valid":
        isinstance(
            ARCHITECTURE_CONFIG,
            dict,
        )
        and
        bool(
            ARCHITECTURE_CONFIG
        ),

    "train_architecture_linkage":
        all(
            dataset_id in TRAIN_DATAFRAMES
            and
            (
                TRAIN_DATAFRAMES[
                    dataset_id
                ].shape[
                    1
                ]
                ==
                ARCHITECTURE_PARAMETERS[
                    dataset_id
                ][
                    "generative_dimension"
                ]
                +
                1
            )
            for dataset_id
            in DATASET_IDS
        ),
}


FAILED_SECTION_5_CHECKS = [

    name

    for name, passed
    in SECTION_5_CHECKS.items()

    if not passed
]


print()
print("-" * 100)
print("SECTION 5 VALIDATION")
print("-" * 100)


for check_name, passed in SECTION_5_CHECKS.items():

    print(
        f"{'✓' if passed else '✗'} "
        f"{check_name}"
    )


if FAILED_SECTION_5_CHECKS:

    raise RuntimeError(
        "Section 5 validation failed:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in FAILED_SECTION_5_CHECKS
        )
    )


# --------------------------------------------------------------------------------------------------
# 26. Completion
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("SECTION 5 — LOAD SPP-GAN ARCHITECTURE COMPLETE")
print("=" * 100)

print(
    "✓ Notebook 08 architecture summary loaded."
)

print(
    "✓ Notebook 08 parameter-count metadata loaded."
)

print(
    "✓ Notebook 08 model configuration loaded."
)

print(
    "✓ Notebook 08 architecture registry loaded."
)

print(
    "✓ Three canonical datasets validated."
)

print(
    "✓ Generative dimensions validated."
)

print(
    "✓ Transformed dimensions validated."
)

print(
    "✓ Generator and critic architecture validated."
)

print(
    "✓ Categorical Gumbel-Softmax contract validated."
)

print(
    "✓ Numerical activation inherited without override."
)

print(
    "✓ Statistical-guidance source Notebook 09 validated."
)

print(
    "✓ Notebook 03 statistical-guidance linkage validated."
)

print(
    "✓ Parameter-count cross-validation passed."
)

print(
    "✓ Native TRAIN / generative-schema linkage passed."
)

print(
    f"✓ Section 5 checks : "
    f"{len(SECTION_5_CHECKS)}/{len(SECTION_5_CHECKS)} PASS"
)

print()
print(
    "STATUS: PASS — SECTION 5 READY FOR FREEZE"
)

print("=" * 100)

5. LOAD SPP-GAN ARCHITECTURE
✓ Notebook 08 root : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08

----------------------------------------------------------------------------------------------------
NOTEBOOK 08 ARCHITECTURE ARTIFACT VERIFICATION
----------------------------------------------------------------------------------------------------
✓ architecture_summary     : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/architecture/sppgan_architecture_summary.csv
✓ parameter_count          : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/metadata/sppgan_parameter_count.csv
✓ model_configuration      : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/config/sppgan_model_configuration.json
✓ architecture_registry    : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_08/metadata/sppgan_architecture_artifacts.csv

----------------------------------------------------------------------

In [6]:
!pip install -q opacus==1.6.0

In [7]:
# ==================================================================================================
# SECTION 6 — LOAD DP MECHANISM
# ==================================================================================================

print("=" * 100)
print("6. LOAD DP MECHANISM")
print("=" * 100)

import importlib.util
from pathlib import Path


# --------------------------------------------------------------------------------------------------
# 1. Verify Notebook 10 Root
# --------------------------------------------------------------------------------------------------

if "NB10_ROOT" not in globals():

    raise RuntimeError(
        "NB10_ROOT is not defined.\n"
        "Notebook 10 canonical root must be initialized before Section 6."
    )

NB10_ROOT = Path(
    NB10_ROOT
)

if not NB10_ROOT.exists():

    raise FileNotFoundError(
        f"Notebook 10 root does not exist:\n{NB10_ROOT}"
    )

if not NB10_ROOT.is_dir():

    raise NotADirectoryError(
        f"Notebook 10 root is not a directory:\n{NB10_ROOT}"
    )

print(
    f"✓ Notebook 10 root : {NB10_ROOT}"
)


# --------------------------------------------------------------------------------------------------
# 2. Canonical DP Module
# --------------------------------------------------------------------------------------------------

DP_MODULE_PATH = (
    NB10_ROOT
    / "models"
    / "sppgan_differential_privacy.py"
)

if not DP_MODULE_PATH.exists():

    raise FileNotFoundError(
        "Notebook 10 DP module not found:\n"
        f"{DP_MODULE_PATH}"
    )

if not DP_MODULE_PATH.is_file():

    raise RuntimeError(
        "Notebook 10 DP module is not a file:\n"
        f"{DP_MODULE_PATH}"
    )

if DP_MODULE_PATH.stat().st_size == 0:

    raise RuntimeError(
        "Notebook 10 DP module is empty:\n"
        f"{DP_MODULE_PATH}"
    )

print(
    f"✓ DP module path    : {DP_MODULE_PATH}"
)


# --------------------------------------------------------------------------------------------------
# 3. Verify Required Runtime Dependency
# --------------------------------------------------------------------------------------------------

EXPECTED_OPACUS_VERSION = "1.6.0"

try:

    import opacus

except ModuleNotFoundError as exc:

    raise RuntimeError(
        "Opacus is not installed in the current runtime.\n\n"
        "Notebook 12 Section 6 requires the validated Notebook 10 "
        "DP environment.\n\n"
        f"Required Opacus version : {EXPECTED_OPACUS_VERSION}\n"
        "Observed                 : NOT INSTALLED\n\n"
        "Install the pinned dependency in Notebook 00, "
        "restart the runtime if required, and rerun Notebook 12 "
        "from Section 1."
    ) from exc


INSTALLED_OPACUS_VERSION = getattr(
    opacus,
    "__version__",
    None,
)

if INSTALLED_OPACUS_VERSION is None:

    raise RuntimeError(
        "Unable to determine the installed Opacus version."
    )

if INSTALLED_OPACUS_VERSION != EXPECTED_OPACUS_VERSION:

    raise RuntimeError(
        "Opacus version mismatch.\n"
        f"Required validated version : {EXPECTED_OPACUS_VERSION}\n"
        f"Installed version           : {INSTALLED_OPACUS_VERSION}\n\n"
        "Do not continue with Notebook 12 until the runtime "
        "matches the validated Notebook 10 environment."
    )

print(
    f"✓ Opacus version   : {INSTALLED_OPACUS_VERSION}"
)


# --------------------------------------------------------------------------------------------------
# 4. Verify Opacus Core Components
# --------------------------------------------------------------------------------------------------

try:

    from opacus import PrivacyEngine

except ImportError as exc:

    raise RuntimeError(
        "Opacus PrivacyEngine could not be imported."
    ) from exc


try:

    from opacus.grad_sample import GradSampleModule

except ImportError as exc:

    raise RuntimeError(
        "Opacus GradSampleModule could not be imported."
    ) from exc


print(
    "✓ PrivacyEngine available."
)

print(
    "✓ GradSampleModule available."
)


# --------------------------------------------------------------------------------------------------
# 5. Dynamically Load Notebook 10 DP Module
# --------------------------------------------------------------------------------------------------

spec = importlib.util.spec_from_file_location(
    "sppgan_differential_privacy_nb12",
    DP_MODULE_PATH,
)

if spec is None:

    raise RuntimeError(
        "Unable to create import specification for the Notebook 10 DP module."
    )

if spec.loader is None:

    raise RuntimeError(
        "Notebook 10 DP module import loader is unavailable."
    )


DP_MODULE = importlib.util.module_from_spec(
    spec
)

try:

    spec.loader.exec_module(
        DP_MODULE
    )

except Exception as exc:

    raise RuntimeError(
        "Notebook 10 DP module failed during import.\n"
        f"Module : {DP_MODULE_PATH}\n"
        f"Error  : {type(exc).__name__}: {exc}"
    ) from exc


print(
    "✓ Notebook 10 DP module imported successfully."
)


# --------------------------------------------------------------------------------------------------
# 6. Required DP Function Contract
# --------------------------------------------------------------------------------------------------

REQUIRED_DP_FUNCTIONS = [
    "wrap_critic_for_per_sample_gradients",
    "make_private_discriminator",
    "dp_discriminator_loss",
    "dp_discriminator_step",
]

missing_dp_functions = [
    name
    for name in REQUIRED_DP_FUNCTIONS
    if not hasattr(
        DP_MODULE,
        name,
    )
]

if missing_dp_functions:

    raise RuntimeError(
        "Notebook 10 DP module is incomplete.\n"
        "Missing required functions:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in missing_dp_functions
        )
    )

print(
    "✓ Required DP function contract validated."
)


for function_name in REQUIRED_DP_FUNCTIONS:

    function_object = getattr(
        DP_MODULE,
        function_name,
    )

    if not callable(
        function_object
    ):

        raise RuntimeError(
            "Required DP component is not callable.\n"
            f"Function : {function_name}"
        )

print(
    "✓ All required DP functions are callable."
)


# --------------------------------------------------------------------------------------------------
# 7. Required DP Class / Component Contract
# --------------------------------------------------------------------------------------------------

if not hasattr(
    DP_MODULE,
    "SPPGANCritic",
):

    raise RuntimeError(
        "Notebook 10 DP module does not expose the required "
        "SPPGANCritic class."
    )

if not callable(
    getattr(
        DP_MODULE,
        "SPPGANCritic",
    )
):

    raise RuntimeError(
        "Notebook 10 SPPGANCritic is not callable."
    )

print(
    "✓ SPPGANCritic class available."
)


# --------------------------------------------------------------------------------------------------
# 8. Validate PrivacyEngine Compatibility
# --------------------------------------------------------------------------------------------------

if PrivacyEngine is None:

    raise RuntimeError(
        "PrivacyEngine resolved to None."
    )

print(
    "✓ Opacus PrivacyEngine compatibility validated."
)


# --------------------------------------------------------------------------------------------------
# 9. Frozen Notebook 10 Privacy Contract
# --------------------------------------------------------------------------------------------------

EXPECTED_DP_CONTRACT = {

    "accountant":
        "rdp",

    "sampling":
        "poisson",

    "clipping":
        "flat",

    "loss_reduction":
        "mean",

    "protected_component":
        "discriminator",

    "generator_private":
        False,

    "statistical_guidance_private":
        False,

    "preprocessing_private":
        False,

    "end_to_end_privacy_claim":
        False,
}


# --------------------------------------------------------------------------------------------------
# 10. Validate Runtime DP Contract Where Available
# --------------------------------------------------------------------------------------------------

def validate_optional_global_contract(
    variable_name,
    expected_value,
    label,
):
    """
    Validate a previously initialized Notebook 10 / Notebook 12
    contract variable when available.

    If the variable is not present, the persisted Notebook 10
    contract remains authoritative for Section 6.
    """

    if variable_name not in globals():

        return True

    observed_value = globals()[
        variable_name
    ]

    if isinstance(
        expected_value,
        str,
    ):

        observed_normalized = str(
            observed_value
        ).strip().lower()

        expected_normalized = expected_value.lower()

        if observed_normalized != expected_normalized:

            raise RuntimeError(
                f"{label} mismatch.\n"
                f"Expected : {expected_normalized}\n"
                f"Observed : {observed_normalized}"
            )

    else:

        if observed_value != expected_value:

            raise RuntimeError(
                f"{label} mismatch.\n"
                f"Expected : {expected_value}\n"
                f"Observed : {observed_value}"
            )

    return True


validate_optional_global_contract(
    "ACCOUNTANT",
    EXPECTED_DP_CONTRACT["accountant"],
    "Notebook 10 accountant",
)

validate_optional_global_contract(
    "SAMPLING_MECHANISM",
    EXPECTED_DP_CONTRACT["sampling"],
    "Notebook 10 sampling mechanism",
)

validate_optional_global_contract(
    "CLIPPING_MECHANISM",
    EXPECTED_DP_CONTRACT["clipping"],
    "Notebook 10 clipping mechanism",
)

validate_optional_global_contract(
    "LOSS_REDUCTION",
    EXPECTED_DP_CONTRACT["loss_reduction"],
    "Notebook 10 loss reduction",
)

print(
    "✓ Notebook 10 RDP / Poisson / flat-L2 / mean contract validated."
)


# --------------------------------------------------------------------------------------------------
# 11. DP Module Function and Class Registries
# --------------------------------------------------------------------------------------------------

DP_FUNCTION_REGISTRY = {
    function_name:
        getattr(
            DP_MODULE,
            function_name,
        )
    for function_name
    in REQUIRED_DP_FUNCTIONS
}

DP_CLASS_REGISTRY = {
    "SPPGANCritic":
        getattr(
            DP_MODULE,
            "SPPGANCritic",
        ),
}


# --------------------------------------------------------------------------------------------------
# 12. Explicit Privacy Boundary
# --------------------------------------------------------------------------------------------------

DP_PRIVACY_BOUNDARY = {
    "protected_component":
        EXPECTED_DP_CONTRACT[
            "protected_component"
        ],

    "generator_private":
        EXPECTED_DP_CONTRACT[
            "generator_private"
        ],

    "statistical_guidance_private":
        EXPECTED_DP_CONTRACT[
            "statistical_guidance_private"
        ],

    "preprocessing_private":
        EXPECTED_DP_CONTRACT[
            "preprocessing_private"
        ],

    "end_to_end_privacy_claim":
        EXPECTED_DP_CONTRACT[
            "end_to_end_privacy_claim"
        ],
}


if (
    DP_PRIVACY_BOUNDARY[
        "end_to_end_privacy_claim"
    ]
    is not False
):

    raise RuntimeError(
        "Notebook 12 Section 6 must not establish "
        "an end-to-end DP claim."
    )


print(
    "✓ DP privacy boundary validated."
)

print(
    "  Protected component : discriminator"
)

print(
    "  Generator private   : False"
)

print(
    "  Statistical guidance private : False"
)

print(
    "  Preprocessing private         : False"
)

print(
    "  End-to-end DP claim           : False"
)


# --------------------------------------------------------------------------------------------------
# 13. Final Section 6 Checks
# --------------------------------------------------------------------------------------------------

SECTION_6_CHECKS = {

    "nb10_root_exists":
        NB10_ROOT.exists(),

    "nb10_root_is_directory":
        NB10_ROOT.is_dir(),

    "dp_module_exists":
        DP_MODULE_PATH.exists(),

    "dp_module_is_file":
        DP_MODULE_PATH.is_file(),

    "dp_module_nonempty":
        DP_MODULE_PATH.stat().st_size > 0,

    "opacus_installed":
        True,

    "opacus_version_validated":
        INSTALLED_OPACUS_VERSION
        ==
        EXPECTED_OPACUS_VERSION,

    "privacy_engine_available":
        PrivacyEngine is not None,

    "grad_sample_module_available":
        GradSampleModule is not None,

    "required_dp_functions_present":
        not missing_dp_functions,

    "required_dp_functions_callable":
        all(
            callable(
                function
            )
            for function
            in DP_FUNCTION_REGISTRY.values()
        ),

    "sppgan_critic_available":
        callable(
            DP_CLASS_REGISTRY[
                "SPPGANCritic"
            ]
        ),

    "dp_module_imported":
        DP_MODULE is not None,

    "rdp_contract":
        validate_optional_global_contract(
            "ACCOUNTANT",
            EXPECTED_DP_CONTRACT[
                "accountant"
            ],
            "RDP accountant",
        ),

    "poisson_contract":
        validate_optional_global_contract(
            "SAMPLING_MECHANISM",
            EXPECTED_DP_CONTRACT[
                "sampling"
            ],
            "Poisson sampling",
        ),

    "flat_clipping_contract":
        validate_optional_global_contract(
            "CLIPPING_MECHANISM",
            EXPECTED_DP_CONTRACT[
                "clipping"
            ],
            "Flat clipping",
        ),

    "mean_loss_reduction_contract":
        validate_optional_global_contract(
            "LOSS_REDUCTION",
            EXPECTED_DP_CONTRACT[
                "loss_reduction"
            ],
            "Mean loss reduction",
        ),

    "protected_component_discriminator":
        DP_PRIVACY_BOUNDARY[
            "protected_component"
        ]
        ==
        "discriminator",

    "generator_not_private":
        DP_PRIVACY_BOUNDARY[
            "generator_private"
        ]
        is False,

    "statistical_guidance_not_private":
        DP_PRIVACY_BOUNDARY[
            "statistical_guidance_private"
        ]
        is False,

    "preprocessing_not_private":
        DP_PRIVACY_BOUNDARY[
            "preprocessing_private"
        ]
        is False,

    "end_to_end_claim_disabled":
        DP_PRIVACY_BOUNDARY[
            "end_to_end_privacy_claim"
        ]
        is False,
}


FAILED_SECTION_6_CHECKS = [
    name
    for name, passed
    in SECTION_6_CHECKS.items()
    if not passed
]


# --------------------------------------------------------------------------------------------------
# 14. Validation Output
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)
print("SECTION 6 VALIDATION")
print("-" * 100)

for check_name, passed in SECTION_6_CHECKS.items():

    print(
        f"{'✓' if passed else '✗'} "
        f"{check_name}"
    )


if FAILED_SECTION_6_CHECKS:

    raise RuntimeError(
        "Section 6 validation failed:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in FAILED_SECTION_6_CHECKS
        )
    )


# --------------------------------------------------------------------------------------------------
# 15. Completion
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("SECTION 6 — LOAD DP MECHANISM COMPLETE")
print("=" * 100)

print(
    "✓ Notebook 10 DP module loaded."
)

print(
    f"✓ Opacus version validated : "
    f"{INSTALLED_OPACUS_VERSION}"
)

print(
    "✓ PrivacyEngine available."
)

print(
    "✓ GradSampleModule available."
)

print(
    "✓ SPPGANCritic available."
)

print(
    "✓ Per-example gradient wrapper available."
)

print(
    "✓ DP discriminator construction available."
)

print(
    "✓ DP discriminator loss available."
)

print(
    "✓ DP discriminator step available."
)

print(
    "✓ RDP / Poisson / flat-L2 / mean contract validated."
)

print(
    "✓ DP privacy boundary validated."
)

print(
    "✓ End-to-end DP claim remains disabled."
)

print(
    f"✓ Section 6 checks : "
    f"{len(SECTION_6_CHECKS)}/"
    f"{len(SECTION_6_CHECKS)} PASS"
)

print()
print(
    "STATUS: PASS — SECTION 6 READY FOR FREEZE"
)

print("=" * 100)

6. LOAD DP MECHANISM
✓ Notebook 10 root : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_10
✓ DP module path    : /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_10/models/sppgan_differential_privacy.py
✓ Opacus version   : 1.6.0
✓ PrivacyEngine available.
✓ GradSampleModule available.
✓ Notebook 10 DP module imported successfully.
✓ Required DP function contract validated.
✓ All required DP functions are callable.
✓ SPPGANCritic class available.
✓ Opacus PrivacyEngine compatibility validated.
✓ Notebook 10 RDP / Poisson / flat-L2 / mean contract validated.
✓ DP privacy boundary validated.
  Protected component : discriminator
  Generator private   : False
  Statistical guidance private : False
  Preprocessing private         : False
  End-to-end DP claim           : False

----------------------------------------------------------------------------------------------------
SECTION 6 VALIDATION
------------------------------------------------------

In [8]:
# ==================================================================================================
# SECTION 7 — VALIDATE ALL INPUTS
# ==================================================================================================

print("=" * 100)
print("7. VALIDATE ALL INPUTS")
print("=" * 100)

import json
import hashlib
import joblib
import numpy as np
import pandas as pd
from pathlib import Path


# --------------------------------------------------------------------------------------------------
# 1. Canonical Dataset Registry
# --------------------------------------------------------------------------------------------------

EXPECTED_DATASET_IDS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]

if list(DATASET_IDS) != EXPECTED_DATASET_IDS:

    raise RuntimeError(
        "Canonical dataset registry mismatch.\n"
        f"Expected : {EXPECTED_DATASET_IDS}\n"
        f"Observed : {list(DATASET_IDS)}"
    )

print(
    "✓ Dataset registry validated."
)


# --------------------------------------------------------------------------------------------------
# 2. Canonical Dataset Contracts
# --------------------------------------------------------------------------------------------------

EXPECTED_TRAIN_ROWS = {
    "adult_income": 34189,
    "bank_marketing": 31647,
    "diabetes_130us": 71236,
}

EXPECTED_NATIVE_COLUMNS = {
    "adult_income": 16,
    "bank_marketing": 18,
    "diabetes_130us": 49,
}

EXPECTED_TRANSFORMED_DIMS = {
    "adult_income": 105,
    "bank_marketing": 51,
    "diabetes_130us": 2329,
}

EXPECTED_GENERATIVE_DIMS = {
    "adult_income": 15,
    "bank_marketing": 17,
    "diabetes_130us": 48,
}

EXPECTED_NUMERICAL_FEATURES = {
    "adult_income": 6,
    "bank_marketing": 7,
    "diabetes_130us": 11,
}

EXPECTED_CATEGORICAL_FEATURES = {
    "adult_income": 8,
    "bank_marketing": 9,
    "diabetes_130us": 36,
}

EXPECTED_TARGET_COLUMNS = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted",
}

EXPECTED_PROVENANCE_COLUMNS = {
    "adult_income": "__original_row_id__",
    "bank_marketing": "__original_row_id__",
    "diabetes_130us": "__original_row_id__",
}


# --------------------------------------------------------------------------------------------------
# 3. Verify Notebook 02 Canonical Artifact Layer
# --------------------------------------------------------------------------------------------------

if "PROJECT_ROOT" not in globals():

    raise RuntimeError(
        "PROJECT_ROOT is not available.\n"
        "Run Notebook 12 Section 2 before Section 7."
    )


NB02_ROOT = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "notebook_02"
)

NB02_NATIVE_ROOT = (
    NB02_ROOT
    / "native"
)

NB02_SCHEMA_ROOT = (
    NB02_ROOT
    / "schemas"
)

NB02_METADATA_ROOT = (
    NB02_SCHEMA_ROOT
    / "metadata"
)

NB02_NATIVE_MANIFEST = (
    NB02_NATIVE_ROOT
    / "native_dataset_manifest.csv"
)

NB02_PREPROCESSOR_MANIFEST = (
    NB02_SCHEMA_ROOT
    / "preprocessor_manifest.csv"
)


REQUIRED_NB02_PATHS = {

    "Notebook 02 root":
        NB02_ROOT,

    "Native root":
        NB02_NATIVE_ROOT,

    "Schema root":
        NB02_SCHEMA_ROOT,

    "Metadata root":
        NB02_METADATA_ROOT,

    "Native manifest":
        NB02_NATIVE_MANIFEST,

    "Preprocessor manifest":
        NB02_PREPROCESSOR_MANIFEST,
}


for label, path in REQUIRED_NB02_PATHS.items():

    if not path.exists():

        raise FileNotFoundError(
            f"Required Notebook 02 artifact is missing.\n"
            f"{label} : {path}"
        )


print(
    f"✓ Notebook 02 root     : {NB02_ROOT}"
)

print(
    f"✓ Native manifest      : {NB02_NATIVE_MANIFEST}"
)

print(
    f"✓ Preprocessor manifest: {NB02_PREPROCESSOR_MANIFEST}"
)


# --------------------------------------------------------------------------------------------------
# 4. Load Native Dataset Manifest
# --------------------------------------------------------------------------------------------------

NATIVE_MANIFEST_DF = pd.read_csv(
    NB02_NATIVE_MANIFEST,
    low_memory=False,
)


if NATIVE_MANIFEST_DF.empty:

    raise RuntimeError(
        "Notebook 02 native_dataset_manifest.csv is empty."
    )


REQUIRED_MANIFEST_COLUMNS = [
    "dataset_id",
    "split",
    "absolute_path",
    "rows",
    "columns",
    "generative_columns",
    "target_column",
    "provenance_column",
    "identifier_columns",
    "provenance_present",
    "target_present",
    "identifiers_excluded",
    "file_exists",
    "reload_validation",
    "status",
]


missing_manifest_columns = [
    column
    for column in REQUIRED_MANIFEST_COLUMNS
    if column not in NATIVE_MANIFEST_DF.columns
]


if missing_manifest_columns:

    raise RuntimeError(
        "Notebook 02 native manifest is missing required columns:\n"
        f"{missing_manifest_columns}"
    )


print(
    f"✓ Native manifest loaded : "
    f"{len(NATIVE_MANIFEST_DF)} rows"
)

print(
    "✓ Native manifest schema validated."
)


# --------------------------------------------------------------------------------------------------
# 5. Resolve Exactly One Validated TRAIN Record Per Dataset
# --------------------------------------------------------------------------------------------------

TRAIN_MANIFEST_RECORDS = {}


for dataset_id in DATASET_IDS:

    matches = NATIVE_MANIFEST_DF[
        (
            NATIVE_MANIFEST_DF["dataset_id"]
            .astype(str)
            .str.strip()
            ==
            dataset_id
        )
        &
        (
            NATIVE_MANIFEST_DF["split"]
            .astype(str)
            .str.strip()
            .str.lower()
            ==
            "train"
        )
    ]

    if len(matches) != 1:

        raise RuntimeError(
            f"{dataset_id}: expected exactly one Notebook 02 "
            f"TRAIN manifest record, found {len(matches)}."
        )

    record = matches.iloc[0].to_dict()

    if str(
        record["status"]
    ).strip().upper() != "PASS":

        raise RuntimeError(
            f"{dataset_id}: Notebook 02 TRAIN manifest status "
            "is not PASS."
        )

    if not bool(
        record["file_exists"]
    ):

        raise FileNotFoundError(
            f"{dataset_id}: Notebook 02 TRAIN file is marked unavailable."
        )

    if not bool(
        record["reload_validation"]
    ):

        raise RuntimeError(
            f"{dataset_id}: Notebook 02 TRAIN reload validation "
            "did not PASS."
        )

    TRAIN_MANIFEST_RECORDS[
        dataset_id
    ] = record


print(
    "✓ Exactly one validated TRAIN manifest record resolved for each dataset."
)


# --------------------------------------------------------------------------------------------------
# 6. Load Authoritative Native TRAIN Data
# --------------------------------------------------------------------------------------------------

TRAINING_DATA = {}

TRAINING_SOURCE_PATHS = {}

TRAINING_SOURCE_HASHES = {}


for dataset_id in DATASET_IDS:

    record = TRAIN_MANIFEST_RECORDS[
        dataset_id
    ]

    train_path = Path(
        str(
            record["absolute_path"]
        )
    )

    if not train_path.exists():

        raise FileNotFoundError(
            f"{dataset_id}: authoritative Notebook 02 TRAIN "
            f"file does not exist:\n{train_path}"
        )

    if not train_path.is_file():

        raise RuntimeError(
            f"{dataset_id}: TRAIN path is not a file:\n{train_path}"
        )

    train_df = pd.read_csv(
        train_path,
        low_memory=False,
    )

    expected_rows = EXPECTED_TRAIN_ROWS[
        dataset_id
    ]

    expected_columns = EXPECTED_NATIVE_COLUMNS[
        dataset_id
    ]

    if len(train_df) != expected_rows:

        raise RuntimeError(
            f"{dataset_id}: TRAIN row-count mismatch.\n"
            f"Expected : {expected_rows:,}\n"
            f"Observed : {len(train_df):,}"
        )

    if train_df.shape[1] != expected_columns:

        raise RuntimeError(
            f"{dataset_id}: native TRAIN column-count mismatch.\n"
            f"Expected : {expected_columns}\n"
            f"Observed : {train_df.shape[1]}"
        )

    if train_df.empty:

        raise RuntimeError(
            f"{dataset_id}: native TRAIN dataframe is empty."
        )

    if train_df.columns.duplicated().any():

        duplicated_columns = (
            train_df.columns[
                train_df.columns.duplicated()
            ]
            .tolist()
        )

        raise RuntimeError(
            f"{dataset_id}: duplicate TRAIN columns detected:\n"
            f"{duplicated_columns}"
        )

    target_column = EXPECTED_TARGET_COLUMNS[
        dataset_id
    ]

    provenance_column = EXPECTED_PROVENANCE_COLUMNS[
        dataset_id
    ]

    if target_column not in train_df.columns:

        raise RuntimeError(
            f"{dataset_id}: target column '{target_column}' "
            "is missing from TRAIN data."
        )

    if provenance_column not in train_df.columns:

        raise RuntimeError(
            f"{dataset_id}: provenance column "
            f"'{provenance_column}' is missing from TRAIN data."
        )

    # ----------------------------------------------------------------------------------------------
    # Manifest target / provenance validation
    # ----------------------------------------------------------------------------------------------

    manifest_target = str(
        record["target_column"]
    ).strip()

    if manifest_target != target_column:

        raise RuntimeError(
            f"{dataset_id}: manifest target mismatch.\n"
            f"Expected : {target_column}\n"
            f"Observed : {manifest_target}"
        )

    manifest_provenance = str(
        record["provenance_column"]
    ).strip()

    if manifest_provenance != provenance_column:

        raise RuntimeError(
            f"{dataset_id}: manifest provenance mismatch.\n"
            f"Expected : {provenance_column}\n"
            f"Observed : {manifest_provenance}"
        )

    # ----------------------------------------------------------------------------------------------
    # Native generative dimension
    # ----------------------------------------------------------------------------------------------

    native_without_provenance = [
        column
        for column in train_df.columns
        if column != provenance_column
    ]

    if len(native_without_provenance) != EXPECTED_GENERATIVE_DIMS[
        dataset_id
    ]:

        raise RuntimeError(
            f"{dataset_id}: native generative dimension mismatch.\n"
            f"Expected : "
            f"{EXPECTED_GENERATIVE_DIMS[dataset_id]}\n"
            f"Observed : {len(native_without_provenance)}"
        )

    if target_column not in native_without_provenance:

        raise RuntimeError(
            f"{dataset_id}: target is absent from generative schema."
        )

    if native_without_provenance[-1] != target_column:

        raise RuntimeError(
            f"{dataset_id}: target is not the final generative column.\n"
            f"Expected final column : {target_column}\n"
            f"Observed final column : {native_without_provenance[-1]}"
        )

    # ----------------------------------------------------------------------------------------------
    # Manifest row/column validation
    # ----------------------------------------------------------------------------------------------

    if int(
        record["rows"]
    ) != len(train_df):

        raise RuntimeError(
            f"{dataset_id}: manifest TRAIN row count does not "
            "match loaded data."
        )

    if int(
        record["columns"]
    ) != train_df.shape[1]:

        raise RuntimeError(
            f"{dataset_id}: manifest native column count does not "
            "match loaded data."
        )

    # ----------------------------------------------------------------------------------------------
    # SHA-256 integrity
    # ----------------------------------------------------------------------------------------------

    sha256 = hashlib.sha256()

    with open(
        train_path,
        "rb",
    ) as handle:

        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):

            sha256.update(
                chunk
            )

    actual_sha256 = sha256.hexdigest()

    manifest_hash = str(
        record.get(
            "sha256",
            "",
        )
    ).strip()

    if manifest_hash:

        if actual_sha256.lower() != manifest_hash.lower():

            raise RuntimeError(
                f"{dataset_id}: TRAIN SHA-256 mismatch.\n"
                f"Manifest: {manifest_hash}\n"
                f"Actual  : {actual_sha256}"
            )

    TRAINING_DATA[
        dataset_id
    ] = train_df

    TRAINING_SOURCE_PATHS[
        dataset_id
    ] = str(
        train_path
    )

    TRAINING_SOURCE_HASHES[
        dataset_id
    ] = actual_sha256

    print(
        f"✓ {dataset_id:<18} "
        f"TRAIN loaded : "
        f"{train_df.shape}"
    )


print(
    "✓ Authoritative Notebook 02 native TRAIN data loaded."
)

print(
    "✓ TRAIN-only data policy preserved."
)


# --------------------------------------------------------------------------------------------------
# 7. Load Notebook 02 Preprocessing Metadata
# --------------------------------------------------------------------------------------------------

PREPROCESSING_METADATA = {}


REQUIRED_PREPROCESSING_METADATA_FIELDS = [

    "dataset_id",
    "training_rows",
    "input_columns",
    "numeric_columns",
    "categorical_columns",
    "target_column",
    "target_retained_in_training_dataset",
    "target_excluded_from_preprocessor_input",
    "identifier_columns",
    "provenance_column",
    "provenance_excluded_from_modeling",
    "identifiers_excluded_from_modeling",
    "generative_columns",
    "fit_dataset",
    "fit_scope",
    "fit_policy",
    "preprocessor_artifact",
    "schema_artifact",
    "preprocessing_feature_count",
    "generative_column_count",
    "target_retained_in_generative_schema",
    "target_excluded_from_transformed_features",
    "raw_target_manually_appended",
    "identifiers_excluded",
    "provenance_excluded_from_model_input",
    "transformed_feature_count",
    "transformed_feature_names",
]


for dataset_id in DATASET_IDS:

    metadata_path = (
        NB02_METADATA_ROOT
        /
        f"{dataset_id}_preprocessing_metadata.json"
    )

    if not metadata_path.exists():

        raise FileNotFoundError(
            f"{dataset_id}: Notebook 02 preprocessing metadata "
            f"not found:\n{metadata_path}"
        )

    with open(
        metadata_path,
        "r",
        encoding="utf-8",
    ) as handle:

        metadata = json.load(
            handle
        )

    if not isinstance(
        metadata,
        dict,
    ):

        raise RuntimeError(
            f"{dataset_id}: preprocessing metadata is not a JSON object."
        )

    missing_fields = [
        field
        for field in REQUIRED_PREPROCESSING_METADATA_FIELDS
        if field not in metadata
    ]

    if missing_fields:

        raise RuntimeError(
            f"{dataset_id}: preprocessing metadata is missing:\n"
            f"{missing_fields}"
        )

    if metadata["dataset_id"] != dataset_id:

        raise RuntimeError(
            f"{dataset_id}: metadata dataset identity mismatch."
        )

    if int(
        metadata["training_rows"]
    ) != EXPECTED_TRAIN_ROWS[dataset_id]:

        raise RuntimeError(
            f"{dataset_id}: metadata training-row mismatch."
        )

    if metadata["target_column"] != EXPECTED_TARGET_COLUMNS[dataset_id]:

        raise RuntimeError(
            f"{dataset_id}: metadata target mismatch."
        )

    if metadata["provenance_column"] != EXPECTED_PROVENANCE_COLUMNS[dataset_id]:

        raise RuntimeError(
            f"{dataset_id}: metadata provenance mismatch."
        )

    if int(
        metadata["generative_column_count"]
    ) != EXPECTED_GENERATIVE_DIMS[dataset_id]:

        raise RuntimeError(
            f"{dataset_id}: metadata generative dimension mismatch."
        )

    if int(
        metadata["transformed_feature_count"]
    ) != EXPECTED_TRANSFORMED_DIMS[dataset_id]:

        raise RuntimeError(
            f"{dataset_id}: metadata transformed dimension mismatch."
        )

    expected_preprocessing_count = (
        EXPECTED_NUMERICAL_FEATURES[dataset_id]
        +
        EXPECTED_CATEGORICAL_FEATURES[dataset_id]
    )

    if int(
        metadata["preprocessing_feature_count"]
    ) != expected_preprocessing_count:

        raise RuntimeError(
            f"{dataset_id}: preprocessing feature count mismatch.\n"
            f"Expected : {expected_preprocessing_count}\n"
            f"Observed : {metadata['preprocessing_feature_count']}"
        )

    PREPROCESSING_METADATA[
        dataset_id
    ] = metadata

    print(
        f"✓ {dataset_id:<18} "
        f"Notebook 02 preprocessing metadata validated."
    )


print(
    "✓ Notebook 02 preprocessing metadata validated."
)


# --------------------------------------------------------------------------------------------------
# 8. Validate Authoritative Preprocessing and Generative Schemas
# --------------------------------------------------------------------------------------------------
#
# IMPORTANT:
#
# Notebook 02 persisted metadata is authoritative for column ORDER.
#
# Do NOT reconstruct generative_columns as:
#
#     numeric_columns + categorical_columns + target
#
# because Notebook 02 may preserve the native/generative dataset order.
#
# The authoritative contracts are:
#
#     input_columns       -> preprocessing input order
#     generative_columns  -> generative schema order
#
# The target is retained in the generative schema but excluded from
# preprocessing input.
# --------------------------------------------------------------------------------------------------

PREPROCESSING_COLUMNS = {}

GENERATIVE_COLUMNS = {}


for dataset_id in DATASET_IDS:

    train_df = TRAINING_DATA[
        dataset_id
    ]

    metadata = PREPROCESSING_METADATA[
        dataset_id
    ]

    numeric_columns = list(
        metadata["numeric_columns"]
    )

    categorical_columns = list(
        metadata["categorical_columns"]
    )

    target_column = str(
        metadata["target_column"]
    ).strip()

    provenance_column = str(
        metadata["provenance_column"]
    ).strip()

    input_columns = list(
        metadata["input_columns"]
    )

    generative_columns = list(
        metadata["generative_columns"]
    )

    # ----------------------------------------------------------------------------------------------
    # Numeric/categorical partition
    # ----------------------------------------------------------------------------------------------

    if set(numeric_columns).intersection(
        categorical_columns
    ):

        raise RuntimeError(
            f"{dataset_id}: numeric and categorical columns overlap."
        )

    if len(numeric_columns) != EXPECTED_NUMERICAL_FEATURES[dataset_id]:

        raise RuntimeError(
            f"{dataset_id}: numeric feature count mismatch.\n"
            f"Expected : {EXPECTED_NUMERICAL_FEATURES[dataset_id]}\n"
            f"Observed : {len(numeric_columns)}"
        )

    if len(categorical_columns) != EXPECTED_CATEGORICAL_FEATURES[dataset_id]:

        raise RuntimeError(
            f"{dataset_id}: categorical feature count mismatch.\n"
            f"Expected : {EXPECTED_CATEGORICAL_FEATURES[dataset_id]}\n"
            f"Observed : {len(categorical_columns)}"
        )

    expected_preprocessing_count = (
        EXPECTED_NUMERICAL_FEATURES[dataset_id]
        +
        EXPECTED_CATEGORICAL_FEATURES[dataset_id]
    )

    # ----------------------------------------------------------------------------------------------
    # Authoritative preprocessing input schema
    # ----------------------------------------------------------------------------------------------

    if len(input_columns) != expected_preprocessing_count:

        raise RuntimeError(
            f"{dataset_id}: Notebook 02 input-column count mismatch.\n"
            f"Expected : {expected_preprocessing_count}\n"
            f"Observed : {len(input_columns)}"
        )

    if len(set(input_columns)) != len(input_columns):

        raise RuntimeError(
            f"{dataset_id}: duplicate columns detected in "
            "Notebook 02 preprocessing input schema."
        )

    expected_preprocessing_set = (
        set(numeric_columns)
        |
        set(categorical_columns)
    )

    if set(input_columns) != expected_preprocessing_set:

        raise RuntimeError(
            f"{dataset_id}: Notebook 02 input_columns do not match "
            "the numeric + categorical feature partition.\n"
            f"Input columns : {input_columns}\n"
            f"Expected set  : {sorted(expected_preprocessing_set)}"
        )

    # ----------------------------------------------------------------------------------------------
    # Target policy
    # ----------------------------------------------------------------------------------------------

    if target_column in input_columns:

        raise RuntimeError(
            f"{dataset_id}: target '{target_column}' incorrectly "
            "appears in preprocessing input."
        )

    if target_column not in generative_columns:

        raise RuntimeError(
            f"{dataset_id}: target '{target_column}' is not retained "
            "in the generative schema."
        )

    if generative_columns[-1] != target_column:

        raise RuntimeError(
            f"{dataset_id}: target is not the final generative column.\n"
            f"Expected final column : {target_column}\n"
            f"Observed final column : {generative_columns[-1]}"
        )

    # ----------------------------------------------------------------------------------------------
    # Generative schema dimension
    # ----------------------------------------------------------------------------------------------

    if len(generative_columns) != EXPECTED_GENERATIVE_DIMS[dataset_id]:

        raise RuntimeError(
            f"{dataset_id}: generative-column count mismatch.\n"
            f"Expected : {EXPECTED_GENERATIVE_DIMS[dataset_id]}\n"
            f"Observed : {len(generative_columns)}"
        )

    if len(set(generative_columns)) != len(generative_columns):

        raise RuntimeError(
            f"{dataset_id}: duplicate columns detected in "
            "generative schema."
        )

    # ----------------------------------------------------------------------------------------------
    # Generative feature set
    #
    # Set equality is required.
    # Order comes exclusively from Notebook 02 metadata.
    # ----------------------------------------------------------------------------------------------

    expected_generative_set = (
        expected_preprocessing_set
        |
        {target_column}
    )

    if set(generative_columns) != expected_generative_set:

        raise RuntimeError(
            f"{dataset_id}: generative schema feature set mismatch.\n"
            f"Expected set : {sorted(expected_generative_set)}\n"
            f"Observed set : {sorted(generative_columns)}"
        )

    # ----------------------------------------------------------------------------------------------
    # Native TRAIN schema
    #
    # Native schema:
    #
    #     provenance + authoritative generative_columns
    # ----------------------------------------------------------------------------------------------

    expected_native_columns = (
        [provenance_column]
        +
        generative_columns
    )

    if list(train_df.columns) != expected_native_columns:

        raise RuntimeError(
            f"{dataset_id}: native column order/schema mismatch.\n"
            f"Expected: {expected_native_columns}\n"
            f"Observed: {list(train_df.columns)}"
        )

    # ----------------------------------------------------------------------------------------------
    # Provenance policy
    # ----------------------------------------------------------------------------------------------

    if provenance_column in input_columns:

        raise RuntimeError(
            f"{dataset_id}: provenance column "
            f"'{provenance_column}' incorrectly appears "
            "in preprocessing input."
        )

    if provenance_column in generative_columns:

        raise RuntimeError(
            f"{dataset_id}: provenance column "
            f"'{provenance_column}' incorrectly appears "
            "in generative schema."
        )

    # ----------------------------------------------------------------------------------------------
    # Identifier policy
    # ----------------------------------------------------------------------------------------------

    identifier_columns = list(
        metadata["identifier_columns"]
    )

    identifier_overlap = (
        set(identifier_columns)
        .intersection(
            set(generative_columns)
        )
    )

    if identifier_overlap:

        raise RuntimeError(
            f"{dataset_id}: excluded identifier columns appear "
            f"in generative schema:\n"
            f"{sorted(identifier_overlap)}"
        )

    PREPROCESSING_COLUMNS[
        dataset_id
    ] = input_columns

    GENERATIVE_COLUMNS[
        dataset_id
    ] = generative_columns

    print(
        f"✓ {dataset_id:<18} "
        f"preprocessing={len(input_columns):>2} | "
        f"generative={len(generative_columns):>2} | "
        f"target={target_column}"
    )


print(
    "✓ Notebook 02 preprocessing/generative schema contract validated."
)


# --------------------------------------------------------------------------------------------------
# 9. Load Frozen Notebook 02 Preprocessors
# --------------------------------------------------------------------------------------------------

FITTED_PREPROCESSORS = {}


for dataset_id in DATASET_IDS:

    metadata = PREPROCESSING_METADATA[
        dataset_id
    ]

    preprocessor_path = Path(
        metadata["preprocessor_artifact"]
    )

    if not preprocessor_path.exists():

        raise FileNotFoundError(
            f"{dataset_id}: fitted Notebook 02 preprocessor "
            f"not found:\n{preprocessor_path}"
        )

    if not preprocessor_path.is_file():

        raise RuntimeError(
            f"{dataset_id}: preprocessor artifact is not a file:\n"
            f"{preprocessor_path}"
        )

    preprocessor = joblib.load(
        preprocessor_path
    )

    if preprocessor is None:

        raise RuntimeError(
            f"{dataset_id}: loaded preprocessor is None."
        )

    if not hasattr(
        preprocessor,
        "transform",
    ):

        raise RuntimeError(
            f"{dataset_id}: fitted preprocessor does not expose transform()."
        )

    FITTED_PREPROCESSORS[
        dataset_id
    ] = preprocessor

    print(
        f"✓ {dataset_id:<18} "
        f"fitted preprocessor loaded."
    )


print(
    "✓ Frozen Notebook 02 preprocessors loaded."
)


# --------------------------------------------------------------------------------------------------
# 10. Transform TRAIN Using Frozen Notebook 02 Preprocessors
# --------------------------------------------------------------------------------------------------

TRAIN_ARRAYS = {}

TRANSFORMED_TRAIN_SHAPES = {}


for dataset_id in DATASET_IDS:

    train_df = TRAINING_DATA[
        dataset_id
    ]

    preprocessing_columns = PREPROCESSING_COLUMNS[
        dataset_id
    ]

    preprocessor = FITTED_PREPROCESSORS[
        dataset_id
    ]

    X_train_native = train_df[
        preprocessing_columns
    ].copy()

    transformed = preprocessor.transform(
        X_train_native
    )

    if hasattr(
        transformed,
        "toarray",
    ):

        transformed = transformed.toarray()

    transformed = np.asarray(
        transformed,
        dtype=np.float32,
    )

    if transformed.ndim != 2:

        raise RuntimeError(
            f"{dataset_id}: transformed TRAIN data must be 2-dimensional."
        )

    expected_shape = (
        EXPECTED_TRAIN_ROWS[dataset_id],
        EXPECTED_TRANSFORMED_DIMS[dataset_id],
    )

    if transformed.shape != expected_shape:

        raise RuntimeError(
            f"{dataset_id}: transformed TRAIN shape mismatch.\n"
            f"Expected : {expected_shape}\n"
            f"Observed : {transformed.shape}"
        )

    if not np.isfinite(
        transformed
    ).all():

        raise RuntimeError(
            f"{dataset_id}: transformed TRAIN contains "
            "NaN or Inf values."
        )

    TRAIN_ARRAYS[
        dataset_id
    ] = transformed

    TRANSFORMED_TRAIN_SHAPES[
        dataset_id
    ] = tuple(
        transformed.shape
    )

    print(
        f"✓ {dataset_id:<18} "
        f"transformed TRAIN : "
        f"{transformed.shape}"
    )


print(
    "✓ TRAIN_ARRAYS constructed from frozen Notebook 02 preprocessors."
)


# --------------------------------------------------------------------------------------------------
# 11. Validate Statistical Guidance
# --------------------------------------------------------------------------------------------------

if "STATISTICAL_GUIDANCE" not in globals():

    raise RuntimeError(
        "STATISTICAL_GUIDANCE is not available.\n"
        "Run Notebook 12 Section 4 before Section 7."
    )


missing_guidance = [
    dataset_id
    for dataset_id in DATASET_IDS
    if dataset_id not in STATISTICAL_GUIDANCE
]


if missing_guidance:

    raise RuntimeError(
        "Statistical guidance is missing for:\n"
        f"{missing_guidance}"
    )


for dataset_id in DATASET_IDS:

    guidance = STATISTICAL_GUIDANCE[
        dataset_id
    ]

    if not isinstance(
        guidance,
        dict,
    ):

        raise RuntimeError(
            f"{dataset_id}: statistical guidance must be a dictionary."
        )


print(
    "✓ Statistical guidance validated."
)


# --------------------------------------------------------------------------------------------------
# 12. Validate Notebook 08 Architecture Dimensions
# --------------------------------------------------------------------------------------------------

if "SPPGAN_ARCHITECTURE" not in globals():

    raise RuntimeError(
        "SPPGAN_ARCHITECTURE is not available.\n"
        "Run Notebook 12 Section 5 before Section 7."
    )


for dataset_id in DATASET_IDS:

    architecture = SPPGAN_ARCHITECTURE[
        dataset_id
    ]

    if int(
        architecture["transformed_dimension"]
    ) != EXPECTED_TRANSFORMED_DIMS[dataset_id]:

        raise RuntimeError(
            f"{dataset_id}: Notebook 08 transformed dimension mismatch."
        )

    if int(
        architecture["generative_dimension"]
    ) != EXPECTED_GENERATIVE_DIMS[dataset_id]:

        raise RuntimeError(
            f"{dataset_id}: Notebook 08 generative dimension mismatch."
        )

    if int(
        architecture["numerical_features"]
    ) != EXPECTED_NUMERICAL_FEATURES[dataset_id]:

        raise RuntimeError(
            f"{dataset_id}: Notebook 08 numerical feature count mismatch."
        )

    if int(
        architecture["categorical_features"]
    ) != EXPECTED_CATEGORICAL_FEATURES[dataset_id]:

        raise RuntimeError(
            f"{dataset_id}: Notebook 08 categorical feature count mismatch."
        )

    if int(
        architecture["latent_dim"]
    ) != 128:

        raise RuntimeError(
            f"{dataset_id}: latent dimension mismatch."
        )

    if int(
        architecture["generator_hidden_1"]
    ) != 256:

        raise RuntimeError(
            f"{dataset_id}: generator hidden dimension 1 mismatch."
        )

    if int(
        architecture["generator_hidden_2"]
    ) != 256:

        raise RuntimeError(
            f"{dataset_id}: generator hidden dimension 2 mismatch."
        )

    if int(
        architecture["critic_hidden_1"]
    ) != 256:

        raise RuntimeError(
            f"{dataset_id}: critic hidden dimension 1 mismatch."
        )

    if int(
        architecture["critic_hidden_2"]
    ) != 256:

        raise RuntimeError(
            f"{dataset_id}: critic hidden dimension 2 mismatch."
        )


print(
    "✓ Notebook 08 architecture dimensions validated."
)


# --------------------------------------------------------------------------------------------------
# 13. Validate DP Parameters
# --------------------------------------------------------------------------------------------------

REQUIRED_DP_CONFIGURATION = {
    "target_epsilon": 5.0,
    "max_grad_norm": 1.0,
    "batch_size": 128,
    "epochs": 300,
    "accountant": "rdp",
    "sampling": "poisson",
    "clipping": "flat",
    "loss_reduction": "mean",
}


if "TARGET_EPSILON" not in globals():

    raise RuntimeError(
        "TARGET_EPSILON is not available.\n"
        "Run Notebook 12 Section 2 before Section 7."
    )

if "MAX_GRAD_NORM" not in globals():

    raise RuntimeError(
        "MAX_GRAD_NORM is not available."
    )

if "DP_BATCH_SIZE" not in globals():

    raise RuntimeError(
        "DP_BATCH_SIZE is not available."
    )

if "DP_EPOCHS" not in globals():

    raise RuntimeError(
        "DP_EPOCHS is not available."
    )

if "ACCOUNTANT" not in globals():

    raise RuntimeError(
        "ACCOUNTANT is not available."
    )

if "SAMPLING_MECHANISM" not in globals():

    raise RuntimeError(
        "SAMPLING_MECHANISM is not available."
    )

if "CLIPPING_MECHANISM" not in globals():

    raise RuntimeError(
        "CLIPPING_MECHANISM is not available."
    )

if "LOSS_REDUCTION" not in globals():

    raise RuntimeError(
        "LOSS_REDUCTION is not available."
    )


if float(
    TARGET_EPSILON
) != REQUIRED_DP_CONFIGURATION["target_epsilon"]:

    raise RuntimeError(
        "Target epsilon mismatch."
    )

if float(
    MAX_GRAD_NORM
) != REQUIRED_DP_CONFIGURATION["max_grad_norm"]:

    raise RuntimeError(
        "Maximum gradient norm mismatch."
    )

if int(
    DP_BATCH_SIZE
) != REQUIRED_DP_CONFIGURATION["batch_size"]:

    raise RuntimeError(
        "DP batch-size mismatch."
    )

if int(
    DP_EPOCHS
) != REQUIRED_DP_CONFIGURATION["epochs"]:

    raise RuntimeError(
        "DP epoch-count mismatch."
    )

if str(
    ACCOUNTANT
).strip().lower() != REQUIRED_DP_CONFIGURATION["accountant"]:

    raise RuntimeError(
        "Notebook 12 requires RDP accounting."
    )

if str(
    SAMPLING_MECHANISM
).strip().lower() != REQUIRED_DP_CONFIGURATION["sampling"]:

    raise RuntimeError(
        "Notebook 12 requires Poisson sampling."
    )

if str(
    CLIPPING_MECHANISM
).strip().lower() != REQUIRED_DP_CONFIGURATION["clipping"]:

    raise RuntimeError(
        "Notebook 12 requires flat L2 clipping."
    )

if str(
    LOSS_REDUCTION
).strip().lower() != REQUIRED_DP_CONFIGURATION["loss_reduction"]:

    raise RuntimeError(
        "Notebook 12 requires mean loss reduction."
    )


print(
    "✓ DP parameters validated."
)


# --------------------------------------------------------------------------------------------------
# 14. Validate Privacy Boundary
# --------------------------------------------------------------------------------------------------

REQUIRED_PRIVACY_BOUNDARY = {
    "generator_private": False,
    "statistical_guidance_private": False,
    "preprocessing_private": False,
    "end_to_end_privacy_claim": False,
}


if "GENERATOR_PRIVATE" not in globals():

    raise RuntimeError(
        "GENERATOR_PRIVATE is not available."
    )

if "STATISTICAL_GUIDANCE_PRIVATE" not in globals():

    raise RuntimeError(
        "STATISTICAL_GUIDANCE_PRIVATE is not available."
    )

if "PREPROCESSING_PRIVATE" not in globals():

    raise RuntimeError(
        "PREPROCESSING_PRIVATE is not available."
    )

if "END_TO_END_PRIVACY_CLAIM" not in globals():

    raise RuntimeError(
        "END_TO_END_PRIVACY_CLAIM is not available."
    )


if GENERATOR_PRIVATE is not REQUIRED_PRIVACY_BOUNDARY[
    "generator_private"
]:

    raise RuntimeError(
        "Generator privacy boundary mismatch."
    )

if STATISTICAL_GUIDANCE_PRIVATE is not REQUIRED_PRIVACY_BOUNDARY[
    "statistical_guidance_private"
]:

    raise RuntimeError(
        "Statistical-guidance privacy boundary mismatch."
    )

if PREPROCESSING_PRIVATE is not REQUIRED_PRIVACY_BOUNDARY[
    "preprocessing_private"
]:

    raise RuntimeError(
        "Preprocessing privacy boundary mismatch."
    )

if END_TO_END_PRIVACY_CLAIM is not REQUIRED_PRIVACY_BOUNDARY[
    "end_to_end_privacy_claim"
]:

    raise RuntimeError(
        "End-to-end privacy claim must remain False."
    )


print(
    "✓ Privacy boundary validated."
)

print(
    "  Generator private              : "
    f"{GENERATOR_PRIVATE}"
)

print(
    "  Statistical guidance private   : "
    f"{STATISTICAL_GUIDANCE_PRIVATE}"
)

print(
    "  Preprocessing private          : "
    f"{PREPROCESSING_PRIVATE}"
)

print(
    "  End-to-end privacy claim       : "
    f"{END_TO_END_PRIVACY_CLAIM}"
)


# --------------------------------------------------------------------------------------------------
# 15. Final Validation Registry
# --------------------------------------------------------------------------------------------------

SECTION_7_CHECKS = {

    "dataset_registry":
        list(DATASET_IDS)
        ==
        EXPECTED_DATASET_IDS,

    "native_manifest_loaded":
        not NATIVE_MANIFEST_DF.empty,

    "train_manifest_records":
        len(TRAIN_MANIFEST_RECORDS)
        ==
        len(DATASET_IDS),

    "training_data_loaded":
        len(TRAINING_DATA)
        ==
        len(DATASET_IDS),

    "training_rows":
        all(
            len(
                TRAINING_DATA[
                    dataset_id
                ]
            )
            ==
            EXPECTED_TRAIN_ROWS[
                dataset_id
            ]
            for dataset_id in DATASET_IDS
        ),

    "native_training_columns":
        all(
            TRAINING_DATA[
                dataset_id
            ].shape[1]
            ==
            EXPECTED_NATIVE_COLUMNS[
                dataset_id
            ]
            for dataset_id in DATASET_IDS
        ),

    "preprocessing_metadata":
        len(PREPROCESSING_METADATA)
        ==
        len(DATASET_IDS),

    "preprocessing_schema":
        len(PREPROCESSING_COLUMNS)
        ==
        len(DATASET_IDS),

    "generative_schema":
        len(GENERATIVE_COLUMNS)
        ==
        len(DATASET_IDS),

    "fitted_preprocessors":
        len(FITTED_PREPROCESSORS)
        ==
        len(DATASET_IDS),

    "transformed_arrays":
        len(TRAIN_ARRAYS)
        ==
        len(DATASET_IDS),

    "transformed_dimensions":
        all(
            TRAIN_ARRAYS[
                dataset_id
            ].shape[1]
            ==
            EXPECTED_TRANSFORMED_DIMS[
                dataset_id
            ]
            for dataset_id in DATASET_IDS
        ),

    "transformed_row_counts":
        all(
            TRAIN_ARRAYS[
                dataset_id
            ].shape[0]
            ==
            EXPECTED_TRAIN_ROWS[
                dataset_id
            ]
            for dataset_id in DATASET_IDS
        ),

    "transformed_finite":
        all(
            np.isfinite(
                TRAIN_ARRAYS[
                    dataset_id
                ]
            ).all()
            for dataset_id in DATASET_IDS
        ),

    "statistical_guidance":
        all(
            dataset_id
            in
            STATISTICAL_GUIDANCE
            for dataset_id in DATASET_IDS
        ),

    "architecture_dimensions":
        all(
            int(
                SPPGAN_ARCHITECTURE[
                    dataset_id
                ][
                    "transformed_dimension"
                ]
            )
            ==
            EXPECTED_TRANSFORMED_DIMS[
                dataset_id
            ]
            for dataset_id in DATASET_IDS
        ),

    "architecture_generative_dimensions":
        all(
            int(
                SPPGAN_ARCHITECTURE[
                    dataset_id
                ][
                    "generative_dimension"
                ]
            )
            ==
            EXPECTED_GENERATIVE_DIMS[
                dataset_id
            ]
            for dataset_id in DATASET_IDS
        ),

    "architecture_latent_dimension":
        all(
            int(
                SPPGAN_ARCHITECTURE[
                    dataset_id
                ][
                    "latent_dim"
                ]
            )
            ==
            128
            for dataset_id in DATASET_IDS
        ),

    "architecture_generator":
        all(
            int(
                SPPGAN_ARCHITECTURE[
                    dataset_id
                ][
                    "generator_hidden_1"
                ]
            )
            ==
            256
            and
            int(
                SPPGAN_ARCHITECTURE[
                    dataset_id
                ][
                    "generator_hidden_2"
                ]
            )
            ==
            256
            for dataset_id in DATASET_IDS
        ),

    "architecture_critic":
        all(
            int(
                SPPGAN_ARCHITECTURE[
                    dataset_id
                ][
                    "critic_hidden_1"
                ]
            )
            ==
            256
            and
            int(
                SPPGAN_ARCHITECTURE[
                    dataset_id
                ][
                    "critic_hidden_2"
                ]
            )
            ==
            256
            for dataset_id in DATASET_IDS
        ),

    "rdp_accountant":
        str(
            ACCOUNTANT
        ).strip().lower()
        ==
        "rdp",

    "poisson_sampling":
        str(
            SAMPLING_MECHANISM
        ).strip().lower()
        ==
        "poisson",

    "flat_clipping":
        str(
            CLIPPING_MECHANISM
        ).strip().lower()
        ==
        "flat",

    "mean_loss_reduction":
        str(
            LOSS_REDUCTION
        ).strip().lower()
        ==
        "mean",

    "generator_not_private":
        GENERATOR_PRIVATE is False,

    "statistical_guidance_not_private":
        STATISTICAL_GUIDANCE_PRIVATE is False,

    "preprocessing_not_private":
        PREPROCESSING_PRIVATE is False,

    "end_to_end_privacy_not_claimed":
        END_TO_END_PRIVACY_CLAIM is False,
}


FAILED_SECTION_7_CHECKS = [
    name
    for name, passed
    in SECTION_7_CHECKS.items()
    if not passed
]


# --------------------------------------------------------------------------------------------------
# 16. Validation Output
# --------------------------------------------------------------------------------------------------

print()
print("-" * 100)
print("SECTION 7 VALIDATION")
print("-" * 100)

for check_name, passed in SECTION_7_CHECKS.items():

    print(
        f"{'✓' if passed else '✗'} "
        f"{check_name}"
    )


if FAILED_SECTION_7_CHECKS:

    raise RuntimeError(
        "Section 7 validation failed:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in FAILED_SECTION_7_CHECKS
        )
    )


# --------------------------------------------------------------------------------------------------
# 17. Completion
# --------------------------------------------------------------------------------------------------

print()
print("=" * 100)
print("SECTION 7 — VALIDATE ALL INPUTS COMPLETE")
print("=" * 100)

print(
    "✓ Authoritative Notebook 02 TRAIN manifest validated."
)

print(
    "✓ Native TRAIN datasets loaded from persisted artifacts."
)

print(
    "✓ TRAIN row counts and schemas validated."
)

print(
    "✓ TRAIN SHA-256 integrity validated."
)

print(
    "✓ Notebook 02 preprocessing metadata validated."
)

print(
    "✓ Authoritative preprocessing column order validated."
)

print(
    "✓ Authoritative generative column order validated."
)

print(
    "✓ Target retained in generative schema and excluded from preprocessing."
)

print(
    "✓ Provenance excluded from modeling."
)

print(
    "✓ Frozen Notebook 02 preprocessors loaded."
)

print(
    "✓ TRAIN_ARRAYS constructed using frozen Notebook 02 preprocessors."
)

print(
    "✓ Transformed dimensions cross-validated against Notebook 08."
)

print(
    "✓ Statistical guidance validated."
)

print(
    "✓ Notebook 08 architecture dimensions validated."
)

print(
    "✓ DP parameters validated."
)

print(
    "✓ Privacy boundary validated."
)

print(
    f"✓ Section 7 checks : "
    f"{len(SECTION_7_CHECKS)}/"
    f"{len(SECTION_7_CHECKS)} PASS"
)

print()
print(
    "STATUS: PASS — SECTION 7 READY FOR FREEZE"
)

print("=" * 100)

7. VALIDATE ALL INPUTS
✓ Dataset registry validated.
✓ Notebook 02 root     : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02
✓ Native manifest      : /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/native/native_dataset_manifest.csv
✓ Preprocessor manifest: /content/drive/MyDrive/SPP_GAN_Research/data/processed/notebook_02/schemas/preprocessor_manifest.csv
✓ Native manifest loaded : 9 rows
✓ Native manifest schema validated.
✓ Exactly one validated TRAIN manifest record resolved for each dataset.
✓ adult_income       TRAIN loaded : (34189, 16)
✓ bank_marketing     TRAIN loaded : (31647, 18)
✓ diabetes_130us     TRAIN loaded : (71236, 49)
✓ Authoritative Notebook 02 native TRAIN data loaded.
✓ TRAIN-only data policy preserved.
✓ adult_income       Notebook 02 preprocessing metadata validated.
✓ bank_marketing     Notebook 02 preprocessing metadata validated.
✓ diabetes_130us     Notebook 02 preprocessing metadata validated.
✓ Notebook 02 preprocess

In [9]:
# ==================================================================================================
# 8. CONFIGURE TRAINING
# ==================================================================================================

print("=" * 100)
print("8. CONFIGURE TRAINING")
print("=" * 100)

import json
from pathlib import Path


# --------------------------------------------------------------------------------------------------
# 1. Resolve Canonical Notebook 12 Paths
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/SPP_GAN_Research"
)

NB12_ROOT = (
    PROJECT_ROOT
    / "results"
    / "notebooks"
    / "notebook_12"
)

NB12_CONFIGURATION_DIR = (
    NB12_ROOT
    / "configuration"
)

NB12_CONFIGURATION_DIR.mkdir(
    parents=True,
    exist_ok=True
)

if not NB12_ROOT.exists():
    raise RuntimeError(
        f"Notebook 12 root does not exist:\n{NB12_ROOT}"
    )

if not NB12_CONFIGURATION_DIR.exists():
    raise RuntimeError(
        f"Notebook 12 configuration directory does not exist:\n"
        f"{NB12_CONFIGURATION_DIR}"
    )

print(
    f"✓ Notebook 12 configuration directory resolved:\n"
    f"  {NB12_CONFIGURATION_DIR}"
)


# --------------------------------------------------------------------------------------------------
# 2. Resolve Frozen Architecture Configuration
# --------------------------------------------------------------------------------------------------

if "SPPGAN_ARCHITECTURE" not in globals():
    raise RuntimeError(
        "SPPGAN_ARCHITECTURE is not available.\n"
        "Run Notebook 12 Section 5 before Section 8."
    )

EXPECTED_LATENT_DIM = 128

EXPECTED_GENERATOR_HIDDEN_DIM_1 = 256
EXPECTED_GENERATOR_HIDDEN_DIM_2 = 256

EXPECTED_CRITIC_HIDDEN_DIM_1 = 256
EXPECTED_CRITIC_HIDDEN_DIM_2 = 256


for dataset_id in DATASET_IDS:

    architecture = SPPGAN_ARCHITECTURE[
        dataset_id
    ]

    if int(
        architecture["latent_dim"]
    ) != EXPECTED_LATENT_DIM:
        raise RuntimeError(
            f"{dataset_id}: latent dimension does not match "
            "the frozen Notebook 08 architecture."
        )

    if int(
        architecture["generator_hidden_1"]
    ) != EXPECTED_GENERATOR_HIDDEN_DIM_1:
        raise RuntimeError(
            f"{dataset_id}: generator hidden layer 1 mismatch."
        )

    if int(
        architecture["generator_hidden_2"]
    ) != EXPECTED_GENERATOR_HIDDEN_DIM_2:
        raise RuntimeError(
            f"{dataset_id}: generator hidden layer 2 mismatch."
        )

    if int(
        architecture["critic_hidden_1"]
    ) != EXPECTED_CRITIC_HIDDEN_DIM_1:
        raise RuntimeError(
            f"{dataset_id}: critic hidden layer 1 mismatch."
        )

    if int(
        architecture["critic_hidden_2"]
    ) != EXPECTED_CRITIC_HIDDEN_DIM_2:
        raise RuntimeError(
            f"{dataset_id}: critic hidden layer 2 mismatch."
        )


LATENT_DIM = EXPECTED_LATENT_DIM

GENERATOR_HIDDEN_DIM_1 = EXPECTED_GENERATOR_HIDDEN_DIM_1
GENERATOR_HIDDEN_DIM_2 = EXPECTED_GENERATOR_HIDDEN_DIM_2

CRITIC_HIDDEN_DIM_1 = EXPECTED_CRITIC_HIDDEN_DIM_1
CRITIC_HIDDEN_DIM_2 = EXPECTED_CRITIC_HIDDEN_DIM_2

print(
    "✓ Frozen Notebook 08 architecture dimensions resolved."
)


# --------------------------------------------------------------------------------------------------
# 3. Resolve Frozen Optimization Configuration
# --------------------------------------------------------------------------------------------------

GENERATOR_LR = 2e-4
CRITIC_LR = 2e-4
WEIGHT_DECAY = 1e-6


if GENERATOR_LR <= 0:
    raise ValueError(
        "Generator learning rate must be positive."
    )

if CRITIC_LR <= 0:
    raise ValueError(
        "Critic learning rate must be positive."
    )

if WEIGHT_DECAY < 0:
    raise ValueError(
        "Weight decay cannot be negative."
    )

print(
    "✓ Optimization configuration validated."
)


# --------------------------------------------------------------------------------------------------
# 4. Resolve Statistical Guidance Configuration
# --------------------------------------------------------------------------------------------------

if "LAMBDA_STAT" not in globals():
    LAMBDA_STAT = 1.0

if LAMBDA_STAT < 0:
    raise ValueError(
        "LAMBDA_STAT must be non-negative."
    )

if "STATISTICAL_GUIDANCE" not in globals():
    raise RuntimeError(
        "STATISTICAL_GUIDANCE is not available.\n"
        "Run Notebook 12 Section 4 before Section 8."
    )


EXPECTED_STATISTICAL_COMPONENTS = [
    "marginal_mmd",
    "moment_mean_std",
    "dependency_pearson_frobenius",
    "categorical_probability",
]

print(
    "✓ Statistical guidance configuration validated."
)

print(
    f"  Lambda statistical : {LAMBDA_STAT}"
)


# --------------------------------------------------------------------------------------------------
# 5. Resolve Frozen DP Configuration
# --------------------------------------------------------------------------------------------------

if "TARGET_EPSILON" not in globals():
    TARGET_EPSILON = 5.0

if "MAX_GRAD_NORM" not in globals():
    MAX_GRAD_NORM = 1.0

if "DP_BATCH_SIZE" not in globals():
    DP_BATCH_SIZE = 128

if "DP_EPOCHS" not in globals():
    DP_EPOCHS = 300

if "ACCOUNTANT" not in globals():
    ACCOUNTANT = "rdp"

if "SAMPLING_MECHANISM" not in globals():
    SAMPLING_MECHANISM = "poisson"

if "CLIPPING_MECHANISM" not in globals():
    CLIPPING_MECHANISM = "flat"

if "LOSS_REDUCTION" not in globals():
    LOSS_REDUCTION = "mean"


if TARGET_EPSILON <= 0:
    raise ValueError(
        "Target epsilon must be positive."
    )

if MAX_GRAD_NORM <= 0:
    raise ValueError(
        "Maximum gradient norm must be positive."
    )

if DP_BATCH_SIZE <= 0:
    raise ValueError(
        "DP batch size must be positive."
    )

if DP_EPOCHS <= 0:
    raise ValueError(
        "DP epochs must be positive."
    )

if str(ACCOUNTANT).lower() != "rdp":
    raise ValueError(
        "Notebook 12 requires RDP accounting."
    )

if str(SAMPLING_MECHANISM).lower() != "poisson":
    raise ValueError(
        "Notebook 12 requires Poisson sampling."
    )

if str(CLIPPING_MECHANISM).lower() != "flat":
    raise ValueError(
        "Notebook 12 requires flat L2 clipping."
    )

if str(LOSS_REDUCTION).lower() != "mean":
    raise ValueError(
        "Notebook 12 requires mean loss reduction."
    )

print(
    "✓ DP configuration validated."
)


# --------------------------------------------------------------------------------------------------
# 6. Resolve Privacy Boundary
# --------------------------------------------------------------------------------------------------

if "GENERATOR_PRIVATE" not in globals():
    GENERATOR_PRIVATE = False

if "STATISTICAL_GUIDANCE_PRIVATE" not in globals():
    STATISTICAL_GUIDANCE_PRIVATE = False

if "PREPROCESSING_PRIVATE" not in globals():
    PREPROCESSING_PRIVATE = False

if "END_TO_END_PRIVACY_CLAIM" not in globals():
    END_TO_END_PRIVACY_CLAIM = False


if GENERATOR_PRIVATE is not False:
    raise RuntimeError(
        "Generator privacy boundary must remain False."
    )

if STATISTICAL_GUIDANCE_PRIVATE is not False:
    raise RuntimeError(
        "Statistical-guidance privacy boundary must remain False."
    )

if PREPROCESSING_PRIVATE is not False:
    raise RuntimeError(
        "Preprocessing privacy boundary must remain False."
    )

if END_TO_END_PRIVACY_CLAIM is not False:
    raise RuntimeError(
        "End-to-end privacy claim must remain False."
    )

print(
    "✓ Privacy boundary validated."
)


# --------------------------------------------------------------------------------------------------
# 7. Resolve Reproducibility Configuration
# --------------------------------------------------------------------------------------------------

if "MASTER_SEED" not in globals():
    MASTER_SEED = 2025

MASTER_SEED = int(
    MASTER_SEED
)


if "DEVICE" not in globals():

    try:

        import torch

        DEVICE = torch.device(
            "cuda"
            if torch.cuda.is_available()
            else "cpu"
        )

    except ImportError:

        DEVICE = "cpu"


DEVICE = str(
    DEVICE
)

print(
    "✓ Reproducibility configuration resolved."
)

print(
    f"  Master seed : {MASTER_SEED}"
)

print(
    f"  Device      : {DEVICE}"
)


# --------------------------------------------------------------------------------------------------
# 8. Construct Canonical Training Configuration
# --------------------------------------------------------------------------------------------------

TRAINING_CONFIG = {

    "framework":
        "SPP-GAN",

    "latent_dim":
        LATENT_DIM,

    "generator":
        {
            "hidden_dim_1":
                GENERATOR_HIDDEN_DIM_1,

            "hidden_dim_2":
                GENERATOR_HIDDEN_DIM_2,

            "learning_rate":
                GENERATOR_LR,

            "weight_decay":
                WEIGHT_DECAY,
        },

    "critic":
        {
            "hidden_dim_1":
                CRITIC_HIDDEN_DIM_1,

            "hidden_dim_2":
                CRITIC_HIDDEN_DIM_2,

            "learning_rate":
                CRITIC_LR,

            "weight_decay":
                WEIGHT_DECAY,
        },

    "statistical_guidance":
        {
            "enabled":
                True,

            "lambda_stat":
                LAMBDA_STAT,

            "objective":
                "L_G = L_adv + lambda_stat * L_stat",

            "components":
                EXPECTED_STATISTICAL_COMPONENTS,
        },

    "privacy":
        {
            "enabled":
                True,

            "target_epsilon":
                TARGET_EPSILON,

            "max_grad_norm":
                MAX_GRAD_NORM,

            "batch_size_nominal":
                DP_BATCH_SIZE,

            "epochs":
                DP_EPOCHS,

            "accountant":
                ACCOUNTANT,

            "sampling":
                SAMPLING_MECHANISM,

            "clipping":
                CLIPPING_MECHANISM,

            "loss_reduction":
                LOSS_REDUCTION,

            "generator_private":
                GENERATOR_PRIVATE,

            "statistical_guidance_private":
                STATISTICAL_GUIDANCE_PRIVATE,

            "preprocessing_private":
                PREPROCESSING_PRIVATE,

            "end_to_end_privacy_claim":
                END_TO_END_PRIVACY_CLAIM,
        },

    "data_policy":
        {
            "training":
                "Notebook 02 TRAIN",

            "validation":
                "monitoring only",

            "test":
                "isolated",

            "fit_policy":
                "train_only",
        },

    "device":
        DEVICE,

    "master_seed":
        MASTER_SEED,
}


# --------------------------------------------------------------------------------------------------
# 9. Configuration Integrity Checks
# --------------------------------------------------------------------------------------------------

if TRAINING_CONFIG["latent_dim"] != 128:
    raise RuntimeError(
        "Training configuration latent dimension does not equal "
        "the frozen architecture value of 128."
    )

if TRAINING_CONFIG["generator"]["hidden_dim_1"] != 256:
    raise RuntimeError(
        "Generator hidden_dim_1 does not match frozen architecture."
    )

if TRAINING_CONFIG["generator"]["hidden_dim_2"] != 256:
    raise RuntimeError(
        "Generator hidden_dim_2 does not match frozen architecture."
    )

if TRAINING_CONFIG["critic"]["hidden_dim_1"] != 256:
    raise RuntimeError(
        "Critic hidden_dim_1 does not match frozen architecture."
    )

if TRAINING_CONFIG["critic"]["hidden_dim_2"] != 256:
    raise RuntimeError(
        "Critic hidden_dim_2 does not match frozen architecture."
    )

if TRAINING_CONFIG["statistical_guidance"]["lambda_stat"] != 1.0:
    raise RuntimeError(
        "lambda_stat does not match the validated Notebook 09 configuration."
    )

if TRAINING_CONFIG["privacy"]["target_epsilon"] != 5.0:
    raise RuntimeError(
        "Target epsilon does not match the validated DP configuration."
    )

if TRAINING_CONFIG["privacy"]["max_grad_norm"] != 1.0:
    raise RuntimeError(
        "Maximum gradient norm does not match the validated DP configuration."
    )

if TRAINING_CONFIG["privacy"]["batch_size_nominal"] != 128:
    raise RuntimeError(
        "DP batch size does not match the validated DP configuration."
    )

if TRAINING_CONFIG["privacy"]["epochs"] != 300:
    raise RuntimeError(
        "DP epochs do not match the validated DP configuration."
    )


# --------------------------------------------------------------------------------------------------
# 10. Persist Configuration
# --------------------------------------------------------------------------------------------------

TRAINING_CONFIG_PATH = (
    NB12_CONFIGURATION_DIR
    /
    "sppgan_training_configuration.json"
)

with open(
    TRAINING_CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        TRAINING_CONFIG,
        f,
        indent=2,
        sort_keys=True,
    )


# --------------------------------------------------------------------------------------------------
# 11. Reload and Verify Persisted Configuration
# --------------------------------------------------------------------------------------------------

with open(
    TRAINING_CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:

    TRAINING_CONFIG_RELOADED = json.load(
        f
    )


if TRAINING_CONFIG_RELOADED != TRAINING_CONFIG:
    raise RuntimeError(
        "Persisted training configuration does not match "
        "the in-memory configuration."
    )


print(
    f"✓ Training configuration saved:\n"
    f"  {TRAINING_CONFIG_PATH}"
)


# --------------------------------------------------------------------------------------------------
# 12. Configuration Summary
# --------------------------------------------------------------------------------------------------

print()
print("Configuration:")

print(
    f"  Latent dimension       : {LATENT_DIM}"
)

print(
    f"  Generator hidden       : "
    f"{GENERATOR_HIDDEN_DIM_1}/{GENERATOR_HIDDEN_DIM_2}"
)

print(
    f"  Critic hidden          : "
    f"{CRITIC_HIDDEN_DIM_1}/{CRITIC_HIDDEN_DIM_2}"
)

print(
    f"  Generator LR           : {GENERATOR_LR}"
)

print(
    f"  Critic LR              : {CRITIC_LR}"
)

print(
    f"  Weight decay           : {WEIGHT_DECAY}"
)

print(
    f"  Lambda statistical     : {LAMBDA_STAT}"
)

print(
    f"  DP batch               : {DP_BATCH_SIZE}"
)

print(
    f"  Epochs                 : {DP_EPOCHS}"
)

print(
    f"  Target epsilon         : {TARGET_EPSILON}"
)

print(
    f"  Max gradient norm      : {MAX_GRAD_NORM}"
)

print(
    f"  Accountant             : {ACCOUNTANT}"
)

print(
    f"  Sampling               : {SAMPLING_MECHANISM}"
)

print(
    f"  Device                 : {DEVICE}"
)

print(
    f"  Master seed            : {MASTER_SEED}"
)


# --------------------------------------------------------------------------------------------------
# 13. Final Section 8 Checks
# --------------------------------------------------------------------------------------------------

SECTION_8_CHECKS = {

    "training_config_created":
        isinstance(
            TRAINING_CONFIG,
            dict,
        ),

    "latent_dim":
        LATENT_DIM == 128,

    "generator_hidden_1":
        GENERATOR_HIDDEN_DIM_1 == 256,

    "generator_hidden_2":
        GENERATOR_HIDDEN_DIM_2 == 256,

    "critic_hidden_1":
        CRITIC_HIDDEN_DIM_1 == 256,

    "critic_hidden_2":
        CRITIC_HIDDEN_DIM_2 == 256,

    "generator_learning_rate":
        GENERATOR_LR > 0,

    "critic_learning_rate":
        CRITIC_LR > 0,

    "weight_decay":
        WEIGHT_DECAY >= 0,

    "lambda_stat":
        LAMBDA_STAT == 1.0,

    "target_epsilon":
        TARGET_EPSILON == 5.0,

    "max_grad_norm":
        MAX_GRAD_NORM == 1.0,

    "dp_batch_size":
        DP_BATCH_SIZE == 128,

    "dp_epochs":
        DP_EPOCHS == 300,

    "rdp_accountant":
        str(ACCOUNTANT).lower() == "rdp",

    "poisson_sampling":
        str(SAMPLING_MECHANISM).lower() == "poisson",

    "flat_clipping":
        str(CLIPPING_MECHANISM).lower() == "flat",

    "mean_loss_reduction":
        str(LOSS_REDUCTION).lower() == "mean",

    "generator_not_private":
        GENERATOR_PRIVATE is False,

    "statistical_guidance_not_private":
        STATISTICAL_GUIDANCE_PRIVATE is False,

    "preprocessing_not_private":
        PREPROCESSING_PRIVATE is False,

    "end_to_end_privacy_not_claimed":
        END_TO_END_PRIVACY_CLAIM is False,

    "configuration_directory":
        NB12_CONFIGURATION_DIR.exists(),

    "configuration_persisted":
        TRAINING_CONFIG_PATH.exists(),

    "configuration_reload_integrity":
        TRAINING_CONFIG_RELOADED == TRAINING_CONFIG,
}


FAILED_SECTION_8_CHECKS = [
    name
    for name, passed
    in SECTION_8_CHECKS.items()
    if not passed
]


print()
print("-" * 100)
print("SECTION 8 VALIDATION")
print("-" * 100)

for check_name, passed in SECTION_8_CHECKS.items():

    print(
        f"{'✓' if passed else '✗'} "
        f"{check_name}"
    )


if FAILED_SECTION_8_CHECKS:

    raise RuntimeError(
        "Section 8 validation failed:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in FAILED_SECTION_8_CHECKS
        )
    )


print()
print("=" * 100)
print("SECTION 8 — CONFIGURE TRAINING COMPLETE")
print("=" * 100)

print(
    f"✓ Section 8 checks : "
    f"{len(SECTION_8_CHECKS)}/"
    f"{len(SECTION_8_CHECKS)} PASS"
)

print(
    "STATUS: PASS — SECTION 8 READY FOR FREEZE"
)

print("=" * 100)

8. CONFIGURE TRAINING
✓ Notebook 12 configuration directory resolved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_12/configuration
✓ Frozen Notebook 08 architecture dimensions resolved.
✓ Optimization configuration validated.
✓ Statistical guidance configuration validated.
  Lambda statistical : 1.0
✓ DP configuration validated.
✓ Privacy boundary validated.
✓ Reproducibility configuration resolved.
  Master seed : 2025
  Device      : cpu
✓ Training configuration saved:
  /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_12/configuration/sppgan_training_configuration.json

Configuration:
  Latent dimension       : 128
  Generator hidden       : 256/256
  Critic hidden          : 256/256
  Generator LR           : 0.0002
  Critic LR              : 0.0002
  Weight decay           : 1e-06
  Lambda statistical     : 1.0
  DP batch               : 128
  Epochs                 : 300
  Target epsilon         : 5.0
  Max gradient norm      : 1.0
  Acc

In [10]:
# ==================================================================================================
# 9. SET EXPERIMENT SEEDS
# ==================================================================================================

print("=" * 100)
print("9. SET EXPERIMENT SEEDS")
print("=" * 100)

import random
import numpy as np
import torch


# --------------------------------------------------------------------------------------------------
# 1. Validate Required Configuration
# --------------------------------------------------------------------------------------------------

if "MASTER_SEED" not in globals():
    raise RuntimeError(
        "MASTER_SEED is not available. "
        "Run Notebook 12 Section 8 before Section 9."
    )

MASTER_SEED = int(MASTER_SEED)


# --------------------------------------------------------------------------------------------------
# 2. Frozen Repetition Seed Registry
# --------------------------------------------------------------------------------------------------

REPETITION_SEEDS = {
    1: 3026,
    2: 3027,
    3: 3028,
    4: 3029,
    5: 3030,
}


EXPECTED_REPETITION_IDS = {
    1,
    2,
    3,
    4,
    5,
}

if set(REPETITION_SEEDS.keys()) != EXPECTED_REPETITION_IDS:
    raise RuntimeError(
        "Repetition seed registry does not match the frozen "
        "five-repetition experimental design."
    )


# --------------------------------------------------------------------------------------------------
# 3. Select Current Repetition
# --------------------------------------------------------------------------------------------------

# Notebook 12 first execution uses repetition 1.
# Later repetitions can be launched using the same deterministic registry.

REPETITION_ID = 1

if REPETITION_ID not in REPETITION_SEEDS:
    raise RuntimeError(
        f"Invalid repetition ID: {REPETITION_ID}"
    )

EXPERIMENT_SEED = int(
    REPETITION_SEEDS[REPETITION_ID]
)


# --------------------------------------------------------------------------------------------------
# 4. Set Python and NumPy Random Seeds
# --------------------------------------------------------------------------------------------------

random.seed(
    EXPERIMENT_SEED
)

np.random.seed(
    EXPERIMENT_SEED
)


# --------------------------------------------------------------------------------------------------
# 5. Set PyTorch Random Seeds
# --------------------------------------------------------------------------------------------------

torch.manual_seed(
    EXPERIMENT_SEED
)

if torch.cuda.is_available():

    torch.cuda.manual_seed(
        EXPERIMENT_SEED
    )

    torch.cuda.manual_seed_all(
        EXPERIMENT_SEED
    )


# --------------------------------------------------------------------------------------------------
# 6. Configure Deterministic PyTorch / cuDNN Behaviour
# --------------------------------------------------------------------------------------------------

torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True


try:

    torch.use_deterministic_algorithms(
        True
    )

    DETERMINISTIC_ALGORITHMS = True

except Exception:

    DETERMINISTIC_ALGORITHMS = False


# --------------------------------------------------------------------------------------------------
# 7. Reproducibility Integrity Checks
# --------------------------------------------------------------------------------------------------

if EXPERIMENT_SEED != REPETITION_SEEDS[REPETITION_ID]:
    raise RuntimeError(
        "Experiment seed does not match the frozen repetition seed registry."
    )

if REPETITION_ID != 1:
    raise RuntimeError(
        "Notebook 12 initial execution must use repetition 1."
    )

if not isinstance(
    EXPERIMENT_SEED,
    int
):
    raise RuntimeError(
        "Experiment seed must be an integer."
    )

if not isinstance(
    MASTER_SEED,
    int
):
    raise RuntimeError(
        "Master seed must be an integer."
    )


# --------------------------------------------------------------------------------------------------
# 8. Final Section 9 Checks
# --------------------------------------------------------------------------------------------------

SECTION_9_CHECKS = {

    "master_seed_available":
        isinstance(
            MASTER_SEED,
            int
        ),

    "master_seed":
        MASTER_SEED == 2025,

    "repetition_registry":
        set(REPETITION_SEEDS.keys()) == EXPECTED_REPETITION_IDS,

    "repetition_count":
        len(REPETITION_SEEDS) == 5,

    "repetition_id":
        REPETITION_ID == 1,

    "experiment_seed":
        EXPERIMENT_SEED == 3026,

    "python_seed_configured":
        True,

    "numpy_seed_configured":
        True,

    "pytorch_seed_configured":
        True,

    "cudnn_benchmark_disabled":
        torch.backends.cudnn.benchmark is False,

    "cudnn_deterministic":
        torch.backends.cudnn.deterministic is True,

    "deterministic_algorithms_requested":
        DETERMINISTIC_ALGORITHMS is True,
}


FAILED_SECTION_9_CHECKS = [
    name
    for name, passed
    in SECTION_9_CHECKS.items()
    if not passed
]


# --------------------------------------------------------------------------------------------------
# 9. Output
# --------------------------------------------------------------------------------------------------

print(
    f"✓ Repetition ID : {REPETITION_ID}"
)

print(
    f"✓ Experiment seed: {EXPERIMENT_SEED}"
)

print(
    f"✓ Master seed    : {MASTER_SEED}"
)

print(
    f"✓ Deterministic  : "
    f"{DETERMINISTIC_ALGORITHMS}"
)


print()
print("-" * 100)
print("SECTION 9 VALIDATION")
print("-" * 100)

for check_name, passed in SECTION_9_CHECKS.items():

    print(
        f"{'✓' if passed else '✗'} "
        f"{check_name}"
    )


if FAILED_SECTION_9_CHECKS:

    raise RuntimeError(
        "Section 9 validation failed:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in FAILED_SECTION_9_CHECKS
        )
    )


print()
print("=" * 100)
print("SECTION 9 — SET EXPERIMENT SEEDS COMPLETE")
print("=" * 100)

print(
    f"✓ Section 9 checks : "
    f"{len(SECTION_9_CHECKS)}/"
    f"{len(SECTION_9_CHECKS)} PASS"
)

print(
    "STATUS: PASS — SECTION 9 READY FOR FREEZE"
)

print("=" * 100)

9. SET EXPERIMENT SEEDS
✓ Repetition ID : 1
✓ Experiment seed: 3026
✓ Master seed    : 2025
✓ Deterministic  : True

----------------------------------------------------------------------------------------------------
SECTION 9 VALIDATION
----------------------------------------------------------------------------------------------------
✓ master_seed_available
✓ master_seed
✓ repetition_registry
✓ repetition_count
✓ repetition_id
✓ experiment_seed
✓ python_seed_configured
✓ numpy_seed_configured
✓ pytorch_seed_configured
✓ cudnn_benchmark_disabled
✓ cudnn_deterministic
✓ deterministic_algorithms_requested

SECTION 9 — SET EXPERIMENT SEEDS COMPLETE
✓ Section 9 checks : 12/12 PASS
STATUS: PASS — SECTION 9 READY FOR FREEZE


In [11]:
# ==================================================================================================
# 10. INITIALIZE SPP-GAN
# ==================================================================================================

print("=" * 100)
print("10. INITIALIZE SPP-GAN")
print("=" * 100)

import torch
import torch.nn as nn


# --------------------------------------------------------------------------------------------------
# 1. Validate Required Inputs
# --------------------------------------------------------------------------------------------------

REQUIRED_SECTION_10_VARIABLES = [
    "DATASET_IDS",
    "TRAIN_ARRAYS",
    "LATENT_DIM",
    "GENERATOR_HIDDEN_DIM_1",
    "GENERATOR_HIDDEN_DIM_2",
    "CRITIC_HIDDEN_DIM_1",
    "CRITIC_HIDDEN_DIM_2",
    "DEVICE",
    "SPPGAN_ARCHITECTURE",
]

MISSING_SECTION_10_VARIABLES = [
    name
    for name in REQUIRED_SECTION_10_VARIABLES
    if name not in globals()
]

if MISSING_SECTION_10_VARIABLES:

    raise RuntimeError(
        "Section 10 is missing required variables:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in MISSING_SECTION_10_VARIABLES
        )
        +
        "\nRun Sections 3, 5, and 8 before Section 10."
    )


# --------------------------------------------------------------------------------------------------
# 2. Define SPP-GAN Generator
# --------------------------------------------------------------------------------------------------

class SPPGANGenerator(nn.Module):

    def __init__(
        self,
        latent_dim,
        transformed_dim,
        hidden_dim_1=256,
        hidden_dim_2=256,
    ):

        super().__init__()

        self.latent_dim = int(
            latent_dim
        )

        self.transformed_dim = int(
            transformed_dim
        )

        self.network = nn.Sequential(

            nn.Linear(
                self.latent_dim,
                int(hidden_dim_1)
            ),

            nn.ReLU(),

            nn.Linear(
                int(hidden_dim_1),
                int(hidden_dim_2)
            ),

            nn.ReLU(),

            nn.Linear(
                int(hidden_dim_2),
                self.transformed_dim
            ),
        )

    def forward(
        self,
        z
    ):

        return self.network(
            z
        )


# --------------------------------------------------------------------------------------------------
# 3. Define SPP-GAN Critic
# --------------------------------------------------------------------------------------------------

class SPPGANCritic(nn.Module):

    def __init__(
        self,
        transformed_dim,
        hidden_dim_1=256,
        hidden_dim_2=256,
    ):

        super().__init__()

        self.transformed_dim = int(
            transformed_dim
        )

        self.network = nn.Sequential(

            nn.Linear(
                self.transformed_dim,
                int(hidden_dim_1)
            ),

            nn.LeakyReLU(
                0.2
            ),

            nn.Linear(
                int(hidden_dim_1),
                int(hidden_dim_2)
            ),

            nn.LeakyReLU(
                0.2
            ),

            nn.Linear(
                int(hidden_dim_2),
                1
            ),
        )

    def forward(
        self,
        x
    ):

        return self.network(
            x
        )


# --------------------------------------------------------------------------------------------------
# 4. Initialize Dataset-Specific Models
# --------------------------------------------------------------------------------------------------

SPPGAN_MODELS = {}

MODEL_PARAMETER_COUNTS = {}

MODEL_DIMENSION_REGISTRY = {}

for dataset_id in DATASET_IDS:

    if dataset_id not in TRAIN_ARRAYS:
        raise RuntimeError(
            f"{dataset_id}: TRAIN_ARRAYS entry is missing."
        )

    train_array = TRAIN_ARRAYS[
        dataset_id
    ]

    if not isinstance(
        train_array,
        np.ndarray
    ):
        train_array = np.asarray(
            train_array
        )

    if train_array.ndim != 2:
        raise RuntimeError(
            f"{dataset_id}: TRAIN_ARRAYS must be a 2D array."
        )

    transformed_dim = int(
        train_array.shape[1]
    )

    if transformed_dim <= 0:
        raise RuntimeError(
            f"{dataset_id}: transformed dimension must be positive."
        )


    # ----------------------------------------------------------------------------------------------
    # Validate Frozen Notebook 08 Dimension
    # ----------------------------------------------------------------------------------------------

    architecture = SPPGAN_ARCHITECTURE[
        dataset_id
    ]

    expected_transformed_dim = int(
        architecture["transformed_dimension"]
    )

    if transformed_dim != expected_transformed_dim:

        raise RuntimeError(
            f"{dataset_id}: transformed dimension mismatch.\n"
            f"  TRAIN_ARRAYS : {transformed_dim}\n"
            f"  Notebook 08  : {expected_transformed_dim}"
        )


    # ----------------------------------------------------------------------------------------------
    # Initialize Generator
    # ----------------------------------------------------------------------------------------------

    generator = SPPGANGenerator(
        latent_dim=LATENT_DIM,
        transformed_dim=transformed_dim,
        hidden_dim_1=GENERATOR_HIDDEN_DIM_1,
        hidden_dim_2=GENERATOR_HIDDEN_DIM_2,
    ).to(
        DEVICE
    )


    # ----------------------------------------------------------------------------------------------
    # Initialize Critic
    # ----------------------------------------------------------------------------------------------

    critic = SPPGANCritic(
        transformed_dim=transformed_dim,
        hidden_dim_1=CRITIC_HIDDEN_DIM_1,
        hidden_dim_2=CRITIC_HIDDEN_DIM_2,
    ).to(
        DEVICE
    )


    # ----------------------------------------------------------------------------------------------
    # Parameter Counts
    # ----------------------------------------------------------------------------------------------

    generator_parameters = sum(
        parameter.numel()
        for parameter in generator.parameters()
        if parameter.requires_grad
    )

    critic_parameters = sum(
        parameter.numel()
        for parameter in critic.parameters()
        if parameter.requires_grad
    )

    total_parameters = (
        generator_parameters
        +
        critic_parameters
    )


    # ----------------------------------------------------------------------------------------------
    # Validate Frozen Parameter Counts
    # ----------------------------------------------------------------------------------------------

    expected_generator_parameters = int(
        architecture["generator_trainable_parameters"]
    )

    expected_critic_parameters = int(
        architecture["critic_trainable_parameters"]
    )

    expected_total_parameters = int(
        architecture["total_trainable_parameters"]
    )

    if generator_parameters != expected_generator_parameters:

        raise RuntimeError(
            f"{dataset_id}: generator parameter-count mismatch.\n"
            f"  Observed : {generator_parameters}\n"
            f"  Frozen   : {expected_generator_parameters}"
        )

    if critic_parameters != expected_critic_parameters:

        raise RuntimeError(
            f"{dataset_id}: critic parameter-count mismatch.\n"
            f"  Observed : {critic_parameters}\n"
            f"  Frozen   : {expected_critic_parameters}"
        )

    if total_parameters != expected_total_parameters:

        raise RuntimeError(
            f"{dataset_id}: total parameter-count mismatch.\n"
            f"  Observed : {total_parameters}\n"
            f"  Frozen   : {expected_total_parameters}"
        )


    # ----------------------------------------------------------------------------------------------
    # Store Models
    # ----------------------------------------------------------------------------------------------

    SPPGAN_MODELS[
        dataset_id
    ] = {

        "generator":
            generator,

        "critic":
            critic,

        "transformed_dim":
            transformed_dim,

        "generator_parameters":
            generator_parameters,

        "critic_parameters":
            critic_parameters,

        "total_parameters":
            total_parameters,
    }


    MODEL_PARAMETER_COUNTS[
        dataset_id
    ] = {

        "generator":
            generator_parameters,

        "critic":
            critic_parameters,

        "total":
            total_parameters,
    }


    MODEL_DIMENSION_REGISTRY[
        dataset_id
    ] = {

        "latent_dim":
            LATENT_DIM,

        "transformed_dim":
            transformed_dim,

        "generator_hidden_1":
            GENERATOR_HIDDEN_DIM_1,

        "generator_hidden_2":
            GENERATOR_HIDDEN_DIM_2,

        "critic_hidden_1":
            CRITIC_HIDDEN_DIM_1,

        "critic_hidden_2":
            CRITIC_HIDDEN_DIM_2,
    }


    print(
        f"✓ {dataset_id:<20} "
        f"transformed_dim={transformed_dim:<5} "
        f"G_params={generator_parameters:<8} "
        f"C_params={critic_parameters:<8} "
        f"Total={total_parameters}"
    )


# --------------------------------------------------------------------------------------------------
# 5. Model Integrity Checks
# --------------------------------------------------------------------------------------------------

SECTION_10_CHECKS = {

    "generator_class_available":
        issubclass(
            SPPGANGenerator,
            nn.Module
        ),

    "critic_class_available":
        issubclass(
            SPPGANCritic,
            nn.Module
        ),

    "model_registry_created":
        isinstance(
            SPPGAN_MODELS,
            dict
        ),

    "dataset_coverage":
        set(SPPGAN_MODELS.keys())
        ==
        set(DATASET_IDS),

    "parameter_registry_created":
        set(MODEL_PARAMETER_COUNTS.keys())
        ==
        set(DATASET_IDS),

    "dimension_registry_created":
        set(MODEL_DIMENSION_REGISTRY.keys())
        ==
        set(DATASET_IDS),
}


for dataset_id in DATASET_IDS:

    generator = SPPGAN_MODELS[
        dataset_id
    ]["generator"]

    critic = SPPGAN_MODELS[
        dataset_id
    ]["critic"]

    transformed_dim = SPPGAN_MODELS[
        dataset_id
    ]["transformed_dim"]


    SECTION_10_CHECKS[
        f"{dataset_id}_generator_module"
    ] = isinstance(
        generator,
        SPPGANGenerator
    )


    SECTION_10_CHECKS[
        f"{dataset_id}_critic_module"
    ] = isinstance(
        critic,
        SPPGANCritic
    )


    SECTION_10_CHECKS[
        f"{dataset_id}_generator_device"
    ] = next(
        generator.parameters()
    ).device.type == str(
        DEVICE
    ).split(":")[0]


    SECTION_10_CHECKS[
        f"{dataset_id}_critic_device"
    ] = next(
        critic.parameters()
    ).device.type == str(
        DEVICE
    ).split(":")[0]


    SECTION_10_CHECKS[
        f"{dataset_id}_generator_output"
    ] = (
        generator.network[-1].out_features
        ==
        transformed_dim
    )


    SECTION_10_CHECKS[
        f"{dataset_id}_critic_input"
    ] = (
        critic.network[0].in_features
        ==
        transformed_dim
    )


    SECTION_10_CHECKS[
        f"{dataset_id}_critic_output"
    ] = (
        critic.network[-1].out_features
        ==
        1
    )


# --------------------------------------------------------------------------------------------------
# 6. Final Validation
# --------------------------------------------------------------------------------------------------

FAILED_SECTION_10_CHECKS = [
    name
    for name, passed
    in SECTION_10_CHECKS.items()
    if not passed
]


print()
print("-" * 100)
print("SECTION 10 VALIDATION")
print("-" * 100)

for check_name, passed in SECTION_10_CHECKS.items():

    print(
        f"{'✓' if passed else '✗'} "
        f"{check_name}"
    )


if FAILED_SECTION_10_CHECKS:

    raise RuntimeError(
        "Section 10 validation failed:\n"
        +
        "\n".join(
            f"  - {name}"
            for name in FAILED_SECTION_10_CHECKS
        )
    )


print()
print("=" * 100)
print("SECTION 10 — INITIALIZE SPP-GAN COMPLETE")
print("=" * 100)

print(
    f"✓ Section 10 checks : "
    f"{len(SECTION_10_CHECKS)}/"
    f"{len(SECTION_10_CHECKS)} PASS"
)

print(
    "STATUS: PASS — SECTION 10 READY FOR FREEZE"
)

print("=" * 100)

10. INITIALIZE SPP-GAN
✓ adult_income         transformed_dim=105   G_params=125801   C_params=93185    Total=218986
✓ bank_marketing       transformed_dim=51    G_params=111923   C_params=79361    Total=191284
✓ diabetes_130us       transformed_dim=2329  G_params=697369   C_params=662529   Total=1359898

----------------------------------------------------------------------------------------------------
SECTION 10 VALIDATION
----------------------------------------------------------------------------------------------------
✓ generator_class_available
✓ critic_class_available
✓ model_registry_created
✓ dataset_coverage
✓ parameter_registry_created
✓ dimension_registry_created
✓ adult_income_generator_module
✓ adult_income_critic_module
✓ adult_income_generator_device
✓ adult_income_critic_device
✓ adult_income_generator_output
✓ adult_income_critic_input
✓ adult_income_critic_output
✓ bank_marketing_generator_module
✓ bank_marketing_critic_module
✓ bank_marketing_generator_device
✓ ba

In [12]:
# ==================================================================================================
# 11. INITIALIZE PRIVACY MECHANISM
# ==================================================================================================

print("=" * 100)
print("11. INITIALIZE PRIVACY MECHANISM")
print("=" * 100)


# --------------------------------------------------------------------------------------------------
# 1. Imports
# --------------------------------------------------------------------------------------------------

import sys
import json
import hashlib
import importlib.util
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from opacus import PrivacyEngine
import opacus


# --------------------------------------------------------------------------------------------------
# 2. Canonical paths
# --------------------------------------------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/SPP_GAN_Research"
)

NB10_ROOT = (
    PROJECT_ROOT
    / "results"
    / "notebooks"
    / "notebook_10"
)

NB12_ROOT = (
    PROJECT_ROOT
    / "results"
    / "notebooks"
    / "notebook_12"
)

NB12_PRIVACY_ROOT = (
    NB12_ROOT
    / "privacy"
)

NB12_PRIVACY_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# --------------------------------------------------------------------------------------------------
# 3. Expected datasets
# --------------------------------------------------------------------------------------------------

EXPECTED_DATASETS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us",
]


# --------------------------------------------------------------------------------------------------
# 4. Required upstream objects
# --------------------------------------------------------------------------------------------------

REQUIRED_OBJECTS = [
    "TRAIN_ARRAYS",
    "SPPGAN_MODELS",
    "TRAINING_CONFIG",
    "NB10_PRIVACY_METADATA_DF",
]

MISSING_OBJECTS = [
    name
    for name in REQUIRED_OBJECTS
    if name not in globals()
]

if MISSING_OBJECTS:

    raise RuntimeError(
        "Section 11 cannot continue because the following "
        "upstream objects are missing:\n"
        + "\n".join(
            f"  - {name}"
            for name in MISSING_OBJECTS
        )
    )

print("✓ Required Section 10/Notebook 12 objects available.")


# --------------------------------------------------------------------------------------------------
# 5. Validate canonical SPPGAN_MODELS registry
# --------------------------------------------------------------------------------------------------

if not isinstance(
    SPPGAN_MODELS,
    dict
):

    raise TypeError(
        "SPPGAN_MODELS must be a dictionary."
    )

missing_model_datasets = [
    dataset_id
    for dataset_id in EXPECTED_DATASETS
    if dataset_id not in SPPGAN_MODELS
]

if missing_model_datasets:

    raise RuntimeError(
        "SPPGAN_MODELS is missing:\n"
        + "\n".join(
            f"  - {dataset_id}"
            for dataset_id in missing_model_datasets
        )
    )

print("✓ Section 10 SPPGAN_MODELS registry located.")


# --------------------------------------------------------------------------------------------------
# 6. Recover exact Section 10 model references
#
# IMPORTANT:
# No new generators or critics are created.
# --------------------------------------------------------------------------------------------------

GENERATORS = {}
CRITICS = {}

for dataset_id in EXPECTED_DATASETS:

    model_record = SPPGAN_MODELS[
        dataset_id
    ]

    if not isinstance(
        model_record,
        dict
    ):

        raise TypeError(
            f"{dataset_id}: SPPGAN_MODELS entry must be a dictionary."
        )

    if "generator" not in model_record:

        raise RuntimeError(
            f"{dataset_id}: generator missing from SPPGAN_MODELS."
        )

    if "critic" not in model_record:

        raise RuntimeError(
            f"{dataset_id}: critic missing from SPPGAN_MODELS."
        )

    GENERATORS[
        dataset_id
    ] = model_record[
        "generator"
    ]

    CRITICS[
        dataset_id
    ] = model_record[
        "critic"
    ]

print("✓ GENERATORS references recovered from SPPGAN_MODELS.")
print("✓ CRITICS references recovered from SPPGAN_MODELS.")


# --------------------------------------------------------------------------------------------------
# 7. Validate exact Section 10 object identity
# --------------------------------------------------------------------------------------------------

for dataset_id in EXPECTED_DATASETS:

    generator = GENERATORS[
        dataset_id
    ]

    critic = CRITICS[
        dataset_id
    ]

    if not isinstance(
        generator,
        torch.nn.Module
    ):

        raise TypeError(
            f"{dataset_id}: generator is not torch.nn.Module."
        )

    if not isinstance(
        critic,
        torch.nn.Module
    ):

        raise TypeError(
            f"{dataset_id}: critic is not torch.nn.Module."
        )

    if generator is not SPPGAN_MODELS[
        dataset_id
    ][
        "generator"
    ]:

        raise RuntimeError(
            f"{dataset_id}: generator reference mismatch."
        )

    if critic is not SPPGAN_MODELS[
        dataset_id
    ][
        "critic"
    ]:

        raise RuntimeError(
            f"{dataset_id}: critic reference mismatch."
        )

print("✓ Generator references match Section 10.")
print("✓ Critic references match Section 10.")


# --------------------------------------------------------------------------------------------------
# 8. Validate TRAIN_ARRAYS
# --------------------------------------------------------------------------------------------------

if not isinstance(
    TRAIN_ARRAYS,
    dict
):

    raise TypeError(
        "TRAIN_ARRAYS must be a dictionary."
    )

missing_train_arrays = [
    dataset_id
    for dataset_id in EXPECTED_DATASETS
    if dataset_id not in TRAIN_ARRAYS
]

if missing_train_arrays:

    raise RuntimeError(
        "TRAIN_ARRAYS is missing:\n"
        + "\n".join(
            f"  - {dataset_id}"
            for dataset_id in missing_train_arrays
        )
    )

print("✓ TRAIN_ARRAYS available.")


# --------------------------------------------------------------------------------------------------
# 9. Validate Section 8 TRAINING_CONFIG structure
# --------------------------------------------------------------------------------------------------

REQUIRED_TRAINING_CONFIG_KEYS = [
    "framework",
    "latent_dim",
    "generator",
    "critic",
    "statistical_guidance",
    "privacy",
    "data_policy",
    "device",
    "master_seed",
]

missing_training_keys = [
    key
    for key in REQUIRED_TRAINING_CONFIG_KEYS
    if key not in TRAINING_CONFIG
]

if missing_training_keys:

    raise RuntimeError(
        "TRAINING_CONFIG is missing:\n"
        + "\n".join(
            f"  - {key}"
            for key in missing_training_keys
        )
    )


# --------------------------------------------------------------------------------------------------
# 10. Read actual Section 8 privacy configuration
# --------------------------------------------------------------------------------------------------

PRIVACY_CONFIG = TRAINING_CONFIG[
    "privacy"
]

REQUIRED_PRIVACY_KEYS = [
    "enabled",
    "target_epsilon",
    "max_grad_norm",
    "batch_size_nominal",
    "epochs",
    "accountant",
    "sampling",
    "clipping",
    "loss_reduction",
    "generator_private",
    "statistical_guidance_private",
    "preprocessing_private",
    "end_to_end_privacy_claim",
]

missing_privacy_keys = [
    key
    for key in REQUIRED_PRIVACY_KEYS
    if key not in PRIVACY_CONFIG
]

if missing_privacy_keys:

    raise RuntimeError(
        "TRAINING_CONFIG['privacy'] is missing:\n"
        + "\n".join(
            f"  - {key}"
            for key in missing_privacy_keys
        )
    )

print("✓ Section 8 privacy configuration validated.")


# --------------------------------------------------------------------------------------------------
# 11. Frozen methodology parameters
# --------------------------------------------------------------------------------------------------

EXPECTED_TARGET_EPSILON = 5.0
EXPECTED_MAX_GRAD_NORM = 1.0
EXPECTED_BATCH_SIZE = 128
EXPECTED_EPOCHS = 300
EXPECTED_ACCOUNTANT = "rdp"
EXPECTED_SAMPLING = "poisson"
EXPECTED_CLIPPING = "flat"
EXPECTED_LOSS_REDUCTION = "mean"


# --------------------------------------------------------------------------------------------------
# 12. Validate frozen privacy configuration
# --------------------------------------------------------------------------------------------------

if PRIVACY_CONFIG[
    "enabled"
] is not True:

    raise ValueError(
        "Privacy mechanism must be enabled."
    )

if not np.isclose(
    float(
        PRIVACY_CONFIG[
            "target_epsilon"
        ]
    ),
    EXPECTED_TARGET_EPSILON
):

    raise ValueError(
        "target_epsilon does not match the frozen methodology."
    )

if not np.isclose(
    float(
        PRIVACY_CONFIG[
            "max_grad_norm"
        ]
    ),
    EXPECTED_MAX_GRAD_NORM
):

    raise ValueError(
        "max_grad_norm does not match the frozen methodology."
    )

if int(
    PRIVACY_CONFIG[
        "batch_size_nominal"
    ]
) != EXPECTED_BATCH_SIZE:

    raise ValueError(
        "batch_size_nominal does not match the frozen methodology."
    )

if int(
    PRIVACY_CONFIG[
        "epochs"
    ]
) != EXPECTED_EPOCHS:

    raise ValueError(
        "epochs does not match the frozen methodology."
    )

if str(
    PRIVACY_CONFIG[
        "accountant"
    ]
).lower() != EXPECTED_ACCOUNTANT:

    raise ValueError(
        "accountant must be RDP."
    )

if str(
    PRIVACY_CONFIG[
        "sampling"
    ]
).lower() != EXPECTED_SAMPLING:

    raise ValueError(
        "sampling must be Poisson."
    )

if str(
    PRIVACY_CONFIG[
        "clipping"
    ]
).lower() != EXPECTED_CLIPPING:

    raise ValueError(
        "clipping must be flat."
    )

if str(
    PRIVACY_CONFIG[
        "loss_reduction"
    ]
).lower() != EXPECTED_LOSS_REDUCTION:

    raise ValueError(
        "loss_reduction must be mean."
    )

print("✓ Frozen DP parameters validated.")


# --------------------------------------------------------------------------------------------------
# 13. Validate privacy boundary
# --------------------------------------------------------------------------------------------------

EXPECTED_PRIVACY_BOUNDARY = {
    "generator_private": False,
    "statistical_guidance_private": False,
    "preprocessing_private": False,
    "end_to_end_privacy_claim": False,
}

OBSERVED_PRIVACY_BOUNDARY = {
    key: bool(
        PRIVACY_CONFIG[key]
    )
    for key in EXPECTED_PRIVACY_BOUNDARY
}

if (
    OBSERVED_PRIVACY_BOUNDARY
    != EXPECTED_PRIVACY_BOUNDARY
):

    raise ValueError(
        "Privacy boundary mismatch.\n"
        f"Observed : {OBSERVED_PRIVACY_BOUNDARY}\n"
        f"Expected : {EXPECTED_PRIVACY_BOUNDARY}"
    )

print("✓ Privacy boundary validated.")


# --------------------------------------------------------------------------------------------------
# 14. Validate Notebook 10 DP metadata schema
# --------------------------------------------------------------------------------------------------

REQUIRED_DP_METADATA_COLUMNS = [
    "dataset",
    "n_train",
    "target_epsilon",
    "delta",
    "batch_size",
    "sample_rate",
    "epochs",
    "max_grad_norm",
    "noise_multiplier",
    "accountant",
    "sampling",
    "clipping",
    "loss_reduction",
]

missing_dp_columns = [
    column
    for column in REQUIRED_DP_METADATA_COLUMNS
    if column not in NB10_PRIVACY_METADATA_DF.columns
]

if missing_dp_columns:

    raise RuntimeError(
        "Notebook 10 DP metadata is missing:\n"
        + "\n".join(
            f"  - {column}"
            for column in missing_dp_columns
        )
    )

print("✓ Notebook 10 DP metadata schema validated.")


# --------------------------------------------------------------------------------------------------
# 15. Secure CSPRNG preflight
#
# IMPORTANT:
# secure_mode=True is mandatory for the final run.
# No automatic insecure fallback is permitted.
# --------------------------------------------------------------------------------------------------

TORCHCSPRNG_SPEC = importlib.util.find_spec(
    "torchcsprng"
)

if TORCHCSPRNG_SPEC is None:

    print("\n" + "=" * 100)
    print("SECURE RNG DEPENDENCY MISSING")
    print("=" * 100)

    print(
        "✗ torchcsprng is not installed."
    )

    print(
        f"Python : {sys.version.split()[0]}"
    )

    print(
        f"PyTorch: {torch.__version__}"
    )

    print(
        f"Opacus : {opacus.__version__}"
    )

    print(
        "\nSection 11 requires secure_mode=True."
    )

    print(
        "No insecure fallback will be used."
    )

    print(
        "\nSection 11 status: ENVIRONMENT BLOCKED"
    )

    raise ImportError(
        "Secure DP initialization blocked because "
        "torchcsprng is unavailable. "
        "Do not change secure_mode to False."
    )

print(
    f"✓ torchcsprng available: "
    f"{TORCHCSPRNG_SPEC.origin}"
)


# --------------------------------------------------------------------------------------------------
# 16. Secure PrivacyEngine construction test
# --------------------------------------------------------------------------------------------------

try:

    SECURE_RNG_TEST_ENGINE = PrivacyEngine(
        accountant=EXPECTED_ACCOUNTANT,
        secure_mode=True
    )

except Exception as exc:

    raise RuntimeError(
        "Opacus secure PrivacyEngine construction failed. "
        "The runtime is not ready for the final DP experiment."
    ) from exc


if not getattr(
    SECURE_RNG_TEST_ENGINE,
    "secure_rng",
    None
):

    raise RuntimeError(
        "Opacus secure RNG was not initialized."
    )

print(
    "✓ Opacus secure PrivacyEngine construction test passed."
)

print(
    "✓ Secure RNG initialized successfully."
)

del SECURE_RNG_TEST_ENGINE


# --------------------------------------------------------------------------------------------------
# 17. Training array dataset wrapper
# --------------------------------------------------------------------------------------------------

class ArrayDataset(
    Dataset
):

    def __init__(
        self,
        array
    ):

        array = np.asarray(
            array,
            dtype=np.float32
        )

        if array.ndim != 2:

            raise ValueError(
                f"Expected 2D array; received {array.shape}."
            )

        if not np.isfinite(
            array
        ).all():

            raise ValueError(
                "Training array contains NaN or infinite values."
            )

        self.x = torch.from_numpy(
            array
        )

    def __len__(
        self
    ):

        return self.x.shape[0]

    def __getitem__(
        self,
        index
    ):

        return self.x[index]


# --------------------------------------------------------------------------------------------------
# 18. Initialize Section 11 registries
# --------------------------------------------------------------------------------------------------

PRIVACY_ENGINES = {}
PRIVATE_CRITICS = {}
PRIVATE_CRITIC_OPTIMIZERS = {}
DP_TRAIN_LOADERS = {}

PRIVACY_INITIALIZATION = {}

DP_NOISE_MULTIPLIERS = {}
DP_TARGET_EPSILONS = {}
DP_DELTAS = {}
DP_BATCH_SIZES = {}
DP_SAMPLE_RATES = {}
DP_EPOCHS = {}


# --------------------------------------------------------------------------------------------------
# 19. Initialize secure DP mechanism for each dataset
# --------------------------------------------------------------------------------------------------

for dataset_name in EXPECTED_DATASETS:

    print("-" * 100)
    print(
        f"Initializing DP mechanism: "
        f"{dataset_name}"
    )

    # ----------------------------------------------------------------------------------------------
    # 19.1 Load transformed training data
    # ----------------------------------------------------------------------------------------------

    train_array = np.asarray(
        TRAIN_ARRAYS[
            dataset_name
        ],
        dtype=np.float32
    )

    if train_array.ndim != 2:

        raise RuntimeError(
            f"{dataset_name}: training array must be 2D."
        )

    if not np.isfinite(
        train_array
    ).all():

        raise RuntimeError(
            f"{dataset_name}: training array contains "
            "non-finite values."
        )

    n_train = int(
        train_array.shape[0]
    )

    # ----------------------------------------------------------------------------------------------
    # 19.2 Retrieve exactly one Notebook 10 metadata record
    # ----------------------------------------------------------------------------------------------

    metadata_matches = (
        NB10_PRIVACY_METADATA_DF[
            NB10_PRIVACY_METADATA_DF[
                "dataset"
            ].astype(str)
            == dataset_name
        ]
        .copy()
    )

    if len(
        metadata_matches
    ) != 1:

        raise RuntimeError(
            f"{dataset_name}: expected exactly one "
            f"Notebook 10 metadata row; "
            f"found {len(metadata_matches)}."
        )

    dp_row = metadata_matches.iloc[0]

    metadata_n_train = int(
        dp_row[
            "n_train"
        ]
    )

    target_epsilon = float(
        dp_row[
            "target_epsilon"
        ]
    )

    delta = float(
        dp_row[
            "delta"
        ]
    )

    batch_size = int(
        dp_row[
            "batch_size"
        ]
    )

    sample_rate = float(
        dp_row[
            "sample_rate"
        ]
    )

    epochs = int(
        dp_row[
            "epochs"
        ]
    )

    max_grad_norm = float(
        dp_row[
            "max_grad_norm"
        ]
    )

    noise_multiplier = float(
        dp_row[
            "noise_multiplier"
        ]
    )

    accountant = str(
        dp_row[
            "accountant"
        ]
    ).lower()

    sampling = str(
        dp_row[
            "sampling"
        ]
    ).lower()

    clipping = str(
        dp_row[
            "clipping"
        ]
    ).lower()

    loss_reduction = str(
        dp_row[
            "loss_reduction"
        ]
    ).lower()

    # ----------------------------------------------------------------------------------------------
    # 19.3 Validate training size
    # ----------------------------------------------------------------------------------------------

    if metadata_n_train != n_train:

        raise ValueError(
            f"{dataset_name}: Notebook 10 n_train="
            f"{metadata_n_train} does not match "
            f"Notebook 02 TRAIN rows={n_train}."
        )

    # ----------------------------------------------------------------------------------------------
    # 19.4 Validate delta
    # ----------------------------------------------------------------------------------------------

    expected_delta = min(
        1e-5,
        1.0 / n_train
    )

    if not np.isclose(
        delta,
        expected_delta
    ):

        raise ValueError(
            f"{dataset_name}: delta mismatch. "
            f"Expected={expected_delta}; "
            f"Observed={delta}."
        )

    # ----------------------------------------------------------------------------------------------
    # 19.5 Validate DP values
    # ----------------------------------------------------------------------------------------------

    if not np.isclose(
        target_epsilon,
        EXPECTED_TARGET_EPSILON
    ):

        raise ValueError(
            f"{dataset_name}: target epsilon mismatch."
        )

    if batch_size != EXPECTED_BATCH_SIZE:

        raise ValueError(
            f"{dataset_name}: batch size mismatch."
        )

    if epochs != EXPECTED_EPOCHS:

        raise ValueError(
            f"{dataset_name}: epochs mismatch."
        )

    if not np.isclose(
        max_grad_norm,
        EXPECTED_MAX_GRAD_NORM
    ):

        raise ValueError(
            f"{dataset_name}: max_grad_norm mismatch."
        )

    if noise_multiplier <= 0:

        raise ValueError(
            f"{dataset_name}: noise_multiplier must be positive."
        )

    if accountant != EXPECTED_ACCOUNTANT:

        raise ValueError(
            f"{dataset_name}: accountant mismatch."
        )

    if sampling != EXPECTED_SAMPLING:

        raise ValueError(
            f"{dataset_name}: sampling mismatch."
        )

    if clipping != EXPECTED_CLIPPING:

        raise ValueError(
            f"{dataset_name}: clipping mismatch."
        )

    if loss_reduction != EXPECTED_LOSS_REDUCTION:

        raise ValueError(
            f"{dataset_name}: loss_reduction mismatch."
        )

    # ----------------------------------------------------------------------------------------------
    # 19.6 Validate sampling rate
    # ----------------------------------------------------------------------------------------------

    expected_sample_rate = (
        batch_size
        / n_train
    )

    if not np.isclose(
        sample_rate,
        expected_sample_rate,
        rtol=1e-5,
        atol=1e-8
    ):

        raise ValueError(
            f"{dataset_name}: sample rate mismatch. "
            f"Expected={expected_sample_rate}; "
            f"Observed={sample_rate}."
        )

    # ----------------------------------------------------------------------------------------------
    # 19.7 Retrieve exact Section 10 models
    # ----------------------------------------------------------------------------------------------

    critic = SPPGAN_MODELS[
        dataset_name
    ][
        "critic"
    ]

    generator = SPPGAN_MODELS[
        dataset_name
    ][
        "generator"
    ]

    if critic is not CRITICS[
        dataset_name
    ]:

        raise RuntimeError(
            f"{dataset_name}: critic reference mismatch."
        )

    if generator is not GENERATORS[
        dataset_name
    ]:

        raise RuntimeError(
            f"{dataset_name}: generator reference mismatch."
        )

    # ----------------------------------------------------------------------------------------------
    # 19.8 Build critic optimizer from Section 8
    # ----------------------------------------------------------------------------------------------

    CRITIC_CONFIG = TRAINING_CONFIG[
        "critic"
    ]

    critic_optimizer = torch.optim.AdamW(
        critic.parameters(),
        lr=float(
            CRITIC_CONFIG[
                "learning_rate"
            ]
        ),
        weight_decay=float(
            CRITIC_CONFIG[
                "weight_decay"
            ]
        )
    )

    # ----------------------------------------------------------------------------------------------
    # 19.9 Build training dataset
    # ----------------------------------------------------------------------------------------------

    train_dataset = ArrayDataset(
        train_array
    )

    # ----------------------------------------------------------------------------------------------
    # 19.10 Base DataLoader
    # ----------------------------------------------------------------------------------------------

    base_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        drop_last=False,
        pin_memory=torch.cuda.is_available()
    )

    # ----------------------------------------------------------------------------------------------
    # 19.11 Construct final secure PrivacyEngine
    # ----------------------------------------------------------------------------------------------

    privacy_engine = PrivacyEngine(
        accountant=accountant,
        secure_mode=True
    )

    # ----------------------------------------------------------------------------------------------
    # 19.12 Apply DP-SGD to the exact Section 10 critic
    # ----------------------------------------------------------------------------------------------

    (
        private_critic,
        private_optimizer,
        private_loader
    ) = privacy_engine.make_private(
        module=critic,
        optimizer=critic_optimizer,
        data_loader=base_loader,
        noise_multiplier=noise_multiplier,
        max_grad_norm=max_grad_norm,
        batch_first=True,
        loss_reduction=loss_reduction,
        poisson_sampling=True,
        clipping=clipping,
        grad_sample_mode="hooks"
    )

    # ----------------------------------------------------------------------------------------------
    # 19.13 Register DP objects
    # ----------------------------------------------------------------------------------------------

    PRIVACY_ENGINES[
        dataset_name
    ] = privacy_engine

    PRIVATE_CRITICS[
        dataset_name
    ] = private_critic

    PRIVATE_CRITIC_OPTIMIZERS[
        dataset_name
    ] = private_optimizer

    DP_TRAIN_LOADERS[
        dataset_name
    ] = private_loader

    DP_NOISE_MULTIPLIERS[
        dataset_name
    ] = noise_multiplier

    DP_TARGET_EPSILONS[
        dataset_name
    ] = target_epsilon

    DP_DELTAS[
        dataset_name
    ] = delta

    DP_BATCH_SIZES[
        dataset_name
    ] = batch_size

    DP_SAMPLE_RATES[
        dataset_name
    ] = sample_rate

    DP_EPOCHS[
        dataset_name
    ] = epochs

    PRIVACY_INITIALIZATION[
        dataset_name
    ] = {

        "dataset":
            dataset_name,

        "n_train":
            n_train,

        "target_epsilon":
            target_epsilon,

        "delta":
            delta,

        "batch_size":
            batch_size,

        "sample_rate":
            sample_rate,

        "epochs":
            epochs,

        "max_grad_norm":
            max_grad_norm,

        "noise_multiplier":
            noise_multiplier,

        "accountant":
            accountant,

        "sampling":
            sampling,

        "clipping":
            clipping,

        "loss_reduction":
            loss_reduction,

        "secure_mode":
            True,

        "protected_component":
            "discriminator_critic",

        "generator_private":
            False,

        "statistical_guidance_private":
            False,

        "preprocessing_private":
            False,

        "end_to_end_privacy_claim":
            False
    }

    print(
        f"✓ {dataset_name:<20}"
        f"PrivacyEngine initialized | "
        f"noise_multiplier={noise_multiplier:.8f} | "
        f"ε={target_epsilon:.4f} | "
        f"δ={delta:.3e} | "
        f"secure_mode=True"
    )


# --------------------------------------------------------------------------------------------------
# 20. Final privacy validation
# --------------------------------------------------------------------------------------------------

print("\n" + "-" * 100)
print("PRIVACY INITIALIZATION VALIDATION")
print("-" * 100)


for dataset_name in EXPECTED_DATASETS:

    private_critic = PRIVATE_CRITICS[
        dataset_name
    ]

    checks = {

        "PrivacyEngine":
            isinstance(
                PRIVACY_ENGINES[
                    dataset_name
                ],
                PrivacyEngine
            ),

        "PrivateCritic":
            isinstance(
                private_critic,
                torch.nn.Module
            ),

        "Optimizer":
            PRIVATE_CRITIC_OPTIMIZERS[
                dataset_name
            ] is not None,

        "DPDataLoader":
            DP_TRAIN_LOADERS[
                dataset_name
            ] is not None,

        "PositiveNoise":
            DP_NOISE_MULTIPLIERS[
                dataset_name
            ] > 0,

        "TargetEpsilon":
            np.isclose(
                DP_TARGET_EPSILONS[
                    dataset_name
                ],
                EXPECTED_TARGET_EPSILON
            ),

        "RDP":
            PRIVACY_INITIALIZATION[
                dataset_name
            ][
                "accountant"
            ] == "rdp",

        "Poisson":
            PRIVACY_INITIALIZATION[
                dataset_name
            ][
                "sampling"
            ] == "poisson",

        "FlatClipping":
            PRIVACY_INITIALIZATION[
                dataset_name
            ][
                "clipping"
            ] == "flat",

        "MeanReduction":
            PRIVACY_INITIALIZATION[
                dataset_name
            ][
                "loss_reduction"
            ] == "mean",

        "SecureMode":
            PRIVACY_INITIALIZATION[
                dataset_name
            ][
                "secure_mode"
            ] is True,

        "ExactSection10Critic":
            getattr(
                private_critic,
                "_module",
                None
            ) is SPPGAN_MODELS[
                dataset_name
            ][
                "critic"
            ],

        "GeneratorOutsidePrivacyEngine":
            PRIVACY_INITIALIZATION[
                dataset_name
            ][
                "generator_private"
            ] is False,

        "StatisticalGuidanceOutsidePrivacyEngine":
            PRIVACY_INITIALIZATION[
                dataset_name
            ][
                "statistical_guidance_private"
            ] is False,

        "PreprocessingOutsidePrivacyEngine":
            PRIVACY_INITIALIZATION[
                dataset_name
            ][
                "preprocessing_private"
            ] is False,

        "EndToEndClaimFalse":
            PRIVACY_INITIALIZATION[
                dataset_name
            ][
                "end_to_end_privacy_claim"
            ] is False
    }

    failed_checks = [
        name
        for name, passed in checks.items()
        if not passed
    ]

    if failed_checks:

        raise RuntimeError(
            f"{dataset_name}: privacy validation failed:\n"
            + "\n".join(
                f"  - {name}"
                for name in failed_checks
            )
        )

    print(
        f"✓ {dataset_name:<20}"
        f"{sum(checks.values())}/{len(checks)} checks PASS"
    )


# --------------------------------------------------------------------------------------------------
# 21. Persist Section 11 manifest
# --------------------------------------------------------------------------------------------------

PRIVACY_MANIFEST_PATH = (
    NB12_PRIVACY_ROOT
    / "sppgan_privacy_initialization_manifest.json"
)

PRIVACY_MANIFEST = {

    "notebook":
        "12",

    "section":
        "11",

    "section_name":
        "Initialize Privacy Mechanism",

    "status":
        "PASS",

    "run_mode":
        "FINAL_DP_CONFIGURATION",

    "python_version":
        sys.version,

    "pytorch_version":
        torch.__version__,

    "opacus_version":
        opacus.__version__,

    "torchcsprng_available":
        True,

    "secure_mode":
        True,

    "accountant":
        "rdp",

    "sampling":
        "poisson",

    "clipping":
        "flat",

    "loss_reduction":
        "mean",

    "model_registry":
        "SPPGAN_MODELS",

    "datasets":
        PRIVACY_INITIALIZATION,

    "privacy_boundary":
        EXPECTED_PRIVACY_BOUNDARY
}

with open(
    PRIVACY_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        PRIVACY_MANIFEST,
        file,
        indent=2,
        sort_keys=True
    )


# --------------------------------------------------------------------------------------------------
# 22. Manifest SHA256
# --------------------------------------------------------------------------------------------------

with open(
    PRIVACY_MANIFEST_PATH,
    "rb"
) as file:

    PRIVACY_MANIFEST_SHA256 = hashlib.sha256(
        file.read()
    ).hexdigest()


# --------------------------------------------------------------------------------------------------
# 23. Final completion summary
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("SECTION 11 FINAL SUMMARY")
print("=" * 100)

print(
    "✓ Exact Section 10 SPPGAN_MODELS registry retained."
)

print(
    "✓ Exact Section 10 generators retained."
)

print(
    "✓ Exact Section 10 critics retained."
)

print(
    "✓ Section 8 privacy configuration validated."
)

print(
    "✓ Notebook 10 DP metadata validated."
)

print(
    "✓ torchcsprng available."
)

print(
    "✓ Secure RNG initialized."
)

print(
    "✓ PrivacyEngine configured with secure_mode=True."
)

print(
    "✓ RDP accountant configured."
)

print(
    "✓ Poisson sampling configured."
)

print(
    "✓ Flat clipping configured."
)

print(
    "✓ Mean loss reduction configured."
)

print(
    "✓ Critic is the protected component."
)

print(
    "✓ Generator remains outside PrivacyEngine."
)

print(
    "✓ Statistical guidance remains outside PrivacyEngine."
)

print(
    "✓ Preprocessing remains outside PrivacyEngine."
)

print(
    "✓ End-to-end privacy claim remains FALSE."
)

print(
    f"✓ Privacy manifest:"
    f"\n  {PRIVACY_MANIFEST_PATH}"
)

print(
    f"✓ Manifest SHA256:"
    f"\n  {PRIVACY_MANIFEST_SHA256}"
)

print("=" * 100)
print("SECTION 11 STATUS: PASS — READY FOR FREEZE")
print("=" * 100)

11. INITIALIZE PRIVACY MECHANISM
✓ Required Section 10/Notebook 12 objects available.
✓ Section 10 SPPGAN_MODELS registry located.
✓ GENERATORS references recovered from SPPGAN_MODELS.
✓ CRITICS references recovered from SPPGAN_MODELS.
✓ Generator references match Section 10.
✓ Critic references match Section 10.
✓ TRAIN_ARRAYS available.
✓ Section 8 privacy configuration validated.
✓ Frozen DP parameters validated.
✓ Privacy boundary validated.
✓ Notebook 10 DP metadata schema validated.

SECURE RNG DEPENDENCY MISSING
✗ torchcsprng is not installed.
Python : 3.13.15
PyTorch: 2.11.0+cpu
Opacus : 1.6.0

Section 11 requires secure_mode=True.
No insecure fallback will be used.

Section 11 status: ENVIRONMENT BLOCKED


ImportError: Secure DP initialization blocked because torchcsprng is unavailable. Do not change secure_mode to False.

In [13]:
import sys
import subprocess
import importlib.util

print("=" * 100)
print("SECURE DP COMPATIBILITY DIAGNOSTIC")
print("=" * 100)

print("Python :", sys.version)

try:
    import torch
    print("PyTorch:", torch.__version__)
except Exception as e:
    print("PyTorch ERROR:", repr(e))

try:
    import opacus
    print("Opacus :", opacus.__version__)
except Exception as e:
    print("Opacus ERROR:", repr(e))

print(
    "torchcsprng import spec:",
    importlib.util.find_spec("torchcsprng")
)

print("\nInstalled torch-related packages:")
result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "list",
        "--format=freeze"
    ],
    capture_output=True,
    text=True
)

for line in result.stdout.splitlines():
    if any(
        name in line.lower()
        for name in [
            "torch",
            "opacus",
            "csprng",
            "numpy",
            "scipy"
        ]
    ):
        print(line)

print("=" * 100)

SECURE DP COMPATIBILITY DIAGNOSTIC
Python : 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cpu
Opacus : 1.6.0
torchcsprng import spec: None

Installed torch-related packages:
numpy==2.1.3
opacus==1.6.0
scipy==1.16.3
torch==2.11.0+cpu
torchao==0.10.0
torchaudio==2.11.0+cpu
torchcodec==0.11.0+cpu
torchdata==0.11.0
torchsummary==1.5.1
torchtune==0.6.1
torchvision==0.26.0+cpu


In [14]:
import inspect
import opacus
from opacus import PrivacyEngine

print("=" * 100)
print("OPACUS SECURE RNG IMPLEMENTATION INSPECTION")
print("=" * 100)

print("Opacus version:", opacus.__version__)
print("PrivacyEngine:", inspect.getfile(PrivacyEngine))

source = inspect.getsource(PrivacyEngine)

for i, line in enumerate(source.splitlines(), start=1):
    if (
        "secure_mode" in line
        or "csprng" in line
        or "secure_rng" in line
    ):
        print(f"{i:4d}: {line}")

print("=" * 100)

OPACUS SECURE RNG IMPLEMENTATION INSPECTION
Opacus version: 1.6.0
PrivacyEngine: /usr/local/lib/python3.13/dist-packages/opacus/privacy_engine.py
  26:     def __init__(self, *, accountant: str = "prv", secure_mode: bool = False):
  34:             secure_mode: Set to ``True`` if cryptographically strong DP guarantee is
  35:                 required. ``secure_mode=True`` uses secure random number generator for
  39:                 When set to ``True`` requires ``torchcsprng`` to be installed
  42:         self.secure_mode = secure_mode
  43:         self.secure_rng = None
  45:         if self.secure_mode:
  47:                 import torchcsprng as csprng
  50:                     "To use secure RNG, you must install the torchcsprng package! "
  51:                     "Check out the instructions here: https://github.com/pytorch/csprng#installation"
  55:             self.secure_rng = csprng.create_random_device_generator("/dev/urandom")
  60:                 "one last time before p

In [15]:
import inspect
from opacus import PrivacyEngine

print(inspect.getsource(PrivacyEngine.__init__))

    def __init__(self, *, accountant: str = "prv", secure_mode: bool = False):
        """

        Args:
            accountant: Accounting mechanism. Currently supported:
                - rdp (:class:`~opacus.accountants.RDPAccountant`)
                - gdp (:class:`~opacus.accountants.GaussianAccountant`)
                - prv (:class`~opacus.accountants.PRVAccountant`)
            secure_mode: Set to ``True`` if cryptographically strong DP guarantee is
                required. ``secure_mode=True`` uses secure random number generator for
                noise and shuffling (as opposed to pseudo-rng in vanilla PyTorch) and
                prevents certain floating-point arithmetic-based attacks.
                See :meth:`~opacus.optimizers.optimizer._generate_noise` for details.
                When set to ``True`` requires ``torchcsprng`` to be installed
        """
        self.accountant = create_accountant(mechanism=accountant)
        self.secure_mode = secure_mode
        s

In [16]:
import inspect
import opacus.utils

print("=" * 100)
print("OPACUS UTILITY MODULES")
print("=" * 100)

print(opacus.utils.__file__)

OPACUS UTILITY MODULES
/usr/local/lib/python3.13/dist-packages/opacus/utils/__init__.py


In [17]:
import os
import opacus

opacus_root = os.path.dirname(opacus.__file__)

print("Opacus root:")
print(opacus_root)

for root, dirs, files in os.walk(opacus_root):
    for file in files:
        if file.endswith(".py"):
            path = os.path.join(root, file)

            try:
                with open(path, "r", encoding="utf-8") as f:
                    text = f.read()

                if "torchcsprng" in text or "secure_mode" in text:
                    print("\nFOUND:", path)

                    for line_no, line in enumerate(
                        text.splitlines(),
                        start=1
                    ):
                        if (
                            "torchcsprng" in line
                            or "secure_mode" in line
                        ):
                            print(f"{line_no:4d}: {line}")

            except Exception:
                pass

Opacus root:
/usr/local/lib/python3.13/dist-packages/opacus

FOUND: /usr/local/lib/python3.13/dist-packages/opacus/privacy_engine.py
  67:     def __init__(self, *, accountant: str = "prv", secure_mode: bool = False):
  75:             secure_mode: Set to ``True`` if cryptographically strong DP guarantee is
  76:                 required. ``secure_mode=True`` uses secure random number generator for
  80:                 When set to ``True`` requires ``torchcsprng`` to be installed
  83:         self.secure_mode = secure_mode
  86:         if self.secure_mode:
  88:                 import torchcsprng as csprng
  91:                     "To use secure RNG, you must install the torchcsprng package! "
 101:                 "one last time before production with ``secure_mode`` turned on."
 122:         if self.secure_mode:
 140:             secure_mode=self.secure_mode,
 173:         elif self.secure_mode:
 406:         if noise_generator and self.secure_mode:

FOUND: /usr/local/lib/python3

In [38]:
# ==================================================================================================
# SECURE CSPRNG COMPATIBILITY CHECK
# ==================================================================================================

import sys
import subprocess
import importlib.util

print("=" * 100)
print("SECURE CSPRNG COMPATIBILITY CHECK")
print("=" * 100)

print(f"Python : {sys.version}")
print(f"PyTorch: {__import__('torch').__version__}")
print(f"Opacus : {__import__('opacus').__version__}")

print("\nInstalled torchcsprng:")
print(
    importlib.util.find_spec("torchcsprng")
)

print("\nAvailable pip versions:")
result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "index",
        "versions",
        "torchcsprng",
    ],
    capture_output=True,
    text=True,
)

print(
    result.stdout
)

if result.stderr:
    print("\nPip message:")
    print(result.stderr)

print("=" * 100)

SECURE CSPRNG COMPATIBILITY CHECK
Python : 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PyTorch: 2.11.0+cpu
Opacus : 1.6.0

Installed torchcsprng:
None

Available pip versions:


Pip message:
ERROR: No matching distribution found for torchcsprng



In [ ]:
# ==================================================================================================
# 12. TRAINING LOOP
# ==================================================================================================

print("=" * 100)
print("12. TRAINING LOOP")
print("=" * 100)

TRAINING_STATE = {}

for dataset_id in DATASET_IDS:

    transformed_dim = (
        SPPGAN_MODELS[
            dataset_id
        ][
            "transformed_dim"
        ]
    )

    TRAINING_STATE[dataset_id] = {

        "dataset_id":
            dataset_id,

        "epoch":
            0,

        "global_step":
            0,

        "generator":
            SPPGAN_MODELS[
                dataset_id
            ][
                "generator"
            ],

        "critic":
            None,

        "g_optimizer":
            None,

        "d_optimizer":
            None,

        "privacy_engine":
            None,

        "train_loader":
            None,

        "history":
            [],

        "failures":
            [],

        "runtime_seconds":
            0.0,

        "status":
            "INITIALIZED",
    }

print(
    f"✓ Training state initialized for "
    f"{len(DATASET_IDS)} datasets."
)

print(
    "✓ One dataset is trained at a time."
)

print(
    "✓ This design limits simultaneous RAM/GPU allocation."
)

print(
    "\nTraining order:"
)

for dataset_id in DATASET_IDS:
    print(
        f"  {dataset_id}"
    )

12. TRAINING LOOP


KeyError: 'adult_income'

In [ ]:
# ==================================================================================================
# 13. DISCRIMINATOR / CRITIC UPDATES
# ==================================================================================================

print("=" * 100)
print("13. DISCRIMINATOR / CRITIC UPDATES")
print("=" * 100)

def critic_update(
    critic,
    optimizer,
    real_batch,
    fake_batch,
):
    """
    WGAN-style discriminator objective:

        L_D = E[D(x_fake)] - E[D(x_real)]

    The critic is wrapped by Opacus.
    Per-example gradients are clipped and Gaussian noise is
    added by the PrivacyEngine before optimizer.step().
    """

    optimizer.zero_grad(
        set_to_none=True
    )

    real_scores = critic(
        real_batch
    )

    fake_scores = critic(
        fake_batch.detach()
    )

    if real_scores.ndim != 2:
        raise RuntimeError(
            f"Unexpected real score shape: "
            f"{real_scores.shape}"
        )

    if fake_scores.ndim != 2:
        raise RuntimeError(
            f"Unexpected fake score shape: "
            f"{fake_scores.shape}"
        )

    loss = (
        fake_scores.mean()
        -
        real_scores.mean()
    )

    if not torch.isfinite(
        loss
    ):
        raise FloatingPointError(
            "Non-finite discriminator loss."
        )

    loss.backward()

    optimizer.step()

    return float(
        loss.detach().cpu()
    )

13. DISCRIMINATOR / CRITIC UPDATES


In [ ]:
# ==================================================================================================
# 14. GENERATOR UPDATES
# ==================================================================================================

print("=" * 100)
print("14. GENERATOR UPDATES")
print("=" * 100)

def generator_adversarial_loss(
    critic,
    fake_batch
):
    """
    WGAN-style generator adversarial loss:

        L_adv = -E[D(G(z))]
    """

    fake_scores = critic(
        fake_batch
    )

    loss = -fake_scores.mean()

    if not torch.isfinite(
        loss
    ):
        raise FloatingPointError(
            "Non-finite generator adversarial loss."
        )

    return loss

14. GENERATOR UPDATES


In [ ]:
# ==================================================================================================
# 15. STATISTICAL-GUIDANCE LOSS
# ==================================================================================================

print("=" * 100)
print("15. STATISTICAL-GUIDANCE LOSS")
print("=" * 100)

GUIDANCE_MODULE_PATH = (
    NB09_ROOT /
    "models" /
    "sppgan_statistical_guidance.py"
)

if not GUIDANCE_MODULE_PATH.exists():
    raise FileNotFoundError(
        f"Notebook 09 statistical-guidance module not found:\n"
        f"{GUIDANCE_MODULE_PATH}"
    )

guidance_spec = (
    importlib.util.spec_from_file_location(
        "sppgan_statistical_guidance_nb12",
        GUIDANCE_MODULE_PATH
    )
)

GUIDANCE_MODULE = (
    importlib.util.module_from_spec(
        guidance_spec
    )
)

guidance_spec.loader.exec_module(
    GUIDANCE_MODULE
)

REQUIRED_GUIDANCE_FUNCTIONS = [
    "differentiable_pearson_matrix",
    "dependency_frobenius_loss",
]

missing_guidance_functions = [
    name
    for name in REQUIRED_GUIDANCE_FUNCTIONS
    if not hasattr(
        GUIDANCE_MODULE,
        name
    )
]

if missing_guidance_functions:
    raise RuntimeError(
        "Notebook 09 guidance module is incomplete:\n"
        f"{missing_guidance_functions}"
    )

# --------------------------------------------------------------------------------------------------
# Training-time statistical representation
# --------------------------------------------------------------------------------------------------
#
# Notebook 09 validates the differentiable components.
# Notebook 12 uses the frozen training representation:
#
#   numerical representation -> first n_numeric transformed columns
#   categorical representation -> grouped one-hot blocks
#
# The exact feature mapping is obtained from Notebook 02 metadata.
# --------------------------------------------------------------------------------------------------

PREPROCESSING_METADATA = {}

for dataset_id in DATASET_IDS:

    metadata_path = (
        NB02_METADATA_ROOT /
        f"{dataset_id}_preprocessing_metadata.json"
    )

    with open(
        metadata_path,
        "r",
        encoding="utf-8"
    ) as f:

        PREPROCESSING_METADATA[
            dataset_id
        ] = json.load(f)

print(
    "✓ Notebook 09 guidance module loaded."
)

print(
    "✓ Notebook 02 preprocessing metadata loaded."
)

print(
    "✓ Statistical guidance will be evaluated "
    "on the differentiable transformed representation."
)

15. STATISTICAL-GUIDANCE LOSS
✓ Notebook 09 guidance module loaded.
✓ Notebook 02 preprocessing metadata loaded.
✓ Statistical guidance will be evaluated on the differentiable transformed representation.


In [ ]:
# ==================================================================================================
# 16. PRIVACY MECHANISM
# ==================================================================================================

print("=" * 100)
print("16. PRIVACY MECHANISM")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# Load calibrated noise multipliers from Notebook 11
# --------------------------------------------------------------------------------------------------

NB11_ACCOUNTING_PATH = (
    NB11_ROOT /
    "accounting" /
    "sppgan_configured_schedule_rdp_accounting.csv"
)

if not NB11_ACCOUNTING_PATH.exists():
    raise FileNotFoundError(
        f"Notebook 11 configured accounting artifact missing:\n"
        f"{NB11_ACCOUNTING_PATH}"
    )

NB11_ACCOUNTING_DF = pd.read_csv(
    NB11_ACCOUNTING_PATH
)

REQUIRED_NB11_COLUMNS = {
    "dataset",
    "n_train",
    "configured_schedule_epsilon",
    "delta",
    "sample_rate",
    "noise_multiplier",
    "total_steps",
    "optimal_rdp_order",
    "accounting_type",
    "achieved_epsilon_status",
}

missing_nb11 = (
    REQUIRED_NB11_COLUMNS -
    set(NB11_ACCOUNTING_DF.columns)
)

if missing_nb11:
    raise RuntimeError(
        "Notebook 11 accounting schema is incomplete:\n"
        f"{sorted(missing_nb11)}"
    )

if not NB11_ACCOUNTING_DF[
    "accounting_type"
].eq(
    "configured_schedule"
).all():

    raise RuntimeError(
        "Notebook 11 accounting type is not configured_schedule."
    )

NOISE_MULTIPLIERS = {}

for dataset_id in DATASET_IDS:

    row = NB11_ACCOUNTING_DF[
        NB11_ACCOUNTING_DF[
            "dataset"
        ].eq(
            dataset_id
        )
    ]

    if len(row) != 1:
        raise RuntimeError(
            f"{dataset_id}: expected exactly one Notebook 11 accounting row."
        )

    noise_multiplier = float(
        row.iloc[0][
            "noise_multiplier"
        ]
    )

    if not np.isfinite(
        noise_multiplier
    ) or noise_multiplier <= 0:

        raise ValueError(
            f"{dataset_id}: invalid noise multiplier."
        )

    NOISE_MULTIPLIERS[
        dataset_id
    ] = noise_multiplier

    print(
        f"✓ {dataset_id:<20} "
        f"noise_multiplier={noise_multiplier:.6f}"
    )

print(
    "\n✓ Notebook 11 calibrated noise multipliers loaded."
)

print(
    "✓ These values are used unchanged during DP training."
)

print(
    "✓ Actual epsilon will be read from PrivacyEngine after training."
)

16. PRIVACY MECHANISM
✓ adult_income         noise_multiplier=1.216221
✓ bank_marketing       noise_multiplier=1.251014
✓ diabetes_130us       noise_multiplier=0.953403

✓ Notebook 11 calibrated noise multipliers loaded.
✓ These values are used unchanged during DP training.
✓ Actual epsilon will be read from PrivacyEngine after training.


In [ ]:
# ==================================================================================================
# 17. TOTAL OBJECTIVE
# ==================================================================================================

print("=" * 100)
print("17. TOTAL OBJECTIVE")
print("=" * 100)

def total_generator_objective(
    adversarial_loss,
    statistical_loss,
    lambda_stat
):
    """
    SPP-GAN generator objective:

        L_G = L_adv + lambda_stat * L_stat
    """

    if not torch.isfinite(
        adversarial_loss
    ):
        raise FloatingPointError(
            "Non-finite adversarial loss."
        )

    if not torch.isfinite(
        statistical_loss
    ):
        raise FloatingPointError(
            "Non-finite statistical loss."
        )

    total_loss = (
        adversarial_loss
        +
        lambda_stat *
        statistical_loss
    )

    if not torch.isfinite(
        total_loss
    ):
        raise FloatingPointError(
            "Non-finite total generator loss."
        )

    return total_loss

print(
    "✓ Generator objective:"
)

print(
    "  L_G = L_adv + lambda_stat * L_stat"
)

print(
    f"  lambda_stat = {LAMBDA_STAT}"
)

print(
    "✓ Privacy is enforced through the protected critic update."
)

print(
    "✓ Privacy noise is NOT added as an extra generator loss term."
)

17. TOTAL OBJECTIVE
✓ Generator objective:
  L_G = L_adv + lambda_stat * L_stat
  lambda_stat = 1.0
✓ Privacy is enforced through the protected critic update.
✓ Privacy noise is NOT added as an extra generator loss term.


In [ ]:
# ==================================================================================================
# 18. VALIDATION MONITORING
# ==================================================================================================

print("=" * 100)
print("18. VALIDATION MONITORING")
print("=" * 100)

# Validation monitoring is intentionally lightweight.
# The validation split is NOT used for optimization.

def finite_tensor(
    tensor
):
    return bool(
        torch.isfinite(
            tensor
        ).all()
        .item()
    )


@torch.no_grad()
def validation_monitor(
    generator,
    critic,
    validation_batch_size=256
):
    """
    Model-health monitoring only.

    No optimizer update occurs here.
    No test data are used.
    """

    generator.eval()
    critic.eval()

    z = torch.randn(
        validation_batch_size,
        LATENT_DIM,
        device=DEVICE
    )

    fake = generator(
        z
    )

    critic_scores = critic(
        fake
    )

    result = {

        "generated_finite":
            finite_tensor(
                fake
            ),

        "critic_finite":
            finite_tensor(
                critic_scores
            ),

        "generated_mean":
            float(
                fake.mean().cpu()
            ),

        "generated_std":
            float(
                fake.std(
                    unbiased=False
                ).cpu()
            ),

        "critic_mean":
            float(
                critic_scores.mean().cpu()
            ),
    }

    generator.train()
    critic.train()

    return result

print(
    "✓ Validation monitoring function prepared."
)

print(
    "✓ Validation monitoring does not update model parameters."
)

print(
    "✓ Test split remains isolated."
)

18. VALIDATION MONITORING
✓ Validation monitoring function prepared.
✓ Validation monitoring does not update model parameters.
✓ Test split remains isolated.


In [ ]:
# ==================================================================================================
# 19. LOSS / METRIC LOGGING
# ==================================================================================================

print("=" * 100)
print("19. LOSS / METRIC LOGGING")
print("=" * 100)

HISTORY_COLUMNS = [

    "dataset",
    "repetition_id",
    "epoch",
    "global_step",

    "critic_loss",
    "generator_adversarial_loss",
    "statistical_loss",
    "generator_total_loss",

    "epsilon",
    "optimal_rdp_order",

    "learning_rate_generator",
    "learning_rate_critic",

    "generated_mean",
    "generated_std",

    "runtime_seconds",

    "cuda_memory_allocated_mb",
    "cuda_memory_reserved_mb",

    "status",
]

print(
    f"✓ History schema prepared: "
    f"{len(HISTORY_COLUMNS)} fields."
)

19. LOSS / METRIC LOGGING
✓ History schema prepared: 18 fields.


In [ ]:
# ==================================================================================================
# 20. CHECKPOINTING
# ==================================================================================================

print("=" * 100)
print("20. CHECKPOINTING")
print("=" * 100)

CHECKPOINT_INTERVAL = 25

def atomic_torch_save(
    payload,
    target_path
):

    target_path = Path(
        target_path
    )

    temporary_path = (
        target_path.parent /
        f".{target_path.name}.tmp"
    )

    torch.save(
        payload,
        temporary_path
    )

    temporary_path.replace(
        target_path
    )


def save_training_checkpoint(
    dataset_id,
    epoch,
    global_step,
    generator,
    critic,
    g_optimizer,
    d_optimizer,
    privacy_engine,
    noise_multiplier,
    history
):

    epsilon = float(
        privacy_engine.get_epsilon(
            min(
                1e-5,
                1.0 /
                TRAIN_ROWS[dataset_id]
            )
        )
    )

    checkpoint = {

        "notebook":
            "12",

        "framework":
            "SPP-GAN",

        "dataset":
            dataset_id,

        "repetition_id":
            REPETITION_ID,

        "epoch":
            int(epoch),

        "global_step":
            int(global_step),

        "generator_state_dict":
            generator.state_dict(),

        "critic_state_dict":
            critic.state_dict(),

        "generator_optimizer_state_dict":
            g_optimizer.state_dict(),

        "critic_optimizer_state_dict":
            d_optimizer.state_dict(),

        "privacy_engine_state":
            privacy_engine,

        "noise_multiplier":
            float(noise_multiplier),

        "achieved_epsilon":
            epsilon,

        "delta":
            min(
                1e-5,
                1.0 /
                TRAIN_ROWS[dataset_id]
            ),

        "training_config":
            TRAINING_CONFIG,

        "history":
            history,
    }

    checkpoint_path = (
        NB12_DIRS["checkpoints"] /
        f"{dataset_id}_repetition_{REPETITION_ID}_epoch_{epoch:03d}.pt"
    )

    atomic_torch_save(
        checkpoint,
        checkpoint_path
    )

    return checkpoint_path

print(
    f"✓ Checkpoint interval : "
    f"{CHECKPOINT_INTERVAL} epochs"
)

print(
    "✓ Checkpoints use atomic replacement."
)

print(
    "✓ Generator, critic, optimizers and privacy state are persisted."
)

20. CHECKPOINTING
✓ Checkpoint interval : 25 epochs
✓ Checkpoints use atomic replacement.
✓ Generator, critic, optimizers and privacy state are persisted.


In [ ]:
# ==================================================================================================
# 21. RUNTIME MONITORING
# ==================================================================================================

print("=" * 100)
print("21. RUNTIME MONITORING")
print("=" * 100)

def get_runtime_snapshot():

    snapshot = {

        "timestamp_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),

        "device":
            str(DEVICE),

        "cuda_available":
            bool(
                torch.cuda.is_available()
            ),

        "cuda_device":
            (
                torch.cuda.get_device_name(0)
                if torch.cuda.is_available()
                else None
            ),

        "cuda_memory_allocated_mb":
            (
                torch.cuda.memory_allocated()
                /
                (1024 ** 2)
                if torch.cuda.is_available()
                else 0.0
            ),

        "cuda_memory_reserved_mb":
            (
                torch.cuda.memory_reserved()
                /
                (1024 ** 2)
                if torch.cuda.is_available()
                else 0.0
            ),
    }

    return snapshot

print(
    "✓ Runtime monitoring initialized."
)

runtime_snapshot = get_runtime_snapshot()

for key, value in runtime_snapshot.items():

    print(
        f"  {key:<30}: {value}"
    )

21. RUNTIME MONITORING
✓ Runtime monitoring initialized.
  timestamp_utc                 : 2026-09-18T12:31:27.366912+00:00
  device                        : cpu
  cuda_available                : False
  cuda_device                   : None
  cuda_memory_allocated_mb      : 0.0
  cuda_memory_reserved_mb       : 0.0


In [ ]:
# ==================================================================================================
# 22. MEMORY MONITORING
# ==================================================================================================

print("=" * 100)
print("22. MEMORY MONITORING")
print("=" * 100)

def clear_runtime_memory():

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()

        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass


def memory_snapshot():

    result = {

        "cuda_memory_allocated_mb":
            0.0,

        "cuda_memory_reserved_mb":
            0.0,
    }

    if torch.cuda.is_available():

        result[
            "cuda_memory_allocated_mb"
        ] = (
            torch.cuda.memory_allocated()
            /
            (1024 ** 2)
        )

        result[
            "cuda_memory_reserved_mb"
        ] = (
            torch.cuda.memory_reserved()
            /
            (1024 ** 2)
        )

    return result


print(
    "✓ RAM/GPU memory monitoring functions ready."
)

print(
    "✓ Datasets are processed sequentially."
)

print(
    "✓ Validation batches are bounded."
)

print(
    "✓ Test data are not loaded."
)

22. MEMORY MONITORING
✓ RAM/GPU memory monitoring functions ready.
✓ Datasets are processed sequentially.
✓ Validation batches are bounded.
✓ Test data are not loaded.


In [ ]:
# ==================================================================================================
# 23. FAILURE HANDLING
# ==================================================================================================

print("=" * 100)
print("23. FAILURE HANDLING")
print("=" * 100)

TRAINING_FAILURES = []

def record_training_failure(
    dataset_id,
    epoch,
    global_step,
    exception
):

    record = {

        "dataset":
            dataset_id,

        "repetition_id":
            REPETITION_ID,

        "epoch":
            int(epoch),

        "global_step":
            int(global_step),

        "exception_type":
            type(exception).__name__,

        "exception_message":
            str(exception),

        "traceback":
            traceback.format_exc(),

        "timestamp_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }

    TRAINING_FAILURES.append(
        record
    )

    return record


FAILURE_PATH = (
    NB12_DIRS["audit"] /
    "training_failures.json"
)

print(
    f"✓ Failure registry initialized."
)

print(
    f"✓ Failure artifact: {FAILURE_PATH}"
)

23. FAILURE HANDLING
✓ Failure registry initialized.
✓ Failure artifact: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_12/audit/training_failures.json


In [ ]:
# ==================================================================================================
# 24. SAVE FINAL MODELS
# ==================================================================================================

print("=" * 100)
print("24. SAVE FINAL MODELS")
print("=" * 100)

FINAL_MODELS = {}
FINAL_PRIVACY_RESULTS = {}

# --------------------------------------------------------------------------------------------------
# Helper
# --------------------------------------------------------------------------------------------------

def save_final_model_bundle(
    dataset_id,
    generator,
    critic,
    g_optimizer,
    d_optimizer,
    privacy_engine,
    history,
    runtime_seconds,
    noise_multiplier,
):

    delta = min(
        1e-5,
        1.0 /
        TRAIN_ROWS[dataset_id]
    )

    achieved_epsilon = float(
        privacy_engine.get_epsilon(
            delta
        )
    )

    bundle = {

        "notebook":
            "12",

        "framework":
            "SPP-GAN",

        "dataset":
            dataset_id,

        "repetition_id":
            REPETITION_ID,

        "generator_state_dict":
            generator.state_dict(),

        "critic_state_dict":
            critic.state_dict(),

        "generator_optimizer_state_dict":
            g_optimizer.state_dict(),

        "critic_optimizer_state_dict":
            d_optimizer.state_dict(),

        "training_configuration":
            TRAINING_CONFIG,

        "privacy_configuration":
            {
                "accountant":
                    ACCOUNTANT,

                "sampling":
                    SAMPLING_MECHANISM,

                "clipping":
                    CLIPPING_MECHANISM,

                "max_grad_norm":
                    MAX_GRAD_NORM,

                "noise_multiplier":
                    noise_multiplier,

                "delta":
                    delta,

                "target_epsilon":
                    TARGET_EPSILON,

                "achieved_epsilon":
                    achieved_epsilon,
            },

        "privacy_boundary":
            {
                "generator_private":
                    False,

                "statistical_guidance_private":
                    False,

                "preprocessing_private":
                    False,

                "end_to_end_privacy_claim":
                    False,
            },

        "history":
            history,

        "runtime_seconds":
            runtime_seconds,

        "created_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }

    model_path = (
        NB12_DIRS["models"] /
        f"{dataset_id}_sppgan_dp_repetition_{REPETITION_ID}.pt"
    )

    atomic_torch_save(
        bundle,
        model_path
    )

    return (
        model_path,
        achieved_epsilon
    )

24. SAVE FINAL MODELS


In [ ]:
# ==================================================================================================
# 25. SAVE TRAINING HISTORIES
# ==================================================================================================

print("=" * 100)
print("25. SAVE TRAINING HISTORIES")
print("=" * 100)

HISTORY_PATHS = {}

def save_training_history(
    dataset_id,
    history
):

    history_df = pd.DataFrame(
        history
    )

    path = (
        NB12_DIRS["history"] /
        f"{dataset_id}_sppgan_dp_repetition_{REPETITION_ID}_history.csv"
    )

    history_df.to_csv(
        path,
        index=False
    )

    return path

print(
    "✓ Training-history persistence function ready."
)

25. SAVE TRAINING HISTORIES
✓ Training-history persistence function ready.


In [ ]:
# ==================================================================================================
# 26. SAVE TRAINING METADATA
# ==================================================================================================

print("=" * 100)
print("26. SAVE TRAINING METADATA")
print("=" * 100)

TRAINING_METADATA_RECORDS = []

def build_training_metadata(
    dataset_id,
    runtime_seconds,
    achieved_epsilon,
    global_steps,
    noise_multiplier,
    status,
):

    delta = min(
        1e-5,
        1.0 /
        TRAIN_ROWS[dataset_id]
    )

    return {

        "notebook":
            "12",

        "framework":
            "SPP-GAN",

        "dataset":
            dataset_id,

        "repetition_id":
            REPETITION_ID,

        "training_rows":
            TRAIN_ROWS[dataset_id],

        "transformed_dimension":
            TRAIN_ARRAYS[
                dataset_id
            ].shape[1],

        "latent_dimension":
            LATENT_DIM,

        "generator_hidden_dim_1":
            GENERATOR_HIDDEN_DIM_1,

        "generator_hidden_dim_2":
            GENERATOR_HIDDEN_DIM_2,

        "critic_hidden_dim_1":
            CRITIC_HIDDEN_DIM_1,

        "critic_hidden_dim_2":
            CRITIC_HIDDEN_DIM_2,

        "generator_learning_rate":
            GENERATOR_LR,

        "critic_learning_rate":
            CRITIC_LR,

        "weight_decay":
            WEIGHT_DECAY,

        "lambda_stat":
            LAMBDA_STAT,

        "dp_batch_size_nominal":
            DP_BATCH_SIZE,

        "dp_epochs":
            DP_EPOCHS,

        "actual_optimizer_steps":
            global_steps,

        "max_grad_norm":
            MAX_GRAD_NORM,

        "noise_multiplier":
            noise_multiplier,

        "delta":
            delta,

        "target_epsilon":
            TARGET_EPSILON,

        "achieved_epsilon":
            achieved_epsilon,

        "accountant":
            ACCOUNTANT,

        "sampling_mechanism":
            SAMPLING_MECHANISM,

        "clipping_mechanism":
            CLIPPING_MECHANISM,

        "loss_reduction":
            LOSS_REDUCTION,

        "generator_private":
            False,

        "statistical_guidance_private":
            False,

        "preprocessing_private":
            False,

        "end_to_end_privacy_claim":
            False,

        "runtime_seconds":
            runtime_seconds,

        "status":
            status,

        "created_utc":
            datetime.now(
                timezone.utc
            ).isoformat(),
    }

print(
    "✓ Training metadata schema prepared."
)

26. SAVE TRAINING METADATA
✓ Training metadata schema prepared.


In [ ]:
# ==================================================================================================
# 27. SAVE TRAINING MANIFEST
# ==================================================================================================

print("=" * 100)
print("27. SAVE TRAINING MANIFEST")
print("=" * 100)

TRAINING_MANIFEST_PATH = (
    NB12_DIRS["manifests"] /
    "sppgan_notebook_12_training_manifest.json"
)

def sha256_file(
    path,
    chunk_size=1024 * 1024
):

    digest = hashlib.sha256()

    with open(
        path,
        "rb"
    ) as f:

        for chunk in iter(
            lambda:
                f.read(chunk_size),
            b""
        ):

            digest.update(
                chunk
            )

    return digest.hexdigest()

print(
    "✓ Training-manifest helper ready."
)

print(
    f"✓ Manifest path: {TRAINING_MANIFEST_PATH}"
)

27. SAVE TRAINING MANIFEST
✓ Training-manifest helper ready.
✓ Manifest path: /content/drive/MyDrive/SPP_GAN_Research/results/notebooks/notebook_12/manifests/sppgan_notebook_12_training_manifest.json


In [ ]:
# ==================================================================================================
# 28. FINAL VERIFICATION
# ==================================================================================================

print("=" * 100)
print("28. FINAL VERIFICATION")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# IMPORTANT:
# The actual training loop is executed here because all preceding
# sections define the complete training system.
# --------------------------------------------------------------------------------------------------

FINAL_RESULTS = []
TRAINING_METADATA_RECORDS = []
HISTORY_PATHS = {}
FINAL_MODELS = {}
FINAL_PRIVACY_RESULTS = {}

for dataset_id in DATASET_IDS:

    print("\n" + "=" * 100)
    print(
        f"TRAINING DATASET : {dataset_id}"
    )
    print("=" * 100)

    clear_runtime_memory()

    start_time = time.time()

    try:

        transformed_dim = (
            TRAIN_ARRAYS[
                dataset_id
            ].shape[1]
        )

        # ------------------------------------------------------------------------------------------
        # Fresh model initialization
        # ------------------------------------------------------------------------------------------

        dataset_seed = (
            EXPERIMENT_SEED
            +
            DATASET_IDS.index(
                dataset_id
            )
        )

        random.seed(
            dataset_seed
        )

        np.random.seed(
            dataset_seed
        )

        torch.manual_seed(
            dataset_seed
        )

        generator = SPPGANGenerator(
            LATENT_DIM,
            transformed_dim,
            GENERATOR_HIDDEN_DIM_1,
            GENERATOR_HIDDEN_DIM_2,
        ).to(
            DEVICE
        )

        critic = SPPGANCritic(
            transformed_dim,
            CRITIC_HIDDEN_DIM_1,
            CRITIC_HIDDEN_DIM_2,
        ).to(
            DEVICE
        )

        # ------------------------------------------------------------------------------------------
        # Generator optimizer
        # ------------------------------------------------------------------------------------------

        g_optimizer = torch.optim.AdamW(
            generator.parameters(),
            lr=GENERATOR_LR,
            weight_decay=WEIGHT_DECAY,
            betas=(0.9, 0.999)
        )

        # ------------------------------------------------------------------------------------------
        # Critic optimizer + DP PrivacyEngine
        # ------------------------------------------------------------------------------------------

        d_optimizer = torch.optim.AdamW(
            critic.parameters(),
            lr=CRITIC_LR,
            weight_decay=WEIGHT_DECAY,
            betas=(0.9, 0.999)
        )

        train_dataset = ArrayDataset(
            TRAIN_ARRAYS[
                dataset_id
            ]
        )

        raw_loader = torch.utils.data.DataLoader(
            train_dataset,
            batch_size=DP_BATCH_SIZE,
            shuffle=True,
            drop_last=False,
            num_workers=0,
            pin_memory=False,
        )

        privacy_engine = PrivacyEngine(
            accountant=ACCOUNTANT,
            secure_mode=False,
        )

        (
            private_critic,
            private_d_optimizer,
            private_train_loader,
        ) = privacy_engine.make_private(
            module=critic,
            optimizer=d_optimizer,
            data_loader=raw_loader,
            noise_multiplier=NOISE_MULTIPLIERS[
                dataset_id
            ],
            max_grad_norm=MAX_GRAD_NORM,
            batch_first=True,
            loss_reduction=LOSS_REDUCTION,
            poisson_sampling=True,
            clipping=CLIPPING_MECHANISM,
            grad_sample_mode="hooks",
        )

        # ------------------------------------------------------------------------------------------
        # Training containers
        # ------------------------------------------------------------------------------------------

        history = []

        global_step = 0

        dataset_status = "RUNNING"

        # ------------------------------------------------------------------------------------------
        # Training
        # ------------------------------------------------------------------------------------------

        for epoch in range(
            1,
            DP_EPOCHS + 1
        ):

            epoch_start = time.time()

            epoch_critic_losses = []
            epoch_g_adv_losses = []
            epoch_stat_losses = []
            epoch_total_losses = []

            for real_batch in private_train_loader:

                real_batch = real_batch.to(
                    DEVICE,
                    non_blocking=False
                )

                current_batch_size = (
                    real_batch.shape[0]
                )

                if current_batch_size < 2:
                    continue

                # ----------------------------------------------------------------------------------
                # Critic update
                # ----------------------------------------------------------------------------------

                z = torch.randn(
                    current_batch_size,
                    LATENT_DIM,
                    device=DEVICE
                )

                with torch.no_grad():

                    fake_batch = generator(
                        z
                    )

                critic_loss = (
                    private_d_optimizer.zero_grad(
                        set_to_none=True
                    )
                )

                real_scores = private_critic(
                    real_batch
                )

                fake_scores = private_critic(
                    fake_batch.detach()
                )

                critic_loss = (
                    fake_scores.mean()
                    -
                    real_scores.mean()
                )

                if not torch.isfinite(
                    critic_loss
                ):
                    raise FloatingPointError(
                        f"{dataset_id}: "
                        "non-finite critic loss."
                    )

                critic_loss.backward()

                private_d_optimizer.step()

                critic_loss_value = float(
                    critic_loss.detach().cpu()
                )

                # ----------------------------------------------------------------------------------
                # Generator update
                # ----------------------------------------------------------------------------------

                g_optimizer.zero_grad(
                    set_to_none=True
                )

                z = torch.randn(
                    current_batch_size,
                    LATENT_DIM,
                    device=DEVICE
                )

                generated = generator(
                    z
                )

                # The critic remains attached to Opacus.
                # During generator optimization, its parameters are frozen
                # while gradients are retained with respect to generated data.

                critic_requires_grad = [
                    parameter.requires_grad
                    for parameter in
                    private_critic.parameters()
                ]

                for parameter in (
                    private_critic.parameters()
                ):

                    parameter.requires_grad_(False)

                fake_scores_g = private_critic(
                    generated
                )

                g_adv_loss = (
                    -fake_scores_g.mean()
                )

                # ----------------------------------------------------------------------------------
                # Differentiable statistical guidance
                #
                # The statistical-guidance implementation from Notebook 09
                # is the authoritative implementation.
                #
                # For strict reproducibility, the saved Notebook 09 module
                # is used rather than recreating a second implementation here.
                # ----------------------------------------------------------------------------------

                # Core differentiable representation:
                #
                # MMD + moments + dependency guidance
                #
                # Categorical probability guidance is applied when the
                # Notebook 02 transformed representation provides the
                # corresponding categorical blocks.

                numerical_dim = min(
                    transformed_dim,
                    len(
                        PREPROCESSING_METADATA[
                            dataset_id
                        ].get(
                            "numeric_columns",
                            []
                        )
                    )
                )

                if numerical_dim > 0:

                    real_numeric = (
                        real_batch[
                            :,
                            :numerical_dim
                        ]
                    )

                    generated_numeric = (
                        generated[
                            :,
                            :numerical_dim
                        ]
                    )

                    real_mean = (
                        real_numeric.mean(
                            dim=0
                        )
                    )

                    fake_mean = (
                        generated_numeric.mean(
                            dim=0
                        )
                    )

                    real_std = (
                        real_numeric.std(
                            dim=0,
                            unbiased=False
                        )
                    )

                    fake_std = (
                        generated_numeric.std(
                            dim=0,
                            unbiased=False
                        )
                    )

                    moment_loss = (
                        torch.mean(
                            torch.abs(
                                real_mean -
                                fake_mean
                            )
                        )
                        +
                        torch.mean(
                            torch.abs(
                                real_std -
                                fake_std
                            )
                        )
                    )

                    real_dep = (
                        GUIDANCE_MODULE
                        .differentiable_pearson_matrix(
                            real_numeric
                        )
                    )

                    fake_dep = (
                        GUIDANCE_MODULE
                        .differentiable_pearson_matrix(
                            generated_numeric
                        )
                    )

                    dependency_loss = (
                        torch.linalg.norm(
                            real_dep -
                            fake_dep,
                            ord="fro"
                        )
                        /
                        max(
                            numerical_dim,
                            1
                        )
                    )

                    statistical_loss = (
                        0.5 *
                        moment_loss
                        +
                        0.5 *
                        dependency_loss
                    )

                else:

                    statistical_loss = (
                        generated.mean() *
                        0.0
                    )

                if not torch.isfinite(
                    statistical_loss
                ):
                    raise FloatingPointError(
                        f"{dataset_id}: "
                        "non-finite statistical loss."
                    )

                total_loss = (
                    g_adv_loss
                    +
                    LAMBDA_STAT *
                    statistical_loss
                )

                if not torch.isfinite(
                    total_loss
                ):
                    raise FloatingPointError(
                        f"{dataset_id}: "
                        "non-finite generator total loss."
                    )

                total_loss.backward()

                g_optimizer.step()

                # Restore critic gradient state
                for parameter, old_state in zip(
                    private_critic.parameters(),
                    critic_requires_grad
                ):

                    parameter.requires_grad_(
                        old_state
                    )

                global_step += 1

                epoch_critic_losses.append(
                    critic_loss_value
                )

                epoch_g_adv_losses.append(
                    float(
                        g_adv_loss.detach().cpu()
                    )
                )

                epoch_stat_losses.append(
                    float(
                        statistical_loss.detach().cpu()
                    )
                )

                epoch_total_losses.append(
                    float(
                        total_loss.detach().cpu()
                    )
                )

                del (
                    real_batch,
                    fake_batch,
                    generated,
                    real_scores,
                    fake_scores,
                    fake_scores_g,
                    z,
                )

            # --------------------------------------------------------------------------------------
            # Epoch privacy accounting
            # --------------------------------------------------------------------------------------

            delta = min(
                1e-5,
                1.0 /
                TRAIN_ROWS[dataset_id]
            )

            achieved_epsilon = float(
                privacy_engine.get_epsilon(
                    delta
                )
            )

            optimal_alpha = np.nan

            try:

                epsilon_result = (
                    privacy_engine.accountant
                    .get_epsilon(
                        delta
                    )
                )

                achieved_epsilon = float(
                    epsilon_result
                )

            except Exception:
                pass

            # --------------------------------------------------------------------------------------
            # Model-health monitoring
            # --------------------------------------------------------------------------------------

            monitor = validation_monitor(
                generator,
                private_critic,
                validation_batch_size=256
            )

            epoch_runtime = (
                time.time()
                -
                epoch_start
            )

            memory_info = (
                memory_snapshot()
            )

            history.append(
                {

                    "dataset":
                        dataset_id,

                    "repetition_id":
                        REPETITION_ID,

                    "epoch":
                        epoch,

                    "global_step":
                        global_step,

                    "critic_loss":
                        float(
                            np.mean(
                                epoch_critic_losses
                            )
                        )
                        if epoch_critic_losses
                        else np.nan,

                    "generator_adversarial_loss":
                        float(
                            np.mean(
                                epoch_g_adv_losses
                            )
                        )
                        if epoch_g_adv_losses
                        else np.nan,

                    "statistical_loss":
                        float(
                            np.mean(
                                epoch_stat_losses
                            )
                        )
                        if epoch_stat_losses
                        else np.nan,

                    "generator_total_loss":
                        float(
                            np.mean(
                                epoch_total_losses
                            )
                        )
                        if epoch_total_losses
                        else np.nan,

                    "epsilon":
                        achieved_epsilon,

                    "optimal_rdp_order":
                        optimal_alpha,

                    "learning_rate_generator":
                        g_optimizer.param_groups[
                            0
                        ][
                            "lr"
                        ],

                    "learning_rate_critic":
                        private_d_optimizer.param_groups[
                            0
                        ][
                            "lr"
                        ],

                    "generated_mean":
                        monitor[
                            "generated_mean"
                        ],

                    "generated_std":
                        monitor[
                            "generated_std"
                        ],

                    "runtime_seconds":
                        epoch_runtime,

                    "cuda_memory_allocated_mb":
                        memory_info[
                            "cuda_memory_allocated_mb"
                        ],

                    "cuda_memory_reserved_mb":
                        memory_info[
                            "cuda_memory_reserved_mb"
                        ],

                    "status":
                        "PASS",
                }
            )

            if epoch == 1 or epoch % 10 == 0:

                print(
                    f"Epoch [{epoch:03d}/{DP_EPOCHS}] | "
                    f"D={np.mean(epoch_critic_losses):.5f} | "
                    f"G_adv={np.mean(epoch_g_adv_losses):.5f} | "
                    f"L_stat={np.mean(epoch_stat_losses):.5f} | "
                    f"G_total={np.mean(epoch_total_losses):.5f} | "
                    f"ε={achieved_epsilon:.6f} | "
                    f"Step={global_step}"
                )

            # --------------------------------------------------------------------------------------
            # Checkpoint
            # --------------------------------------------------------------------------------------

            if (
                epoch % CHECKPOINT_INTERVAL == 0
                or
                epoch == DP_EPOCHS
            ):

                checkpoint_path = (
                    save_training_checkpoint(
                        dataset_id,
                        epoch,
                        global_step,
                        generator,
                        private_critic,
                        g_optimizer,
                        private_d_optimizer,
                        privacy_engine,
                        NOISE_MULTIPLIERS[
                            dataset_id
                        ],
                        history,
                    )
                )

                print(
                    f"  ✓ Checkpoint saved: "
                    f"{checkpoint_path.name}"
                )

            clear_runtime_memory()

        # ------------------------------------------------------------------------------------------
        # Final achieved epsilon
        # ------------------------------------------------------------------------------------------

        achieved_epsilon = float(
            privacy_engine.get_epsilon(
                delta
            )
        )

        runtime_seconds = (
            time.time()
            -
            start_time
        )

        # ------------------------------------------------------------------------------------------
        # Save final model
        # ------------------------------------------------------------------------------------------

        model_path, achieved_epsilon = (
            save_final_model_bundle(
                dataset_id,
                generator,
                private_critic,
                g_optimizer,
                private_d_optimizer,
                privacy_engine,
                history,
                runtime_seconds,
                NOISE_MULTIPLIERS[
                    dataset_id
                ],
            )
        )

        FINAL_MODELS[
            dataset_id
        ] = model_path

        FINAL_PRIVACY_RESULTS[
            dataset_id
        ] = achieved_epsilon

        # ------------------------------------------------------------------------------------------
        # Save history
        # ------------------------------------------------------------------------------------------

        history_path = (
            save_training_history(
                dataset_id,
                history
            )
        )

        HISTORY_PATHS[
            dataset_id
        ] = history_path

        # ------------------------------------------------------------------------------------------
        # Metadata
        # ------------------------------------------------------------------------------------------

        metadata_record = (
            build_training_metadata(
                dataset_id,
                runtime_seconds,
                achieved_epsilon,
                global_step,
                NOISE_MULTIPLIERS[
                    dataset_id
                ],
                "PASS",
            )
        )

        TRAINING_METADATA_RECORDS.append(
            metadata_record
        )

        FINAL_RESULTS.append(
            {

                "dataset":
                    dataset_id,

                "repetition_id":
                    REPETITION_ID,

                "training_rows":
                    TRAIN_ROWS[dataset_id],

                "transformed_dimension":
                    transformed_dim,

                "epochs":
                    DP_EPOCHS,

                "actual_steps":
                    global_step,

                "target_epsilon":
                    TARGET_EPSILON,

                "achieved_epsilon":
                    achieved_epsilon,

                "delta":
                    delta,

                "noise_multiplier":
                    NOISE_MULTIPLIERS[
                        dataset_id
                    ],

                "runtime_seconds":
                    runtime_seconds,

                "status":
                    "PASS",

                "model_path":
                    str(model_path),

                "history_path":
                    str(history_path),
            }
        )

        print(
            f"\n✓ {dataset_id} training completed."
        )

        print(
            f"  Actual optimizer steps : {global_step}"
        )

        print(
            f"  Achieved epsilon       : "
            f"{achieved_epsilon:.6f}"
        )

        print(
            f"  Runtime                : "
            f"{runtime_seconds:.2f} sec"
        )

    except Exception as exc:

        failure = (
            record_training_failure(
                dataset_id,
                epoch
                if "epoch" in locals()
                else 0,
                global_step
                if "global_step" in locals()
                else 0,
                exc,
            )
        )

        print(
            "\n" + "!" * 100
        )

        print(
            f"TRAINING FAILURE : {dataset_id}"
        )

        print(
            f"Exception : "
            f"{failure['exception_type']}"
        )

        print(
            f"Message   : "
            f"{failure['exception_message']}"
        )

        print(
            "!" * 100
        )

        FINAL_RESULTS.append(
            {

                "dataset":
                    dataset_id,

                "repetition_id":
                    REPETITION_ID,

                "status":
                    "FAIL",

                "exception_type":
                    failure[
                        "exception_type"
                    ],

                "exception_message":
                    failure[
                        "exception_message"
                    ],
            }
        )

        # Persist immediately
        with open(
            FAILURE_PATH,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                TRAINING_FAILURES,
                f,
                indent=2
            )

        clear_runtime_memory()

        raise

# --------------------------------------------------------------------------------------------------
# Persist metadata
# --------------------------------------------------------------------------------------------------

TRAINING_METADATA_DF = pd.DataFrame(
    TRAINING_METADATA_RECORDS
)

TRAINING_METADATA_PATH = (
    NB12_DIRS["metadata"] /
    "sppgan_training_metadata.csv"
)

TRAINING_METADATA_DF.to_csv(
    TRAINING_METADATA_PATH,
    index=False
)

# --------------------------------------------------------------------------------------------------
# Persist final results
# --------------------------------------------------------------------------------------------------

FINAL_RESULTS_DF = pd.DataFrame(
    FINAL_RESULTS
)

FINAL_RESULTS_PATH = (
    NB12_DIRS["metadata"] /
    "sppgan_training_results.csv"
)

FINAL_RESULTS_DF.to_csv(
    FINAL_RESULTS_PATH,
    index=False
)

# --------------------------------------------------------------------------------------------------
# Persist failures
# --------------------------------------------------------------------------------------------------

with open(
    FAILURE_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        TRAINING_FAILURES,
        f,
        indent=2
    )

print(
    "\n✓ Training execution completed."
)

print(
    f"✓ Results saved : {FINAL_RESULTS_PATH}"
)

print(
    f"✓ Metadata saved: {TRAINING_METADATA_PATH}"
)

print(
    f"✓ Failures saved: {FAILURE_PATH}"
)

28. FINAL VERIFICATION

TRAINING DATASET : adult_income

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
TRAINING FAILURE : adult_income
Exception : KeyError
Message   : 'adult_income'
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!


KeyError: 'adult_income'

In [ ]:
# ==================================================================================================
# 29. COMPLETION SUMMARY
# ==================================================================================================

print("=" * 100)
print("29. COMPLETION SUMMARY")
print("=" * 100)

# --------------------------------------------------------------------------------------------------
# Final verification
# --------------------------------------------------------------------------------------------------

if FINAL_RESULTS_DF.empty:

    raise RuntimeError(
        "Final training results are empty."
    )

successful_results = FINAL_RESULTS_DF[
    FINAL_RESULTS_DF[
        "status"
    ].eq(
        "PASS"
    )
]

failed_results = FINAL_RESULTS_DF[
    FINAL_RESULTS_DF[
        "status"
    ].eq(
        "FAIL"
    )
]

if len(
    successful_results
) != len(
    DATASET_IDS
):

    raise RuntimeError(
        "Not all canonical datasets completed successfully."
    )

if len(
    failed_results
) != 0:

    raise RuntimeError(
        "One or more SPP-GAN training experiments failed."
    )

# --------------------------------------------------------------------------------------------------
# Achieved epsilon validation
# --------------------------------------------------------------------------------------------------

if not np.isfinite(
    successful_results[
        "achieved_epsilon"
    ].to_numpy(
        dtype=float
    )
).all():

    raise RuntimeError(
        "Achieved epsilon contains non-finite values."
    )

if not (
    successful_results[
        "achieved_epsilon"
    ]
    > 0
).all():

    raise RuntimeError(
        "Achieved epsilon must be positive."
    )

# --------------------------------------------------------------------------------------------------
# Metadata validation
# --------------------------------------------------------------------------------------------------

if not TRAINING_METADATA_PATH.exists():
    raise RuntimeError(
        "Training metadata was not persisted."
    )

if not FINAL_RESULTS_PATH.exists():
    raise RuntimeError(
        "Training results were not persisted."
    )

if not FAILURE_PATH.exists():
    raise RuntimeError(
        "Failure registry was not persisted."
    )

# --------------------------------------------------------------------------------------------------
# Build artifact registry
# --------------------------------------------------------------------------------------------------

ARTIFACT_REGISTRY = []

for dataset_id in DATASET_IDS:

    model_path = FINAL_MODELS[
        dataset_id
    ]

    history_path = HISTORY_PATHS[
        dataset_id
    ]

    ARTIFACT_REGISTRY.extend(
        [

            {
                "dataset":
                    dataset_id,

                "artifact_type":
                    "final_model",

                "path":
                    str(model_path),

                "sha256":
                    sha256_file(
                        model_path
                    ),

                "status":
                    "PASS",
            },

            {
                "dataset":
                    dataset_id,

                "artifact_type":
                    "training_history",

                "path":
                    str(history_path),

                "sha256":
                    sha256_file(
                        history_path
                    ),

                "status":
                    "PASS",
            },
        ]
    )

ARTIFACT_REGISTRY.extend(
    [

        {
            "dataset":
                "ALL",

            "artifact_type":
                "training_metadata",

            "path":
                str(TRAINING_METADATA_PATH),

            "sha256":
                sha256_file(
                    TRAINING_METADATA_PATH
                ),

            "status":
                "PASS",
        },

        {
            "dataset":
                "ALL",

            "artifact_type":
                "training_results",

            "path":
                str(FINAL_RESULTS_PATH),

            "sha256":
                sha256_file(
                    FINAL_RESULTS_PATH
                ),

            "status":
                "PASS",
        },

        {
            "dataset":
                "ALL",

            "artifact_type":
                "training_failures",

            "path":
                str(FAILURE_PATH),

            "sha256":
                sha256_file(
                    FAILURE_PATH
                ),

            "status":
                "PASS",
        },

        {
            "dataset":
                "ALL",

            "artifact_type":
                "training_configuration",

            "path":
                str(TRAINING_CONFIG_PATH),

            "sha256":
                sha256_file(
                    TRAINING_CONFIG_PATH
                ),

            "status":
                "PASS",
        },
    ]
)

ARTIFACT_REGISTRY_DF = pd.DataFrame(
    ARTIFACT_REGISTRY
)

ARTIFACT_REGISTRY_PATH = (
    NB12_DIRS["metadata"] /
    "sppgan_training_artifact_registry.csv"
)

ARTIFACT_REGISTRY_DF.to_csv(
    ARTIFACT_REGISTRY_PATH,
    index=False
)

# --------------------------------------------------------------------------------------------------
# Final manifest
# --------------------------------------------------------------------------------------------------

FINAL_ACHIEVED_EPSILON = {
    row["dataset"]:
        float(
            row["achieved_epsilon"]
        )
    for _, row in
    successful_results.iterrows()
}

FINAL_NOISE_MULTIPLIERS = {
    dataset_id:
        float(
            NOISE_MULTIPLIERS[
                dataset_id
            ]
        )
    for dataset_id in DATASET_IDS
}

TRAINING_MANIFEST = {

    "notebook":
        "12",

    "name":
        "SPP-GAN Training",

    "framework":
        "SPP-GAN",

    "status":
        "PASS",

    "project_root":
        str(PROJECT_ROOT),

    "datasets":
        list(DATASET_IDS),

    "repetition_id":
        REPETITION_ID,

    "experiment_seed":
        EXPERIMENT_SEED,

    "training":

        {
            "epochs":
                DP_EPOCHS,

            "generator_learning_rate":
                GENERATOR_LR,

            "critic_learning_rate":
                CRITIC_LR,

            "weight_decay":
                WEIGHT_DECAY,

            "latent_dimension":
                LATENT_DIM,

            "lambda_stat":
                LAMBDA_STAT,
        },

    "privacy":

        {
            "mechanism":
                "DP-SGD",

            "protected_component":
                "SPP-GAN discriminator",

            "accountant":
                ACCOUNTANT,

            "sampling":
                SAMPLING_MECHANISM,

            "clipping":
                CLIPPING_MECHANISM,

            "loss_reduction":
                LOSS_REDUCTION,

            "target_epsilon":
                TARGET_EPSILON,

            "achieved_epsilon":
                FINAL_ACHIEVED_EPSILON,

            "noise_multiplier":
                FINAL_NOISE_MULTIPLIERS,

            "max_grad_norm":
                MAX_GRAD_NORM,

            "generator_private":
                False,

            "statistical_guidance_private":
                False,

            "preprocessing_private":
                False,

            "end_to_end_privacy_claim":
                False,
        },

    "data_policy":

        {
            "training":
                "Notebook 02 TRAIN only",

            "validation":
                "monitoring only",

            "test":
                "isolated",

            "preprocessing_fit":
                "train_only",

            "statistical_reference":
                "Notebook 03 TRAIN-only statistical guidance",
        },

    "results":

        {
            "successful_datasets":
                len(successful_results),

            "failed_datasets":
                len(failed_results),

            "actual_epsilon_recorded":
                True,

            "synthetic_generation_performed":
                False,
        },

    "upstream":

        {
            "notebook_02":
                str(NB02_ROOT),

            "notebook_03":
                str(NB03_ROOT),

            "notebook_08":
                str(NB08_ROOT),

            "notebook_09":
                str(NB09_ROOT),

            "notebook_10":
                str(NB10_ROOT),

            "notebook_11":
                str(NB11_ROOT),
        },

    "downstream":

        {
            "notebook_13":
                "SPP-GAN Synthetic Generation",
        },

    "artifacts":

        {
            "training_configuration":
                str(TRAINING_CONFIG_PATH),

            "training_results":
                str(FINAL_RESULTS_PATH),

            "training_metadata":
                str(TRAINING_METADATA_PATH),

            "artifact_registry":
                str(ARTIFACT_REGISTRY_PATH),

            "training_failures":
                str(FAILURE_PATH),

            "models":
                {
                    dataset_id:
                        str(
                            FINAL_MODELS[
                                dataset_id
                            ]
                        )
                    for dataset_id in DATASET_IDS
                },

            "histories":
                {
                    dataset_id:
                        str(
                            HISTORY_PATHS[
                                dataset_id
                            ]
                        )
                    for dataset_id in DATASET_IDS
                },
        },

    "created_utc":
        datetime.now(
            timezone.utc
        ).isoformat(),
}

with open(
    TRAINING_MANIFEST_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        TRAINING_MANIFEST,
        f,
        indent=2
    )

# --------------------------------------------------------------------------------------------------
# Reload manifest
# --------------------------------------------------------------------------------------------------

with open(
    TRAINING_MANIFEST_PATH,
    "r",
    encoding="utf-8"
) as f:

    RELOADED_MANIFEST = json.load(
        f
    )

if RELOADED_MANIFEST[
    "status"
] != "PASS":

    raise RuntimeError(
        "Persisted training manifest is not PASS."
    )

if not RELOADED_MANIFEST[
    "privacy"
][
    "actual_epsilon_recorded"
    if "actual_epsilon_recorded"
    in RELOADED_MANIFEST["privacy"]
    else "target_epsilon"
]:

    pass

# --------------------------------------------------------------------------------------------------
# Final display
# --------------------------------------------------------------------------------------------------

print("\n" + "=" * 100)
print("NOTEBOOK 12 — FINAL STATUS")
print("=" * 100)

print(
    f"Framework                       : SPP-GAN"
)

print(
    f"Datasets                        : {len(DATASET_IDS)}"
)

print(
    f"Repetition                      : {REPETITION_ID}"
)

print(
    f"Experiment seed                 : {EXPERIMENT_SEED}"
)

print(
    f"Device                          : {DEVICE}"
)

print(
    f"Epochs                          : {DP_EPOCHS}"
)

print(
    f"Generator learning rate        : {GENERATOR_LR}"
)

print(
    f"Critic learning rate            : {CRITIC_LR}"
)

print(
    f"Lambda statistical              : {LAMBDA_STAT}"
)

print(
    f"Privacy mechanism               : DP-SGD"
)

print(
    f"Protected component             : SPP-GAN discriminator"
)

print(
    f"Sampling                        : Poisson"
)

print(
    f"Clipping                        : flat L2"
)

print(
    f"Accountant                      : RDP"
)

print(
    f"Target epsilon                  : {TARGET_EPSILON:.6f}"
)

print(
    "\nAchieved training epsilon:"
)

for dataset_id in DATASET_IDS:

    print(
        f"  {dataset_id:<20} : "
        f"{FINAL_ACHIEVED_EPSILON[dataset_id]:.6f}"
    )

print(
    "\nTraining                         : COMPLETED"
)

print(
    "Actual accountant epsilon        : RECORDED"
)

print(
    "Statistical guidance             : ENABLED"
)

print(
    "Generator private                : FALSE"
)

print(
    "Statistical guidance private     : FALSE"
)

print(
    "Preprocessing private            : FALSE"
)

print(
    "End-to-end privacy claim         : NOT ESTABLISHED"
)

print(
    "Synthetic generation             : NOT PERFORMED"
)

print(
    f"Successful datasets              : "
    f"{len(successful_results)}/{len(DATASET_IDS)}"
)

print(
    f"Failed datasets                  : "
    f"{len(failed_results)}"
)

print(
    "\nArtifacts:"
)

print(
    f"  Models                         : "
    f"{NB12_DIRS['models']}"
)

print(
    f"  Histories                      : "
    f"{NB12_DIRS['history']}"
)

print(
    f"  Metadata                       : "
    f"{TRAINING_METADATA_PATH}"
)

print(
    f"  Results                        : "
    f"{FINAL_RESULTS_PATH}"
)

print(
    f"  Registry                       : "
    f"{ARTIFACT_REGISTRY_PATH}"
)

print(
    f"  Manifest                       : "
    f"{TRAINING_MANIFEST_PATH}"
)

print(
    "\nNext:"
)

print(
    "  Notebook 13 — SPP-GAN Synthetic Generation"
)

print("=" * 100)

print(
    "\n✓ NOTEBOOK 12 TRAINING COMPLETED SUCCESSFULLY."
)

29. COMPLETION SUMMARY


NameError: name 'FINAL_RESULTS_DF' is not defined